# 203 — TCGA Multi-omic Integration

## Objective

Construct the definitive, reproducible TCGA primary-tumor multi-omic cohort by integrating the RNA-seq and DNA-methylation quality-control results generated by notebooks 201 and 202.

This notebook will first define and apply an explicit, deterministic, and provenance-preserving policy for selecting one RNA-seq–methylation pair per TCGA case. It will then construct the final RNA-seq and methylation representations using only the selected paired cohort, while retaining all candidate pairs and excluded alternatives for auditability.

## Authoritative upstream artifacts

This notebook consumes:

- the RNA-seq-QC-annotated candidate-pair inventory produced by notebook 201;
- the candidate-scope raw `unstranded` RNA-seq count matrix and its row and column mappings;
- the RNA-seq file-level QC inventory;
- the methylation-QC-annotated candidate-pair inventory produced by notebook 202;
- the methylation file-level and probe-level QC artifacts;
- the validated local SeSAMe beta-value files; and
- the frozen TCGA file indices and cohort definitions for provenance and consistency checks.

The upstream candidate inventory preserves 10,162 source candidate-pair rows across 9,973 TCGA cases. The post-QC eligible subset and the final selected cohort will be derived from the persisted eligibility fields and validated programmatically rather than assumed from these summary values.

## Biological unit and pairing policy

The independent biological unit is the TCGA case, represented by `case_id`.

The following rules apply:

- each final multi-omic observation must represent one `case_id`;
- RNA-seq and methylation must originate from the same harmonized `sample_id`;
- files from different samples belonging to the same case are not automatically paired;
- aliquot and file identifiers are retained for technical provenance;
- no case may contribute more than one independent final observation; and
- unresolved candidates must not be resolved through arbitrary or undocumented tie-breaking.

The within-case selection hierarchy will be defined and frozen before final cohort-dependent filtering, normalization, or matrix publication.

## Multi-omic representation strategy

The canonical RNA-seq source remains the raw `unstranded` STAR Counts representation. Cohort-dependent abundance filtering, normalization, and transformation will be fitted only after the definitive paired cohort has been selected.

DNA methylation will be represented through explicitly separated analytical spaces:

- an HM27-specific probe space;
- an HM450-specific probe space; and
- a validated shared HM27/HM450 probe space for direct cross-platform comparisons.

HM27 and HM450 will not be merged through naïve platform pooling. Probe identity, ordering, missingness, platform membership, and previously applied technical masks will remain explicit in the resulting mappings and metadata.

## Main tasks

This notebook will:

1. validate the upstream QC inventories and molecular-artifact mappings;
2. propagate RNA-seq and methylation eligibility to the candidate-pair level;
3. characterize residual within-case multiplicity;
4. define and apply a deterministic final-pair selection policy;
5. retain selected, excluded, and unresolved candidates with explicit reasons;
6. construct the final case-level RNA-seq mapping;
7. construct the cohort-frozen raw `unstranded` RNA-seq representation; cohort-dependent filtering, normalization, and transformation remain deferred to downstream preprocessing;
8. construct platform-specific and shared methylation representations;
9. validate sample identity, feature order, missingness, and matrix correspondence; and
10. write versioned downstream-consumable cohort, matrix, mapping, and metadata artifacts.

## Scope boundaries

This notebook will not:

- repeat Phase 1 download or MD5 validation;
- replace the file-level QC performed in notebooks 201 and 202;
- silently discard candidate pairs or technical alternatives;
- pool HM27 and HM450 probes without an explicit shared-probe definition;
- perform batch correction or confounder adjustment;
- perform final lineage, purity, immune-infiltration, stromal-infiltration, or proliferation assessment;
- discover biological programs;
- infer causal methylation–expression mechanisms;
- predict clinical treatment response; or
- assign clinical resistance labels.

## Expected outputs

After the selection policy and matrix schemas have been validated, this notebook will write:

- a complete case-level pair-selection inventory retaining all candidate rows and their selection status;
- a definitive one-pair-per-case RNA-seq–methylation mapping;
- a final RNA-seq matrix with explicit gene-feature and sample mappings;
- HM27-specific, HM450-specific, and shared methylation representations;
- probe- and sample-level missingness and eligibility metadata; and
- reproducibility metadata containing input provenance, output schemas, row and column order, and artifact hashes.

Exact filenames and schemas will be fixed before publication and validated against the in-memory objects.

In [1]:
# =============================================================================
# Imports
# =============================================================================

import json
import gc
import shutil
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from pancancer_epigenetics.utils.paths import (
    Paths,
    project_relative_path,
)

from pancancer_epigenetics.utils.file_checks import (
    calculate_sha256,
)

In [2]:
# =============================================================================
# Inputs and project paths
# =============================================================================


# -----------------------------------------------------------------------------
# Canonical project root
# -----------------------------------------------------------------------------

PROJECT_ROOT = Paths.root


# -----------------------------------------------------------------------------
# Authoritative candidate-pair inventory from notebook 202
# -----------------------------------------------------------------------------

METHYLATION_QC_ANNOTATED_PAIR_INVENTORY_PATH = (
    Paths.metadata
    / (
        "tcga_primary_tumor_rnaseq_methylation_"
        "qc_annotated_candidate_pair_inventory.csv"
    )
)


# -----------------------------------------------------------------------------
# RNA-seq integration inputs from notebook 201
# -----------------------------------------------------------------------------

RAW_COUNT_MATRIX_PATH = (
    Paths.expression
    / "tcga_primary_tumor_rnaseq_candidate_unstranded_raw_counts.npy"
)

RNA_GENE_FEATURE_INDEX_PATH = (
    Paths.expression
    / "tcga_primary_tumor_rnaseq_gene_feature_index.csv"
)

RNA_FILE_QC_INVENTORY_PATH = (
    Paths.qc
    / "tcga_primary_tumor_rnaseq_file_qc_inventory.csv"
)

RNA_RAW_COUNTS_METADATA_PATH = (
    Paths.expression
    / (
        "tcga_primary_tumor_rnaseq_candidate_unstranded_"
        "raw_counts_metadata.json"
    )
)


# -----------------------------------------------------------------------------
# DNA-methylation integration inputs from notebook 202
# -----------------------------------------------------------------------------

METHYLATION_FILE_QC_METRICS_PATH = (
    Paths.qc
    / (
        "tcga_primary_tumor_methylation_"
        "candidate_file_qc_metrics.csv"
    )
)

PROBE_LEVEL_QC_METRICS_PATH = (
    Paths.qc
    / (
        "tcga_primary_tumor_methylation_hm27_hm450_"
        "probe_qc_metrics.csv"
    )
)

SHARED_PROBE_QC_INVENTORY_PATH = (
    Paths.metadata
    / (
        "tcga_primary_tumor_methylation_hm27_hm450_"
        "shared_probe_qc_inventory.csv"
    )
)


# -----------------------------------------------------------------------------
# Frozen TCGA provenance inputs
# -----------------------------------------------------------------------------

RNA_MANIFEST_DIR = (
    Paths.config
    / "manifests"
    / "tcga_rna"
)

RNA_FILE_INDEX_PATH = (
    RNA_MANIFEST_DIR
    / "gdc_file_index_tcga_primary_tumor_rnaseq_star_counts.tsv"
)

RNA_COHORT_FREEZE_PATH = (
    RNA_MANIFEST_DIR
    / "gdc_cohort_freeze_tcga_primary_tumor_rnaseq_star_counts.json"
)

METHYLATION_MANIFEST_DIR = (
    Paths.config
    / "manifests"
    / "tcga_methylation"
)

METHYLATION_FILE_INDEX_PATH = (
    METHYLATION_MANIFEST_DIR
    / (
        "gdc_file_index_tcga_primary_tumor_methylation_array_"
        "sesame_beta_values_hm27_hm450.tsv"
    )
)

METHYLATION_COHORT_FREEZE_PATH = (
    METHYLATION_MANIFEST_DIR
    / (
        "gdc_cohort_freeze_tcga_primary_tumor_methylation_array_"
        "sesame_beta_values_hm27_hm450.json"
    )
)


# -----------------------------------------------------------------------------
# Local molecular payload directories
# -----------------------------------------------------------------------------

RNA_STAR_COUNTS_DIR = (
    Paths.tcga
    / "star_counts"
)

METHYLATION_DOWNLOAD_DIR = (
    Paths.tcga
    / "methylation"
)


# -----------------------------------------------------------------------------
# Downstream output locations
# -----------------------------------------------------------------------------

INTEGRATION_METADATA_OUTPUT_DIR = Paths.metadata
INTEGRATION_EXPRESSION_OUTPUT_DIR = Paths.expression
INTEGRATION_METHYLATION_OUTPUT_DIR = Paths.methylation
INTEGRATION_QC_OUTPUT_DIR = Paths.qc


# -----------------------------------------------------------------------------
# Path summary
# -----------------------------------------------------------------------------

print("TCGA multi-omic integration paths resolved.")

print(
    f"Project root:                    "
    f"{project_relative_path(PROJECT_ROOT)}"
)

print(
    f"QC-annotated pair inventory:     "
    f"{project_relative_path(METHYLATION_QC_ANNOTATED_PAIR_INVENTORY_PATH)}"
)

print(
    f"RNA-seq raw-count matrix:        "
    f"{project_relative_path(RAW_COUNT_MATRIX_PATH)}"
)

print(
    f"RNA-seq gene-feature index:      "
    f"{project_relative_path(RNA_GENE_FEATURE_INDEX_PATH)}"
)

print(
    f"RNA-seq file-QC inventory:       "
    f"{project_relative_path(RNA_FILE_QC_INVENTORY_PATH)}"
)

print(
    f"Methylation file-QC metrics:     "
    f"{project_relative_path(METHYLATION_FILE_QC_METRICS_PATH)}"
)

print(
    f"Probe-level QC metrics:           "
    f"{project_relative_path(PROBE_LEVEL_QC_METRICS_PATH)}"
)

print(
    f"Shared-probe QC inventory:       "
    f"{project_relative_path(SHARED_PROBE_QC_INVENTORY_PATH)}"
)

print(
    f"RNA-seq payload directory:       "
    f"{project_relative_path(RNA_STAR_COUNTS_DIR)}"
)

print(
    f"Methylation payload directory:   "
    f"{project_relative_path(METHYLATION_DOWNLOAD_DIR)}"
)

TCGA multi-omic integration paths resolved.
Project root:                    .
QC-annotated pair inventory:     data/interim/metadata/tcga_primary_tumor_rnaseq_methylation_qc_annotated_candidate_pair_inventory.csv
RNA-seq raw-count matrix:        data/interim/expression/tcga_primary_tumor_rnaseq_candidate_unstranded_raw_counts.npy
RNA-seq gene-feature index:      data/interim/expression/tcga_primary_tumor_rnaseq_gene_feature_index.csv
RNA-seq file-QC inventory:       data/interim/qc/tcga_primary_tumor_rnaseq_file_qc_inventory.csv
Methylation file-QC metrics:     data/interim/qc/tcga_primary_tumor_methylation_candidate_file_qc_metrics.csv
Probe-level QC metrics:           data/interim/qc/tcga_primary_tumor_methylation_hm27_hm450_probe_qc_metrics.csv
Shared-probe QC inventory:       data/interim/metadata/tcga_primary_tumor_methylation_hm27_hm450_shared_probe_qc_inventory.csv
RNA-seq payload directory:       data/raw/tcga/star_counts
Methylation payload directory:   data/raw/tcga/methylat

In [3]:
# =============================================================================
# Validate input and output paths
# =============================================================================


REQUIRED_INPUT_FILES = {
    "methylation_qc_pair_inventory": METHYLATION_QC_ANNOTATED_PAIR_INVENTORY_PATH,
    "rna_raw_count_matrix": RAW_COUNT_MATRIX_PATH,
    "rna_gene_feature_index": RNA_GENE_FEATURE_INDEX_PATH,
    "rna_file_qc_inventory": RNA_FILE_QC_INVENTORY_PATH,
    "rna_raw_counts_metadata": RNA_RAW_COUNTS_METADATA_PATH,
    "methylation_file_qc_metrics": METHYLATION_FILE_QC_METRICS_PATH,
    "probe_level_qc_metrics": PROBE_LEVEL_QC_METRICS_PATH,
    "shared_probe_qc_inventory": SHARED_PROBE_QC_INVENTORY_PATH,
    "rna_file_index": RNA_FILE_INDEX_PATH,
    "rna_cohort_freeze": RNA_COHORT_FREEZE_PATH,
    "methylation_file_index": METHYLATION_FILE_INDEX_PATH,
    "methylation_cohort_freeze": METHYLATION_COHORT_FREEZE_PATH,
}


REQUIRED_INPUT_DIRECTORIES = {
    "rna_star_counts_payloads": RNA_STAR_COUNTS_DIR,
    "methylation_payloads": METHYLATION_DOWNLOAD_DIR,
}


REQUIRED_OUTPUT_DIRECTORIES = {
    "integration_metadata": INTEGRATION_METADATA_OUTPUT_DIR,
    "integration_expression": INTEGRATION_EXPRESSION_OUTPUT_DIR,
    "integration_methylation": INTEGRATION_METHYLATION_OUTPUT_DIR,
    "integration_qc": INTEGRATION_QC_OUTPUT_DIR,
}


missing_files = [
    label
    for label, path in REQUIRED_INPUT_FILES.items()
    if not path.is_file()
]

missing_input_directories = [
    label
    for label, path in REQUIRED_INPUT_DIRECTORIES.items()
    if not path.is_dir()
]

missing_output_directories = [
    label
    for label, path in REQUIRED_OUTPUT_DIRECTORIES.items()
    if not path.is_dir()
]


all_declared_paths = [
    path.resolve()
    for path in (
        list(REQUIRED_INPUT_FILES.values())
        + list(REQUIRED_INPUT_DIRECTORIES.values())
        + list(REQUIRED_OUTPUT_DIRECTORIES.values())
    )
]

duplicate_paths = sorted(
    {
        str(path)
        for path in all_declared_paths
        if all_declared_paths.count(path) > 1
    }
)


if missing_files:
    raise FileNotFoundError(
        f"Missing required input files: {missing_files}"
    )

if missing_input_directories:
    raise FileNotFoundError(
        f"Missing required input directories: {missing_input_directories}"
    )

if missing_output_directories:
    raise FileNotFoundError(
        f"Missing required output directories: {missing_output_directories}"
    )

if duplicate_paths:
    raise ValueError(
        f"Duplicate declared paths detected: {duplicate_paths}"
    )


print("TCGA multi-omic integration path validation completed.")
print(f"Required input files validated: {len(REQUIRED_INPUT_FILES)}")
print(
    "Required input directories validated: "
    f"{len(REQUIRED_INPUT_DIRECTORIES)}"
)
print(
    "Required output directories validated: "
    f"{len(REQUIRED_OUTPUT_DIRECTORIES)}"
)
print("Missing files: 0")
print("Missing directories: 0")
print("Duplicate paths: 0")

TCGA multi-omic integration path validation completed.
Required input files validated: 12
Required input directories validated: 2
Required output directories validated: 4
Missing files: 0
Missing directories: 0
Duplicate paths: 0


In [4]:
# =============================================================================
# Load authoritative integration inputs
# =============================================================================

qc_annotated_pair_inventory = pd.read_csv(
    METHYLATION_QC_ANNOTATED_PAIR_INVENTORY_PATH,
    low_memory=False,
)

rna_file_qc_inventory = pd.read_csv(
    RNA_FILE_QC_INVENTORY_PATH,
    low_memory=False,
)

methylation_file_qc_metrics = pd.read_csv(
    METHYLATION_FILE_QC_METRICS_PATH,
    low_memory=False,
)

probe_level_qc_metrics = pd.read_csv(
    PROBE_LEVEL_QC_METRICS_PATH,
    low_memory=False,
)

shared_probe_qc_inventory = pd.read_csv(
    SHARED_PROBE_QC_INVENTORY_PATH,
    low_memory=False,
)

rna_file_index = pd.read_csv(
    RNA_FILE_INDEX_PATH,
    sep="\t",
    dtype="string",
    low_memory=False,
)

methylation_file_index = pd.read_csv(
    METHYLATION_FILE_INDEX_PATH,
    sep="\t",
    dtype="string",
    low_memory=False,
)

with RNA_COHORT_FREEZE_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    rna_cohort_freeze = json.load(handle)

with METHYLATION_COHORT_FREEZE_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    methylation_cohort_freeze = json.load(handle)

with RNA_RAW_COUNTS_METADATA_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    rna_raw_counts_metadata = json.load(handle)


print("Authoritative TCGA integration inputs loaded.")
print(
    f"QC-annotated pair inventory:   "
    f"{qc_annotated_pair_inventory.shape}"
)
print(
    f"RNA-seq file-QC inventory:      "
    f"{rna_file_qc_inventory.shape}"
)
print(
    f"Methylation file-QC metrics:   "
    f"{methylation_file_qc_metrics.shape}"
)
print(
    f"Probe-level QC metrics:         "
    f"{probe_level_qc_metrics.shape}"
)
print(
    f"Shared-probe QC inventory:      "
    f"{shared_probe_qc_inventory.shape}"
)
print(
    f"RNA-seq file index:             "
    f"{rna_file_index.shape}"
)
print(
    f"Methylation file index:         "
    f"{methylation_file_index.shape}"
)
print(
    f"RNA cohort-freeze keys:         "
    f"{len(rna_cohort_freeze)}"
)
print(
    f"Methylation cohort-freeze keys: "
    f"{len(methylation_cohort_freeze)}"
)
print(
    f"RNA raw-count metadata keys:    "
    f"{len(rna_raw_counts_metadata)}"
)

Authoritative TCGA integration inputs loaded.
QC-annotated pair inventory:   (10162, 63)
RNA-seq file-QC inventory:      (10122, 105)
Methylation file-QC metrics:   (10038, 22)
Probe-level QC metrics:         (514005, 14)
Shared-probe QC inventory:      (25978, 8)
RNA-seq file index:             (10308, 14)
Methylation file index:         (10897, 25)
RNA cohort-freeze keys:         12
Methylation cohort-freeze keys: 12
RNA raw-count metadata keys:    11


In [5]:
# =============================================================================
# Inspect authoritative input schemas
# =============================================================================

authoritative_tabular_inputs = {
    "qc_annotated_pair_inventory": qc_annotated_pair_inventory,
    "rna_file_qc_inventory": rna_file_qc_inventory,
    "methylation_file_qc_metrics": methylation_file_qc_metrics,
    "probe_level_qc_metrics": probe_level_qc_metrics,
    "shared_probe_qc_inventory": shared_probe_qc_inventory,
    "rna_file_index": rna_file_index,
    "methylation_file_index": methylation_file_index,
}

schema_checks = {}

for input_name, input_table in authoritative_tabular_inputs.items():
    schema_checks[
        f"{input_name}_is_not_empty"
    ] = not input_table.empty

    schema_checks[
        f"{input_name}_columns_are_unique"
    ] = input_table.columns.is_unique

schema_checks.update(
    {
        "rna_cohort_freeze_is_dictionary": (
            isinstance(rna_cohort_freeze, dict)
        ),
        "methylation_cohort_freeze_is_dictionary": (
            isinstance(methylation_cohort_freeze, dict)
        ),
        "rna_raw_counts_metadata_is_dictionary": (
            isinstance(rna_raw_counts_metadata, dict)
        ),
    }
)

print("Authoritative input-schema checks:")
for check_name, check_passed in schema_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(schema_checks.values()):
    raise ValueError(
        "One or more authoritative input schemas are invalid."
    )

print("\nAuthoritative tabular inputs:")
for input_name, input_table in authoritative_tabular_inputs.items():
    print(
        f"{input_name}: "
        f"{input_table.shape[0]:,} rows × "
        f"{input_table.shape[1]:,} columns"
    )

print("\nRNA cohort-freeze top-level keys:")
for key in rna_cohort_freeze:
    print(f"- {key}")

print("\nMethylation cohort-freeze top-level keys:")
for key in methylation_cohort_freeze:
    print(f"- {key}")

print("\nRNA raw-count metadata top-level keys:")
for key in rna_raw_counts_metadata:
    print(f"- {key}")

Authoritative input-schema checks:
qc_annotated_pair_inventory_is_not_empty: True
qc_annotated_pair_inventory_columns_are_unique: True
rna_file_qc_inventory_is_not_empty: True
rna_file_qc_inventory_columns_are_unique: True
methylation_file_qc_metrics_is_not_empty: True
methylation_file_qc_metrics_columns_are_unique: True
probe_level_qc_metrics_is_not_empty: True
probe_level_qc_metrics_columns_are_unique: True
shared_probe_qc_inventory_is_not_empty: True
shared_probe_qc_inventory_columns_are_unique: True
rna_file_index_is_not_empty: True
rna_file_index_columns_are_unique: True
methylation_file_index_is_not_empty: True
methylation_file_index_columns_are_unique: True
rna_cohort_freeze_is_dictionary: True
methylation_cohort_freeze_is_dictionary: True
rna_raw_counts_metadata_is_dictionary: True

Authoritative tabular inputs:
qc_annotated_pair_inventory: 10,162 rows × 63 columns
rna_file_qc_inventory: 10,122 rows × 105 columns
methylation_file_qc_metrics: 10,038 rows × 22 columns
probe_level

In [6]:
# =============================================================================
# Validate integration input contracts
# =============================================================================


integration_input_tables = {
    "qc_annotated_pair_inventory": qc_annotated_pair_inventory,
    "rna_file_qc_inventory": rna_file_qc_inventory,
    "methylation_file_qc_metrics": methylation_file_qc_metrics,
    "probe_level_qc_metrics": probe_level_qc_metrics,
    "shared_probe_qc_inventory": shared_probe_qc_inventory,
    "rna_file_index": rna_file_index,
    "methylation_file_index": methylation_file_index,
}


# -----------------------------------------------------------------------------
# Define the minimum downstream interface for each authoritative input
# -----------------------------------------------------------------------------

required_columns_by_input = {
    "qc_annotated_pair_inventory": {
        "case_submitter_id",
        "sample_submitter_id",
        "rna_case_uuid",
        "rna_file_id",
        "rna_file_name",
        "methylation_case_uuid",
        "methylation_file_id",
        "methylation_file_name",
        "methylation_platform",
        "rna_qc_eligible_for_downstream_selection",
        "methylation_qc_eligible_for_downstream_selection",
        "pair_eligible_after_rna_qc",
        "pair_eligible_after_methylation_qc",
        "pair_rna_qc_status",
        "pair_methylation_qc_status",
        "severe_qc_signal_count",
        "methylation_qc_review_flag",
    },
    "rna_file_qc_inventory": {
        "rna_file_id",
        "case_submitter_id",
        "matrix_column_index",
        "rna_qc_eligible_for_downstream_selection",
        "rna_qc_eligibility_status",
        "rna_qc_ineligibility_reason",
    },
    "methylation_file_qc_metrics": {
        "methylation_file_id",
        "methylation_platform",
        "row_count",
        "probe_ids_complete",
        "probe_ids_unique",
        "probe_count_matches_reference",
        "probe_order_matches_reference",
        "missing_beta_count",
        "missing_beta_fraction",
        "numeric_beta_count",
        "beta_values_finite",
        "beta_below_zero_count",
        "beta_above_one_count",
        "beta_min",
        "beta_q01",
        "beta_q05",
        "beta_median",
        "beta_mean",
        "beta_q95",
        "beta_q99",
        "beta_max",
        "beta_sd",
    },
    "probe_level_qc_metrics": {
        "methylation_platform",
        "platform_probe_order",
        "probe_id",
        "probe_space_status",
        "missing_beta_fraction",
        "probe_qc_eligible_within_platform",
    },
    "shared_probe_qc_inventory": {
        "shared_probe_order",
        "probe_id",
        "hm27_probe_qc_eligible",
        "hm450_probe_qc_eligible",
        "shared_probe_qc_eligible",
    },
    "rna_file_index": {
        "file_id",
        "file_name",
        "project_id",
        "case_id",
        "sample_id",
    },
    "methylation_file_index": {
        "file_id",
        "file_name",
        "platform",
        "project_id",
        "case_uuid",
        "case_submitter_id",
        "sample_submitter_id",
        "aliquot_uuid",
        "aliquot_submitter_id",
    },
}


missing_columns_by_input = {
    input_name: sorted(
        set(required_columns)
        - set(integration_input_tables[input_name].columns)
    )
    for input_name, required_columns in required_columns_by_input.items()
}


if any(missing_columns_by_input.values()):
    missing_summary = {
        input_name: missing_columns
        for input_name, missing_columns in missing_columns_by_input.items()
        if missing_columns
    }

    raise ValueError(
        "Required integration columns are missing: "
        f"{missing_summary}"
    )


# -----------------------------------------------------------------------------
# Validate identifier completeness and logical gate columns
# -----------------------------------------------------------------------------

identifier_columns_by_input = {
    "qc_annotated_pair_inventory": [
        "case_submitter_id",
        "sample_submitter_id",
        "rna_case_uuid",
        "rna_file_id",
        "rna_file_name",
        "methylation_case_uuid",
        "methylation_file_id",
        "methylation_file_name",
        "methylation_platform",
    ],
    "rna_file_qc_inventory": [
        "rna_file_id",
        "case_submitter_id",
        "matrix_column_index",
    ],
    "methylation_file_qc_metrics": [
        "methylation_file_id",
        "methylation_platform",
    ],
    "probe_level_qc_metrics": [
        "methylation_platform",
        "platform_probe_order",
        "probe_id",
    ],
    "shared_probe_qc_inventory": [
        "shared_probe_order",
        "probe_id",
    ],
    "rna_file_index": [
        "file_id",
        "file_name",
        "project_id",
        "case_id",
        "sample_id",
    ],
    "methylation_file_index": [
        "file_id",
        "file_name",
        "platform",
        "project_id",
        "case_uuid",
    ],
}


integration_contract_checks = {}

for input_name, columns in identifier_columns_by_input.items():
    input_table = integration_input_tables[input_name]
    identifier_values = input_table[columns].astype("string")

    integration_contract_checks[
        f"{input_name}_identifiers_are_complete"
    ] = (
        identifier_values.notna().all().all()
        and identifier_values.ne("").all().all()
    )


boolean_columns_by_input = {
    "qc_annotated_pair_inventory": [
        "rna_qc_eligible_for_downstream_selection",
        "methylation_qc_eligible_for_downstream_selection",
        "pair_eligible_after_rna_qc",
        "pair_eligible_after_methylation_qc",
        "methylation_qc_review_flag",
    ],
    "rna_file_qc_inventory": [
        "rna_qc_eligible_for_downstream_selection",
    ],
    "probe_level_qc_metrics": [
        "probe_qc_eligible_within_platform",
    ],
    "shared_probe_qc_inventory": [
        "hm27_probe_qc_eligible",
        "hm450_probe_qc_eligible",
        "shared_probe_qc_eligible",
    ],
}


for input_name, columns in boolean_columns_by_input.items():
    input_table = integration_input_tables[input_name]

    integration_contract_checks[
        f"{input_name}_boolean_gates_are_complete"
    ] = all(
        pd.api.types.is_bool_dtype(input_table[column])
        and input_table[column].notna().all()
        for column in columns
    )


# -----------------------------------------------------------------------------
# Validate key uniqueness and cross-modality case identity
# -----------------------------------------------------------------------------

integration_contract_checks.update(
    {
        "candidate_file_pairs_are_unique": (
            not qc_annotated_pair_inventory.duplicated(
                subset=["rna_file_id", "methylation_file_id"]
            ).any()
        ),
        "rna_file_qc_inventory_has_one_row_per_file": (
            rna_file_qc_inventory["rna_file_id"].is_unique
        ),
        "methylation_file_qc_metrics_have_one_row_per_file": (
            methylation_file_qc_metrics["methylation_file_id"].is_unique
        ),
        "probe_metrics_have_unique_platform_probe_pairs": (
            not probe_level_qc_metrics.duplicated(
                subset=["methylation_platform", "probe_id"]
            ).any()
        ),
        "shared_probe_inventory_has_unique_probe_ids": (
            shared_probe_qc_inventory["probe_id"].is_unique
        ),
        "shared_probe_orders_are_unique": (
            shared_probe_qc_inventory["shared_probe_order"].is_unique
        ),
        "rna_file_index_has_unique_file_ids": (
            rna_file_index["file_id"].is_unique
        ),
        "methylation_file_index_has_unique_file_ids": (
            methylation_file_index["file_id"].is_unique
        ),
        "rna_and_methylation_case_uuids_match": (
            qc_annotated_pair_inventory["rna_case_uuid"]
            .eq(qc_annotated_pair_inventory["methylation_case_uuid"])
            .all()
        ),
    }
)


print("Integration input-contract checks:")
for check_name, check_passed in integration_contract_checks.items():
    print(f"{check_name}: {check_passed}")


if not all(integration_contract_checks.values()):
    failed_checks = [
        check_name
        for check_name, check_passed in integration_contract_checks.items()
        if not check_passed
    ]

    raise ValueError(
        "Integration input-contract validation failed: "
        + ", ".join(failed_checks)
    )

print("\nIntegration input contracts validated.")

Integration input-contract checks:
qc_annotated_pair_inventory_identifiers_are_complete: True
rna_file_qc_inventory_identifiers_are_complete: True
methylation_file_qc_metrics_identifiers_are_complete: True
probe_level_qc_metrics_identifiers_are_complete: True
shared_probe_qc_inventory_identifiers_are_complete: True
rna_file_index_identifiers_are_complete: True
methylation_file_index_identifiers_are_complete: True
qc_annotated_pair_inventory_boolean_gates_are_complete: True
rna_file_qc_inventory_boolean_gates_are_complete: True
probe_level_qc_metrics_boolean_gates_are_complete: True
shared_probe_qc_inventory_boolean_gates_are_complete: True
candidate_file_pairs_are_unique: True
rna_file_qc_inventory_has_one_row_per_file: True
methylation_file_qc_metrics_have_one_row_per_file: True
probe_metrics_have_unique_platform_probe_pairs: True
shared_probe_inventory_has_unique_probe_ids: True
shared_probe_orders_are_unique: True
rna_file_index_has_unique_file_ids: True
methylation_file_index_has_u

In [7]:
# =============================================================================
# Reconcile QC file inventories with frozen file indexes
# =============================================================================


# -----------------------------------------------------------------------------
# RNA-seq QC inventory versus frozen RNA-seq file index
# -----------------------------------------------------------------------------

rna_qc_file_map = (
    rna_file_qc_inventory[
        [
            "rna_file_id",
            "matrix_column_index",
        ]
    ]
    .copy()
)

rna_qc_file_map["rna_file_id"] = (
    rna_qc_file_map["rna_file_id"].astype("string")
)

rna_index_file_map = (
    rna_file_index[
        [
            "file_id",
        ]
    ]
    .rename(
        columns={
            "file_id": "rna_file_id",
        }
    )
    .copy()
)

rna_index_file_map["rna_file_id"] = (
    rna_index_file_map["rna_file_id"].astype("string")
)

rna_qc_index_reconciliation = (
    rna_qc_file_map
    .merge(
        rna_index_file_map,
        on="rna_file_id",
        how="left",
        validate="one_to_one",
        indicator=True,
    )
)


# -----------------------------------------------------------------------------
# DNA-methylation QC inventory versus frozen methylation file index
# -----------------------------------------------------------------------------

methylation_qc_file_map = (
    methylation_file_qc_metrics[
        [
            "methylation_file_id",
            "methylation_platform",
        ]
    ]
    .copy()
)

methylation_qc_file_map[
    "methylation_file_id"
] = (
    methylation_qc_file_map[
        "methylation_file_id"
    ].astype("string")
)

methylation_qc_file_map[
    "methylation_platform"
] = (
    methylation_qc_file_map[
        "methylation_platform"
    ].astype("string")
)

methylation_index_file_map = (
    methylation_file_index[
        [
            "file_id",
            "platform",
        ]
    ]
    .rename(
        columns={
            "file_id": "methylation_file_id",
            "platform": "index_platform",
        }
    )
    .copy()
)

methylation_index_file_map[
    "methylation_file_id"
] = (
    methylation_index_file_map[
        "methylation_file_id"
    ].astype("string")
)

methylation_index_file_map[
    "index_platform"
] = (
    methylation_index_file_map[
        "index_platform"
    ].astype("string")
)

methylation_qc_index_reconciliation = (
    methylation_qc_file_map
    .merge(
        methylation_index_file_map,
        on="methylation_file_id",
        how="left",
        validate="one_to_one",
        indicator=True,
    )
)


# -----------------------------------------------------------------------------
# Validate mappings and matrix-column correspondence
# -----------------------------------------------------------------------------

rna_matrix_column_indices = pd.to_numeric(
    rna_file_qc_inventory[
        "matrix_column_index"
    ],
    errors="raise",
).astype("int64")

expected_rna_matrix_column_indices = list(
    range(len(rna_file_qc_inventory))
)

reconciliation_checks = {
    "rna_qc_file_ids_are_unique": (
        rna_qc_file_map["rna_file_id"].is_unique
    ),
    "methylation_qc_file_ids_are_unique": (
        methylation_qc_file_map[
            "methylation_file_id"
        ].is_unique
    ),
    "all_rna_qc_files_match_frozen_index": (
        rna_qc_index_reconciliation[
            "_merge"
        ]
        .eq("both")
        .all()
    ),
    "all_methylation_qc_files_match_frozen_index": (
        methylation_qc_index_reconciliation[
            "_merge"
        ]
        .eq("both")
        .all()
    ),
    "methylation_platforms_match_frozen_index": (
        methylation_qc_index_reconciliation[
            "methylation_platform"
        ]
        .eq(
            methylation_qc_index_reconciliation[
                "index_platform"
            ]
        )
        .fillna(False)
        .all()
    ),
    "rna_matrix_column_indices_are_unique": (
        rna_matrix_column_indices.is_unique
    ),
    "rna_matrix_column_indices_are_complete": (
        sorted(
            rna_matrix_column_indices.tolist()
        )
        == expected_rna_matrix_column_indices
    ),
    "rna_matrix_column_count_matches_qc_inventory": (
        rna_raw_counts_metadata[
            "matrix"
        ][
            "shape"
        ][1]
        == len(rna_file_qc_inventory)
    ),
}


print("QC-to-frozen-index reconciliation checks:")
for check_name, check_passed in reconciliation_checks.items():
    print(f"{check_name}: {check_passed}")


if not all(reconciliation_checks.values()):
    failed_checks = [
        check_name
        for check_name, check_passed
        in reconciliation_checks.items()
        if not check_passed
    ]

    raise ValueError(
        "QC-to-frozen-index reconciliation failed: "
        + ", ".join(failed_checks)
    )


print("\nQC file inventories reconciled with frozen indexes.")
print(
    f"RNA-seq QC files reconciled:       "
    f"{len(rna_qc_file_map):,}"
)
print(
    f"Methylation QC files reconciled:   "
    f"{len(methylation_qc_file_map):,}"
)
print(
    f"RNA-seq matrix columns verified:   "
    f"{len(rna_matrix_column_indices):,}"
)

QC-to-frozen-index reconciliation checks:
rna_qc_file_ids_are_unique: True
methylation_qc_file_ids_are_unique: True
all_rna_qc_files_match_frozen_index: True
all_methylation_qc_files_match_frozen_index: True
methylation_platforms_match_frozen_index: True
rna_matrix_column_indices_are_unique: True
rna_matrix_column_indices_are_complete: True
rna_matrix_column_count_matches_qc_inventory: True

QC file inventories reconciled with frozen indexes.
RNA-seq QC files reconciled:       10,122
Methylation QC files reconciled:   10,038
RNA-seq matrix columns verified:   10,122


In [8]:
# =============================================================================
# Reconcile candidate pairs with file-level QC inventories
# =============================================================================


candidate_pair_qc_reconciliation = (
    qc_annotated_pair_inventory[
        [
            "rna_file_id",
            "methylation_file_id",
            "rna_qc_eligible_for_downstream_selection",
            "methylation_qc_eligible_for_downstream_selection",
            "pair_eligible_after_rna_qc",
            "pair_eligible_after_methylation_qc",
            "pair_methylation_qc_status",
        ]
    ]
    .copy()
)

candidate_pair_qc_reconciliation["rna_file_id"] = (
    candidate_pair_qc_reconciliation["rna_file_id"].astype("string")
)

candidate_pair_qc_reconciliation["methylation_file_id"] = (
    candidate_pair_qc_reconciliation["methylation_file_id"].astype("string")
)


rna_qc_gate_map = (
    rna_file_qc_inventory[
        [
            "rna_file_id",
            "rna_qc_eligible_for_downstream_selection",
        ]
    ]
    .rename(
        columns={
            "rna_qc_eligible_for_downstream_selection": (
                "rna_qc_eligible_from_file_inventory"
            )
        }
    )
    .copy()
)

rna_qc_gate_map["rna_file_id"] = (
    rna_qc_gate_map["rna_file_id"].astype("string")
)


methylation_qc_file_map = (
    methylation_file_qc_metrics[
        ["methylation_file_id"]
    ]
    .copy()
)

methylation_qc_file_map["methylation_file_id"] = (
    methylation_qc_file_map["methylation_file_id"].astype("string")
)


candidate_pair_qc_reconciliation = (
    candidate_pair_qc_reconciliation
    .merge(
        rna_qc_gate_map,
        on="rna_file_id",
        how="left",
        validate="many_to_one",
        indicator="rna_qc_file_match",
    )
    .merge(
        methylation_qc_file_map,
        on="methylation_file_id",
        how="left",
        validate="many_to_one",
        indicator="methylation_qc_file_match",
    )
)


expected_pair_final_gate = (
    candidate_pair_qc_reconciliation[
        "pair_eligible_after_rna_qc"
    ]
    & candidate_pair_qc_reconciliation[
        "methylation_qc_eligible_for_downstream_selection"
    ]
)

expected_pair_status = np.select(
    [
        ~candidate_pair_qc_reconciliation[
            "pair_eligible_after_rna_qc"
        ],
        ~candidate_pair_qc_reconciliation[
            "methylation_qc_eligible_for_downstream_selection"
        ],
    ],
    [
        "Ineligible candidate pair — RNA-seq QC",
        "Ineligible candidate pair — methylation QC",
    ],
    default=(
        "Eligible candidate pair after RNA-seq "
        "and methylation QC"
    ),
)


candidate_pair_reconciliation_checks = {
    "candidate_pair_row_count_is_preserved": (
        len(candidate_pair_qc_reconciliation)
        == len(qc_annotated_pair_inventory)
    ),
    "all_candidate_rna_files_match_qc_inventory": (
        candidate_pair_qc_reconciliation[
            "rna_qc_file_match"
        ]
        .eq("both")
        .all()
    ),
    "all_candidate_methylation_files_match_qc_inventory": (
        candidate_pair_qc_reconciliation[
            "methylation_qc_file_match"
        ]
        .eq("both")
        .all()
    ),
    "pair_rna_gates_match_file_qc_inventory": (
        candidate_pair_qc_reconciliation[
            "rna_qc_eligible_for_downstream_selection"
        ]
        .eq(
            candidate_pair_qc_reconciliation[
                "rna_qc_eligible_from_file_inventory"
            ]
        )
        .all()
    ),
    "pair_final_gate_is_cumulative": (
        candidate_pair_qc_reconciliation[
            "pair_eligible_after_methylation_qc"
        ]
        .eq(expected_pair_final_gate)
        .all()
    ),
    "pair_status_matches_final_gate_logic": (
        candidate_pair_qc_reconciliation[
            "pair_methylation_qc_status"
        ]
        .eq(expected_pair_status)
        .all()
    ),
}


print("Candidate-pair QC reconciliation checks:")
for check_name, check_passed in candidate_pair_reconciliation_checks.items():
    print(f"{check_name}: {check_passed}")


if not all(candidate_pair_reconciliation_checks.values()):
    failed_checks = [
        check_name
        for check_name, check_passed
        in candidate_pair_reconciliation_checks.items()
        if not check_passed
    ]

    raise ValueError(
        "Candidate-pair QC reconciliation failed: "
        + ", ".join(failed_checks)
    )


print("\nCandidate-pair QC reconciliation completed.")
print(
    f"Candidate pairs reconciled: "
    f"{len(candidate_pair_qc_reconciliation):,}"
)
print(
    "QC-eligible candidate pairs: "
    f"{int(candidate_pair_qc_reconciliation['pair_eligible_after_methylation_qc'].sum()):,}"
)
print(
    "QC-ineligible candidate pairs: "
    f"{int((~candidate_pair_qc_reconciliation['pair_eligible_after_methylation_qc']).sum()):,}"
)

Candidate-pair QC reconciliation checks:
candidate_pair_row_count_is_preserved: True
all_candidate_rna_files_match_qc_inventory: True
all_candidate_methylation_files_match_qc_inventory: True
pair_rna_gates_match_file_qc_inventory: True
pair_final_gate_is_cumulative: True
pair_status_matches_final_gate_logic: True

Candidate-pair QC reconciliation completed.
Candidate pairs reconciled: 10,162
QC-eligible candidate pairs: 10,154
QC-ineligible candidate pairs: 8


In [9]:
# =============================================================================
# Characterize residual multiplicity after file-level QC
# =============================================================================


CASE_KEY = "case_submitter_id"
SAMPLE_KEY = "sample_submitter_id"
RNA_FILE_KEY = "rna_file_id"
METHYLATION_FILE_KEY = "methylation_file_id"
PLATFORM_KEY = "methylation_platform"
FINAL_GATE_KEY = "pair_eligible_after_methylation_qc"


multiplicity_input = qc_annotated_pair_inventory[
    [
        CASE_KEY,
        SAMPLE_KEY,
        RNA_FILE_KEY,
        METHYLATION_FILE_KEY,
        PLATFORM_KEY,
        FINAL_GATE_KEY,
    ]
].copy()


multiplicity_input[CASE_KEY] = multiplicity_input[CASE_KEY].astype("string")
multiplicity_input[SAMPLE_KEY] = multiplicity_input[SAMPLE_KEY].astype("string")
multiplicity_input[RNA_FILE_KEY] = multiplicity_input[RNA_FILE_KEY].astype("string")
multiplicity_input[METHYLATION_FILE_KEY] = (
    multiplicity_input[METHYLATION_FILE_KEY].astype("string")
)
multiplicity_input[PLATFORM_KEY] = (
    multiplicity_input[PLATFORM_KEY].astype("string")
)


multiplicity_checks = {
    "candidate_pair_row_count_matches_reconciled_inventory": (
        len(multiplicity_input)
        == len(candidate_pair_qc_reconciliation)
    ),
    "candidate_pair_identity_order_is_preserved": (
        multiplicity_input[
            [RNA_FILE_KEY, METHYLATION_FILE_KEY]
        ]
        .reset_index(drop=True)
        .equals(
            candidate_pair_qc_reconciliation[
                [RNA_FILE_KEY, METHYLATION_FILE_KEY]
            ]
            .astype("string")
            .reset_index(drop=True)
        )
    ),
    "multiplicity_identifiers_are_complete": (
        multiplicity_input[
            [
                CASE_KEY,
                SAMPLE_KEY,
                RNA_FILE_KEY,
                METHYLATION_FILE_KEY,
                PLATFORM_KEY,
            ]
        ]
        .notna()
        .all()
        .all()
    ),
    "candidate_pair_keys_are_unique": (
        not multiplicity_input[
            [RNA_FILE_KEY, METHYLATION_FILE_KEY]
        ]
        .duplicated()
        .any()
    ),
    "final_gate_is_boolean_and_complete": (
        pd.api.types.is_bool_dtype(
            multiplicity_input[FINAL_GATE_KEY]
        )
        and multiplicity_input[FINAL_GATE_KEY].notna().all()
    ),
    "eligible_pair_count_matches_reconciliation": (
        int(multiplicity_input[FINAL_GATE_KEY].sum())
        == int(
            candidate_pair_qc_reconciliation[
                FINAL_GATE_KEY
            ].sum()
        )
    ),
}


print("Multiplicity-input checks:")
for check_name, check_passed in multiplicity_checks.items():
    print(f"{check_name}: {check_passed}")


if not all(multiplicity_checks.values()):
    failed_checks = [
        check_name
        for check_name, check_passed in multiplicity_checks.items()
        if not check_passed
    ]

    raise ValueError(
        "Multiplicity-input validation failed: "
        + ", ".join(failed_checks)
    )


# -----------------------------------------------------------------------------
# Restrict the multiplicity analysis to QC-eligible candidate pairs
# -----------------------------------------------------------------------------

eligible_candidate_pairs = (
    multiplicity_input.loc[
        multiplicity_input[FINAL_GATE_KEY]
    ]
    .copy()
)


all_candidate_cases = (
    multiplicity_input[
        [CASE_KEY]
    ]
    .drop_duplicates()
)


eligible_case_summary = (
    eligible_candidate_pairs
    .groupby(CASE_KEY, as_index=False)
    .agg(
        eligible_pair_count=(
            RNA_FILE_KEY,
            "size",
        ),
        eligible_sample_count=(
            SAMPLE_KEY,
            "nunique",
        ),
        eligible_rna_file_count=(
            RNA_FILE_KEY,
            "nunique",
        ),
        eligible_methylation_file_count=(
            METHYLATION_FILE_KEY,
            "nunique",
        ),
        eligible_platform_count=(
            PLATFORM_KEY,
            "nunique",
        ),
    )
)


case_multiplicity_summary = (
    all_candidate_cases
    .merge(
        eligible_case_summary,
        on=CASE_KEY,
        how="left",
        validate="one_to_one",
    )
)


count_columns = [
    "eligible_pair_count",
    "eligible_sample_count",
    "eligible_rna_file_count",
    "eligible_methylation_file_count",
    "eligible_platform_count",
]

case_multiplicity_summary[count_columns] = (
    case_multiplicity_summary[count_columns]
    .fillna(0)
    .astype("int64")
)


case_multiplicity_summary["has_eligible_pair"] = (
    case_multiplicity_summary["eligible_pair_count"] > 0
)

case_multiplicity_summary["has_multiple_eligible_pairs"] = (
    case_multiplicity_summary["eligible_pair_count"] > 1
)

case_multiplicity_summary["has_multiple_eligible_samples"] = (
    case_multiplicity_summary["eligible_sample_count"] > 1
)


case_multiplicity_summary["case_multiplicity_status"] = np.select(
    [
        ~case_multiplicity_summary["has_eligible_pair"],
        case_multiplicity_summary["has_multiple_eligible_samples"],
        case_multiplicity_summary["has_multiple_eligible_pairs"],
    ],
    [
        "No eligible pair after QC",
        "Multiple eligible samples within case",
        "Multiple eligible pairs within one sample",
    ],
    default="One eligible pair within one sample",
)


# -----------------------------------------------------------------------------
# Characterize multiplicity at the case-sample level
# -----------------------------------------------------------------------------

case_sample_multiplicity_summary = (
    eligible_candidate_pairs
    .groupby(
        [CASE_KEY, SAMPLE_KEY],
        as_index=False,
    )
    .agg(
        eligible_pair_count=(
            RNA_FILE_KEY,
            "size",
        ),
        eligible_rna_file_count=(
            RNA_FILE_KEY,
            "nunique",
        ),
        eligible_methylation_file_count=(
            METHYLATION_FILE_KEY,
            "nunique",
        ),
        eligible_platform_count=(
            PLATFORM_KEY,
            "nunique",
        ),
    )
)


multiplicity_summary_checks = {
    "candidate_case_count_is_preserved": (
        case_multiplicity_summary[CASE_KEY].nunique()
        == multiplicity_input[CASE_KEY].nunique()
    ),
    "case_sample_summary_keys_are_unique": (
        not case_sample_multiplicity_summary[
            [CASE_KEY, SAMPLE_KEY]
        ].duplicated().any()
    ),
    "case_pair_counts_reconstruct_eligible_pairs": (
        int(
            case_multiplicity_summary[
                "eligible_pair_count"
            ].sum()
        )
        == len(eligible_candidate_pairs)
    ),
    "case_sample_pair_counts_reconstruct_eligible_pairs": (
        int(
            case_sample_multiplicity_summary[
                "eligible_pair_count"
            ].sum()
        )
        == len(eligible_candidate_pairs)
    ),
    "eligible_cases_are_represented_in_case_sample_summary": (
        set(
            case_multiplicity_summary.loc[
                case_multiplicity_summary["has_eligible_pair"],
                CASE_KEY,
            ].astype("string")
        )
        == set(
            case_sample_multiplicity_summary[
                CASE_KEY
            ].astype("string")
        )
    ),
}


print("\nMultiplicity-summary checks:")
for check_name, check_passed in multiplicity_summary_checks.items():
    print(f"{check_name}: {check_passed}")


print("\nResidual multiplicity characterization completed.")
print(
    f"Candidate cases:             "
    f"{multiplicity_input[CASE_KEY].nunique():,}"
)
print(
    f"Cases with eligible pairs:   "
    f"{int(case_multiplicity_summary['has_eligible_pair'].sum()):,}"
)
print(
    f"Cases without eligible pair: "
    f"{int((~case_multiplicity_summary['has_eligible_pair']).sum()):,}"
)
print(
    f"Eligible candidate pairs:    "
    f"{len(eligible_candidate_pairs):,}"
)
print(
    f"Case-sample combinations:    "
    f"{len(case_sample_multiplicity_summary):,}"
)

print("\nCase-level multiplicity statuses:")
print(
    case_multiplicity_summary[
        "case_multiplicity_status"
    ]
    .value_counts()
    .sort_index()
)

print("\nEligible-pair count distribution per case:")
print(
    case_multiplicity_summary[
        "eligible_pair_count"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("eligible_pair_count")
    .rename("case_count")
    .reset_index()
)

if not all(multiplicity_summary_checks.values()):
    failed_checks = [
        check_name
        for check_name, check_passed
        in multiplicity_summary_checks.items()
        if not check_passed
    ]

    raise ValueError(
        "Residual multiplicity characterization failed: "
        + ", ".join(failed_checks)
    )

Multiplicity-input checks:
candidate_pair_row_count_matches_reconciled_inventory: True
candidate_pair_identity_order_is_preserved: True
multiplicity_identifiers_are_complete: True
candidate_pair_keys_are_unique: True
final_gate_is_boolean_and_complete: True
eligible_pair_count_matches_reconciliation: True

Multiplicity-summary checks:
candidate_case_count_is_preserved: True
case_sample_summary_keys_are_unique: True
case_pair_counts_reconstruct_eligible_pairs: True
case_sample_pair_counts_reconstruct_eligible_pairs: True
eligible_cases_are_represented_in_case_sample_summary: True

Residual multiplicity characterization completed.
Candidate cases:             9,973
Cases with eligible pairs:   9,973
Cases without eligible pair: 0
Eligible candidate pairs:    10,154
Case-sample combinations:    10,017

Case-level multiplicity statuses:
case_multiplicity_status
Multiple eligible pairs within one sample      72
Multiple eligible samples within case          44
One eligible pair within one s

In [10]:
# =============================================================================
# Characterize residual ambiguity sources after file-level QC
# =============================================================================

def join_unique_strings(values):
    return " | ".join(
        sorted(
            set(
                values.astype("string")
                .dropna()
                .tolist()
            )
        )
    )


case_sample_residual_diagnostic = (
    eligible_candidate_pairs
    .groupby(
        [CASE_KEY, SAMPLE_KEY],
        as_index=False,
        sort=True,
    )
    .agg(
        eligible_pair_count=(
            RNA_FILE_KEY,
            "size",
        ),
        eligible_rna_file_count=(
            RNA_FILE_KEY,
            "nunique",
        ),
        eligible_methylation_file_count=(
            METHYLATION_FILE_KEY,
            "nunique",
        ),
        eligible_platform_count=(
            PLATFORM_KEY,
            "nunique",
        ),
        eligible_platforms=(
            PLATFORM_KEY,
            join_unique_strings,
        ),
    )
)


case_residual_diagnostic = (
    case_sample_residual_diagnostic
    .groupby(
        CASE_KEY,
        as_index=False,
        sort=True,
    )
    .agg(
        eligible_sample_count=(
            SAMPLE_KEY,
            "nunique",
        ),
        eligible_pair_count=(
            "eligible_pair_count",
            "sum",
        ),
        n_samples_with_multiple_pairs=(
            "eligible_pair_count",
            lambda values: int(values.gt(1).sum()),
        ),
        n_samples_with_multiple_platforms=(
            "eligible_platform_count",
            lambda values: int(values.gt(1).sum()),
        ),
        maximum_pairs_within_one_sample=(
            "eligible_pair_count",
            "max",
        ),
    )
    .merge(
        eligible_candidate_pairs
        .groupby(
            CASE_KEY,
            as_index=False,
        )
        .agg(
            eligible_platforms=(
                PLATFORM_KEY,
                join_unique_strings,
            )
        ),
        on=CASE_KEY,
        how="left",
        validate="one_to_one",
    )
)


case_residual_diagnostic["residual_ambiguity_source"] = (
    case_residual_diagnostic.apply(
        lambda row: " + ".join(
            source
            for source, condition in [
                (
                    "multiple eligible samples",
                    row["eligible_sample_count"] > 1,
                ),
                (
                    "multiple pairs within one sample",
                    row["n_samples_with_multiple_pairs"] > 0,
                ),
            ]
            if condition
        )
        or "no residual ambiguity",
        axis=1,
    )
    .astype("string")
)


residual_ambiguity_checks = {
    "case_diagnostic_keys_are_unique": (
        case_residual_diagnostic[CASE_KEY].is_unique
    ),
    "case_sample_diagnostic_keys_are_unique": (
        not case_sample_residual_diagnostic[
            [CASE_KEY, SAMPLE_KEY]
        ].duplicated().any()
    ),
    "case_pair_counts_reconstruct_eligible_pairs": (
        int(
            case_residual_diagnostic[
                "eligible_pair_count"
            ].sum()
        )
        == len(eligible_candidate_pairs)
    ),
    "case_sample_pair_counts_reconstruct_eligible_pairs": (
        int(
            case_sample_residual_diagnostic[
                "eligible_pair_count"
            ].sum()
        )
        == len(eligible_candidate_pairs)
    ),
    "ambiguous_case_count_matches_previous_summary": (
        int(
            case_residual_diagnostic[
                "eligible_pair_count"
            ].gt(1)
            .sum()
        )
        == int(
            case_multiplicity_summary[
                "eligible_pair_count"
            ].gt(1)
            .sum()
        )
    ),
    "all_ambiguous_cases_have_a_source": (
        case_residual_diagnostic.loc[
            case_residual_diagnostic[
                "eligible_pair_count"
            ].gt(1),
            "residual_ambiguity_source",
        ]
        .ne("")
        .all()
    ),
}


print("Residual-ambiguity checks:")
for check_name, check_passed in residual_ambiguity_checks.items():
    print(f"{check_name}: {check_passed}")


if not all(residual_ambiguity_checks.values()):
    failed_checks = [
        check_name
        for check_name, check_passed
        in residual_ambiguity_checks.items()
        if not check_passed
    ]

    raise ValueError(
        "Residual ambiguity characterization failed: "
        + ", ".join(failed_checks)
    )


ambiguous_case_diagnostic = (
    case_residual_diagnostic.loc[
        case_residual_diagnostic[
            "eligible_pair_count"
        ].gt(1)
    ]
    .copy()
)


print("\nResidual ambiguity characterization completed.")
print(
    f"Ambiguous cases: "
    f"{len(ambiguous_case_diagnostic):,}"
)
print(
    "Cases with multiple eligible samples: "
    f"{int(case_residual_diagnostic['eligible_sample_count'].gt(1).sum()):,}"
)
print(
    "Cases with within-sample pair multiplicity: "
    f"{int(case_residual_diagnostic['n_samples_with_multiple_pairs'].gt(0).sum()):,}"
)
both_ambiguity_sources = (
    (
        case_residual_diagnostic["eligible_sample_count"].gt(1)
    )
    & (
        case_residual_diagnostic[
            "n_samples_with_multiple_pairs"
        ].gt(0)
    )
).sum()

print(
    "Cases with both ambiguity sources: "
    f"{int(both_ambiguity_sources):,}"
)

print("\nAmbiguity-source distribution:")
print(
    ambiguous_case_diagnostic[
        "residual_ambiguity_source"
    ]
    .value_counts()
    .sort_index()
)

print("\nAmbiguous-case diagnostic preview:")
display(
    ambiguous_case_diagnostic.sort_values(
        [CASE_KEY],
        kind="stable",
    ).head(30)
)

Residual-ambiguity checks:
case_diagnostic_keys_are_unique: True
case_sample_diagnostic_keys_are_unique: True
case_pair_counts_reconstruct_eligible_pairs: True
case_sample_pair_counts_reconstruct_eligible_pairs: True
ambiguous_case_count_matches_previous_summary: True
all_ambiguous_cases_have_a_source: True

Residual ambiguity characterization completed.
Ambiguous cases: 116
Cases with multiple eligible samples: 44
Cases with within-sample pair multiplicity: 103
Cases with both ambiguity sources: 31

Ambiguity-source distribution:
residual_ambiguity_source
multiple eligible samples                                       13
multiple eligible samples + multiple pairs within one sample    31
multiple pairs within one sample                                72
Name: count, dtype: int64[pyarrow]

Ambiguous-case diagnostic preview:


,case_submitter_id,eligible_sample_count,eligible_pair_count,n_samples_with_multiple_pairs,n_samples_with_multiple_platforms,maximum_pairs_within_one_sample,eligible_platforms,residual_ambiguity_source
3,TCGA-02-0047,1,2,1,0,2,Illumina Human Methylation 27,multiple pairs within one sample
4,TCGA-02-0055,1,2,1,0,2,Illumina Human Methylation 27,multiple pairs within one sample
7,TCGA-02-2483,1,2,1,0,2,Illumina Human Methylation 27,multiple pairs within one sample
8,TCGA-02-2485,1,2,1,0,2,Illumina Human Methylation 27,multiple pairs within one sample
68,TCGA-06-0125,1,2,1,0,2,Illumina Human Methylation 450,multiple pairs within one sample
71,TCGA-06-0139,1,2,1,0,2,Illumina Human Methylation 27,multiple pairs within one sample
73,TCGA-06-0141,1,2,1,0,2,Illumina Human Methylation 27,multiple pairs within one sample
79,TCGA-06-0190,1,2,1,0,2,Illumina Human Methylation 450,multiple pairs within one sample
80,TCGA-06-0210,1,2,1,0,2,Illumina Human Methylation 450,multiple pairs within one sample
81,TCGA-06-0211,1,2,1,0,2,Illumina Human Methylation 450,multiple pairs within one sample


In [11]:
# =============================================================================
# Expand residual ambiguity to pair-level diagnostics
# =============================================================================

ambiguity_case_keys = set(
    ambiguous_case_diagnostic[CASE_KEY].astype("string")
)

pair_metadata_columns = [
    CASE_KEY,
    SAMPLE_KEY,
    RNA_FILE_KEY,
    METHYLATION_FILE_KEY,
    PLATFORM_KEY,
    "rna_file_name",
    "methylation_file_name",
]

source_pair_metadata = qc_annotated_pair_inventory[
    pair_metadata_columns
].copy()

for column in pair_metadata_columns:
    source_pair_metadata[column] = (
        source_pair_metadata[column].astype("string")
    )

ambiguous_pair_diagnostic = (
    eligible_candidate_pairs.loc[
        eligible_candidate_pairs[CASE_KEY].isin(ambiguity_case_keys)
    ]
    .merge(
        source_pair_metadata,
        on=[
            CASE_KEY,
            SAMPLE_KEY,
            RNA_FILE_KEY,
            METHYLATION_FILE_KEY,
            PLATFORM_KEY,
        ],
        how="left",
        validate="one_to_one",
        indicator="source_match",
    )
)

pair_level_ambiguity_checks = {
    "ambiguous_pairs_match_source_inventory": (
        ambiguous_pair_diagnostic["source_match"]
        .eq("both")
        .all()
    ),
    "ambiguous_pair_keys_are_unique": (
        not ambiguous_pair_diagnostic[
            [RNA_FILE_KEY, METHYLATION_FILE_KEY]
        ]
        .duplicated()
        .any()
    ),
    "ambiguous_pair_count_matches_case_summary": (
        len(ambiguous_pair_diagnostic)
        == int(
            ambiguous_case_diagnostic[
                "eligible_pair_count"
            ].sum()
        )
    ),
    "ambiguous_case_keys_are_unique": (
        ambiguous_pair_diagnostic[CASE_KEY]
        .drop_duplicates()
        .is_unique
    ),
}

print("Pair-level ambiguity checks:")
for check_name, check_passed in pair_level_ambiguity_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(pair_level_ambiguity_checks.values()):
    failed_checks = [
        check_name
        for check_name, check_passed
        in pair_level_ambiguity_checks.items()
        if not check_passed
    ]

    raise ValueError(
        "Pair-level ambiguity diagnostics failed: "
        + ", ".join(failed_checks)
    )


case_sample_pair_diagnostic = (
    ambiguous_pair_diagnostic
    .groupby(
        [CASE_KEY, SAMPLE_KEY],
        as_index=False,
        sort=True,
    )
    .agg(
        eligible_pair_count=(
            RNA_FILE_KEY,
            "size",
        ),
        unique_rna_file_count=(
            RNA_FILE_KEY,
            "nunique",
        ),
        unique_methylation_file_count=(
            METHYLATION_FILE_KEY,
            "nunique",
        ),
        eligible_platform_count=(
            PLATFORM_KEY,
            "nunique",
        ),
        rna_file_ids=(
            RNA_FILE_KEY,
            join_unique_strings,
        ),
        methylation_file_ids=(
            METHYLATION_FILE_KEY,
            join_unique_strings,
        ),
        rna_file_names=(
            "rna_file_name",
            join_unique_strings,
        ),
        methylation_file_names=(
            "methylation_file_name",
            join_unique_strings,
        ),
        eligible_platforms=(
            PLATFORM_KEY,
            join_unique_strings,
        ),
    )
)


case_sample_pair_diagnostic[
    "within_sample_pair_pattern"
] = np.select(
    [
        (
            case_sample_pair_diagnostic[
                "eligible_pair_count"
            ].eq(1)
        ),
        (
            case_sample_pair_diagnostic[
                "unique_rna_file_count"
            ].gt(1)
            & case_sample_pair_diagnostic[
                "unique_methylation_file_count"
            ].eq(1)
        ),
        (
            case_sample_pair_diagnostic[
                "unique_rna_file_count"
            ].eq(1)
            & case_sample_pair_diagnostic[
                "unique_methylation_file_count"
            ].gt(1)
        ),
        (
            case_sample_pair_diagnostic[
                "unique_rna_file_count"
            ].gt(1)
            & case_sample_pair_diagnostic[
                "unique_methylation_file_count"
            ].gt(1)
        ),
    ],
    [
        "one eligible pair",
        "multiple RNA-seq files with one methylation file",
        "one RNA-seq file with multiple methylation files",
        "multiple RNA-seq and methylation files",
    ],
    default="unclassified",
)


ambiguous_case_pair_summary = (
    case_sample_pair_diagnostic
    .groupby(
        CASE_KEY,
        as_index=False,
        sort=True,
    )
    .agg(
        eligible_sample_count=(
            SAMPLE_KEY,
            "nunique",
        ),
        eligible_pair_count=(
            "eligible_pair_count",
            "sum",
        ),
        n_samples_with_multiple_pairs=(
            "eligible_pair_count",
            lambda values: int(values.gt(1).sum()),
        ),
        n_samples_with_multiple_platforms=(
            "eligible_platform_count",
            lambda values: int(values.gt(1).sum()),
        ),
    )
)


expanded_ambiguity_checks = {
    "case_summary_keys_are_unique": (
        ambiguous_case_pair_summary[CASE_KEY].is_unique
    ),
    "ambiguous_case_count_is_preserved": (
        len(ambiguous_case_pair_summary)
        == len(ambiguous_case_diagnostic)
    ),
    "ambiguous_pair_counts_reconstruct_pairs": (
        int(
            ambiguous_case_pair_summary[
                "eligible_pair_count"
            ].sum()
        )
        == len(ambiguous_pair_diagnostic)
    ),
    "all_ambiguous_cases_have_pair_diagnostics": (
        set(
            ambiguous_case_pair_summary[
                CASE_KEY
            ].astype("string")
        )
        == ambiguity_case_keys
    ),
}

print("\nExpanded-ambiguity checks:")
for check_name, check_passed in expanded_ambiguity_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(expanded_ambiguity_checks.values()):
    failed_checks = [
        check_name
        for check_name, check_passed
        in expanded_ambiguity_checks.items()
        if not check_passed
    ]

    raise ValueError(
        "Expanded ambiguity characterization failed: "
        + ", ".join(failed_checks)
    )


print("\nWithin-sample pair-pattern distribution:")
print(
    case_sample_pair_diagnostic.loc[
        case_sample_pair_diagnostic[
            "eligible_pair_count"
        ].gt(1),
        "within_sample_pair_pattern",
    ]
    .value_counts()
    .sort_index()
)

print(
    "\nAmbiguous case-sample units with multiple platforms: "
    f"{int(case_sample_pair_diagnostic['eligible_platform_count'].gt(1).sum()):,}"
)

print("\nExpanded ambiguity diagnostic preview:")
display(
    case_sample_pair_diagnostic.sort_values(
        [CASE_KEY, SAMPLE_KEY],
        kind="stable",
    ).head(100)
)

Pair-level ambiguity checks:
ambiguous_pairs_match_source_inventory: True
ambiguous_pair_keys_are_unique: True
ambiguous_pair_count_matches_case_summary: True
ambiguous_case_keys_are_unique: True

Expanded-ambiguity checks:
case_summary_keys_are_unique: True
ambiguous_case_count_is_preserved: True
ambiguous_pair_counts_reconstruct_pairs: True
all_ambiguous_cases_have_pair_diagnostics: True

Within-sample pair-pattern distribution:
within_sample_pair_pattern
multiple RNA-seq and methylation files              18
multiple RNA-seq files with one methylation file    84
one RNA-seq file with multiple methylation files     1
Name: count, dtype: int64

Ambiguous case-sample units with multiple platforms: 0

Expanded ambiguity diagnostic preview:


,case_submitter_id,sample_submitter_id,eligible_pair_count,unique_rna_file_count,unique_methylation_file_count,eligible_platform_count,rna_file_ids,methylation_file_ids,rna_file_names,methylation_file_names,eligible_platforms,within_sample_pair_pattern
0,TCGA-02-0047,TCGA-02-0047-01A,2,2,1,1,6835ae68-2f18-4239-ac8c-6a8291101e45 | 9d15581...,6587bdc3-779e-4fa9-8e2e-aee332e3b675,4ebf51c8-f12e-4c83-9bba-414e5ba20e7c.rna_seq.a...,6aceb16e-2157-4e6d-b01a-035edac0f3f0.methylati...,Illumina Human Methylation 27,multiple RNA-seq files with one methylation file
1,TCGA-02-0055,TCGA-02-0055-01A,2,2,1,1,cadd9094-eb64-401c-9fd1-43db4cc9360a | e0da1f5...,4322eb77-fe07-4a04-a4f0-a0686c8a76f1,328b149f-1c8d-4350-8ed8-a666bdc6a3f9.rna_seq.a...,6512ecfb-7762-4deb-bf0a-83f3fb10bdd1.methylati...,Illumina Human Methylation 27,multiple RNA-seq files with one methylation file
2,TCGA-02-2483,TCGA-02-2483-01A,2,2,1,1,170ed38a-fdce-4c28-a8ed-982d6a21add0 | 5fcd927...,10bc3274-1cce-4fea-860c-1b1ea8fa3400,9e09a872-f5f7-4cc1-bb08-07045fcd0c28.rna_seq.a...,82da608d-2378-4b3b-8db5-8f30bd2eec13.methylati...,Illumina Human Methylation 27,multiple RNA-seq files with one methylation file
3,TCGA-02-2485,TCGA-02-2485-01A,2,2,1,1,67d9092b-ae85-4687-b092-3012dbfab47a | d396a72...,af0a1e7f-795b-4f0f-8b4a-92aac941af27,51cf388b-effc-44bd-bd78-5b25878561a2.rna_seq.a...,e234df16-8669-4cc3-b316-dc23f1bf2121.methylati...,Illumina Human Methylation 27,multiple RNA-seq files with one methylation file
4,TCGA-06-0125,TCGA-06-0125-01A,2,2,1,1,354c79ce-261d-4543-8868-3eee6d7455b7 | 6827c4a...,408cb2a0-d4c8-4f22-815c-f3fd9c7553a5,9c6e0acf-3b58-4c3f-99f1-460b0aa68693.rna_seq.a...,e78565c0-3d3c-4982-9283-7ea032803a67.methylati...,Illumina Human Methylation 450,multiple RNA-seq files with one methylation file
...,...,...,...,...,...,...,...,...,...,...,...,...
95,TCGA-A6-2684,TCGA-A6-2684-01A,4,2,2,1,97dbbc5d-67e7-43a0-bc8f-d3425e0f3ce4 | bfb9163...,33a2708a-6dfc-41c6-99ed-17d7516bee4c | d6587c6...,19645f04-0021-4b2d-ad56-b1342c6ca519.rna_seq.a...,63792c44-e011-4016-a836-9de0bf993299.methylati...,Illumina Human Methylation 450,multiple RNA-seq and methylation files
96,TCGA-A6-2684,TCGA-A6-2684-01C,1,1,1,1,b7aa491d-f50f-4f9b-b8ac-1572cd6cd1ef,416647c3-ed70-4c45-a2bf-25c92557d5b2,ba2b0b86-f7f1-47f7-b0fd-bec3f623febb.rna_seq.a...,746690ef-0353-448e-a358-039271e34c45.methylati...,Illumina Human Methylation 450,one eligible pair
97,TCGA-A6-3809,TCGA-A6-3809-01A,2,2,1,1,b3bec9bc-73c9-41c9-beb2-68a5f06ee2ef | c741f3f...,6852b9a2-de41-45a4-a63b-a4a415b5cac4,1e0f16e6-de1c-4d1e-b1f6-5fabbea1591b.rna_seq.a...,8ddd0988-5711-4a7c-9634-e1f626f59d40.methylati...,Illumina Human Methylation 450,multiple RNA-seq files with one methylation file
98,TCGA-A6-3809,TCGA-A6-3809-01B,1,1,1,1,97fb41bc-f5a9-4b70-b631-9da9ff44b5fd,8b6e4521-ed23-4ddd-96a4-941d36065418,c22c07cb-8215-47e8-9525-074fda8d7d59.rna_seq.a...,fdb6ffaf-3b68-473a-9f8b-81b375197215.methylati...,Illumina Human Methylation 450,one eligible pair


In [12]:
# =============================================================================
# Audit provenance of residual pair alternatives
# =============================================================================

FINAL_PAIR_GATE_KEY = "pair_eligible_after_methylation_qc"

CASE_KEY = "case_submitter_id"
SAMPLE_KEY = "sample_submitter_id"
RNA_FILE_KEY = "rna_file_id"
METHYLATION_FILE_KEY = "methylation_file_id"
PLATFORM_KEY = "methylation_platform"


selection_source_columns = [
    CASE_KEY,
    SAMPLE_KEY,
    RNA_FILE_KEY,
    METHYLATION_FILE_KEY,
    PLATFORM_KEY,
    "severe_qc_signal_count",
    "methylation_qc_review_flag",
]

selection_audit_input = (
    qc_annotated_pair_inventory.loc[
        qc_annotated_pair_inventory[FINAL_PAIR_GATE_KEY],
        selection_source_columns,
    ]
    .copy()
)

for column in [
    CASE_KEY,
    SAMPLE_KEY,
    RNA_FILE_KEY,
    METHYLATION_FILE_KEY,
    PLATFORM_KEY,
]:
    selection_audit_input[column] = (
        selection_audit_input[column].astype("string")
    )


# -----------------------------------------------------------------------------
# Frozen RNA-seq provenance
# -----------------------------------------------------------------------------

rna_provenance = (
    rna_file_index[
        [
            "file_id",
            "file_name",
            "project_id",
            "case_id",
            "sample_id",
        ]
    ]
    .rename(
        columns={
            "file_id": RNA_FILE_KEY,
            "file_name": "rna_index_file_name",
            "project_id": "rna_project_id",
            "case_id": "rna_case_submitter_id",
            "sample_id": "rna_sample_submitter_id",
        }
    )
    .copy()
)

for column in rna_provenance.columns:
    rna_provenance[column] = (
        rna_provenance[column].astype("string")
    )


# -----------------------------------------------------------------------------
# Frozen methylation provenance
# -----------------------------------------------------------------------------

methylation_provenance = (
    methylation_file_index[
        [
            "file_id",
            "platform",
            "project_id",
            "case_uuid",
            "case_submitter_id",
            "sample_submitter_id",
            "aliquot_uuid",
            "aliquot_submitter_id",
            "analysis_id",
        ]
    ]
    .rename(
        columns={
            "file_id": METHYLATION_FILE_KEY,
            "platform": "methylation_index_platform",
            "project_id": "methylation_project_id",
            "case_uuid": "methylation_index_case_uuid",
            "case_submitter_id": "methylation_index_case_submitter_id",
            "sample_submitter_id": "methylation_index_sample_submitter_id",
            "aliquot_uuid": "methylation_aliquot_uuid",
            "aliquot_submitter_id": "methylation_aliquot_submitter_id",
            "analysis_id": "methylation_analysis_id",
        }
    )
    .copy()
)

for column in methylation_provenance.columns:
    methylation_provenance[column] = (
        methylation_provenance[column].astype("string")
    )


# -----------------------------------------------------------------------------
# File-level QC provenance
# -----------------------------------------------------------------------------

rna_qc_map = (
    rna_file_qc_inventory[
        [
            "rna_file_id",
            "matrix_column_index",
            "rna_qc_eligible_for_downstream_selection",
            "rna_qc_eligibility_status",
            "rna_qc_ineligibility_reason",
        ]
    ]
    .rename(
        columns={
            "rna_qc_eligible_for_downstream_selection": (
                "rna_qc_eligible_from_file_inventory"
            )
        }
    )
    .copy()
)

rna_qc_map["rna_file_id"] = (
    rna_qc_map["rna_file_id"].astype("string")
)


methylation_qc_map = (
    methylation_file_qc_metrics[
        [
            "methylation_file_id",
            "missing_beta_fraction",
            "row_count",
            "beta_min",
            "beta_median",
            "beta_max",
        ]
    ]
    .copy()
)

methylation_qc_map["methylation_file_id"] = (
    methylation_qc_map["methylation_file_id"].astype("string")
)


# -----------------------------------------------------------------------------
# Combine provenance layers
# -----------------------------------------------------------------------------

selection_audit_input = (
    selection_audit_input
    .merge(
        rna_provenance,
        on=RNA_FILE_KEY,
        how="left",
        validate="many_to_one",
        indicator="rna_index_match",
    )
    .merge(
        methylation_provenance,
        on=METHYLATION_FILE_KEY,
        how="left",
        validate="many_to_one",
        indicator="methylation_index_match",
    )
    .merge(
        rna_qc_map,
        on=RNA_FILE_KEY,
        how="left",
        validate="many_to_one",
        indicator="rna_qc_match",
    )
    .merge(
        methylation_qc_map,
        on=METHYLATION_FILE_KEY,
        how="left",
        validate="many_to_one",
        indicator="methylation_qc_match",
    )
)


# -----------------------------------------------------------------------------
# Validate provenance reconciliation
# -----------------------------------------------------------------------------

eligible_pair_keys = set(
    map(
        tuple,
        eligible_candidate_pairs[
            [
                RNA_FILE_KEY,
                METHYLATION_FILE_KEY,
            ]
        ]
        .astype("string")
        .itertuples(index=False, name=None),
    )
)

audit_pair_keys = set(
    map(
        tuple,
        selection_audit_input[
            [
                RNA_FILE_KEY,
                METHYLATION_FILE_KEY,
            ]
        ]
        .itertuples(index=False, name=None),
    )
)

selection_audit_checks = {
    "eligible_pair_count_is_preserved": (
        len(selection_audit_input)
        == len(eligible_candidate_pairs)
    ),
    "eligible_pair_identity_set_is_preserved": (
        audit_pair_keys == eligible_pair_keys
    ),
    "eligible_pair_keys_are_unique": (
        not selection_audit_input[
            [RNA_FILE_KEY, METHYLATION_FILE_KEY]
        ]
        .duplicated()
        .any()
    ),
    "all_rna_files_match_frozen_index": (
        selection_audit_input["rna_index_match"]
        .eq("both")
        .all()
    ),
    "all_methylation_files_match_frozen_index": (
        selection_audit_input["methylation_index_match"]
        .eq("both")
        .all()
    ),
    "all_rna_files_match_qc_inventory": (
        selection_audit_input["rna_qc_match"]
        .eq("both")
        .all()
    ),
    "all_methylation_files_match_qc_metrics": (
        selection_audit_input["methylation_qc_match"]
        .eq("both")
        .all()
    ),
    "case_ids_match_rna_provenance": (
        selection_audit_input[CASE_KEY]
        .eq(selection_audit_input["rna_case_submitter_id"])
        .all()
    ),
    "case_ids_match_methylation_provenance": (
        selection_audit_input[CASE_KEY]
        .eq(
            selection_audit_input[
                "methylation_index_case_submitter_id"
            ]
        )
        .all()
    ),
    "sample_ids_match_rna_provenance": (
        selection_audit_input[SAMPLE_KEY]
        .eq(
            selection_audit_input[
                "rna_sample_submitter_id"
            ]
        )
        .all()
    ),
    "sample_ids_match_methylation_provenance": (
        selection_audit_input[SAMPLE_KEY]
        .eq(
            selection_audit_input[
                "methylation_index_sample_submitter_id"
            ]
        )
        .all()
    ),
    "platforms_match_methylation_provenance": (
        selection_audit_input[PLATFORM_KEY]
        .eq(
            selection_audit_input[
                "methylation_index_platform"
            ]
        )
        .all()
    ),
    "rna_qc_gates_match_file_inventory": (
        selection_audit_input[
            "rna_qc_eligible_from_file_inventory"
        ]
        .eq(True)
        .all()
    ),
}

print("Residual-selection provenance checks:")
for check_name, check_passed in selection_audit_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(selection_audit_checks.values()):
    failed_checks = [
        check_name
        for check_name, check_passed in selection_audit_checks.items()
        if not check_passed
    ]

    raise ValueError(
        "Residual-selection provenance audit failed: "
        + ", ".join(failed_checks)
    )


# -----------------------------------------------------------------------------
# Summarize alternatives within each case-sample unit
# -----------------------------------------------------------------------------

case_sample_selection_audit = (
    selection_audit_input
    .groupby(
        [CASE_KEY, SAMPLE_KEY],
        as_index=False,
        sort=True,
    )
    .agg(
        eligible_pair_count=(RNA_FILE_KEY, "size"),
        unique_rna_file_count=(RNA_FILE_KEY, "nunique"),
        unique_methylation_file_count=(
            METHYLATION_FILE_KEY,
            "nunique",
        ),
        unique_methylation_aliquot_count=(
            "methylation_aliquot_uuid",
            "nunique",
        ),
        unique_methylation_analysis_count=(
            "methylation_analysis_id",
            "nunique",
        ),
        methylation_missing_fraction_min=(
            "missing_beta_fraction",
            "min",
        ),
        methylation_missing_fraction_max=(
            "missing_beta_fraction",
            "max",
        ),
        severe_qc_signal_count_min=(
            "severe_qc_signal_count",
            "min",
        ),
        severe_qc_signal_count_max=(
            "severe_qc_signal_count",
            "max",
        ),
    )
)

case_sample_selection_audit["within_sample_pair_pattern"] = (
    np.select(
        [
            (
                case_sample_selection_audit[
                    "unique_rna_file_count"
                ].gt(1)
                & case_sample_selection_audit[
                    "unique_methylation_file_count"
                ].eq(1)
            ),
            (
                case_sample_selection_audit[
                    "unique_rna_file_count"
                ].eq(1)
                & case_sample_selection_audit[
                    "unique_methylation_file_count"
                ].gt(1)
            ),
            (
                case_sample_selection_audit[
                    "unique_rna_file_count"
                ].gt(1)
                & case_sample_selection_audit[
                    "unique_methylation_file_count"
                ].gt(1)
            ),
        ],
        [
            "multiple RNA-seq files with one methylation file",
            "one RNA-seq file with multiple methylation files",
            "multiple RNA-seq and methylation files",
        ],
        default="one eligible pair",
    )
)


within_sample_ambiguity_audit = (
    case_sample_selection_audit.loc[
        case_sample_selection_audit[
            "eligible_pair_count"
        ].gt(1)
    ]
    .copy()
)


print("\nResidual-selection provenance audit completed.")
print(
    f"Eligible pairs audited: "
    f"{len(selection_audit_input):,}"
)
print(
    f"Within-sample ambiguous units: "
    f"{len(within_sample_ambiguity_audit):,}"
)

print("\nWithin-sample pair patterns:")
print(
    within_sample_ambiguity_audit[
        "within_sample_pair_pattern"
    ]
    .value_counts()
    .sort_index()
)

print("\nMethylation-file versus aliquot multiplicity:")
print(
    within_sample_ambiguity_audit[
        [
            "unique_methylation_file_count",
            "unique_methylation_aliquot_count",
        ]
    ]
    .value_counts()
    .sort_index()
)

print("\nWithin-sample provenance and QC preview:")
display(
    within_sample_ambiguity_audit.sort_values(
        [CASE_KEY, SAMPLE_KEY],
        kind="stable",
    ).head(100)
)

print(
    "\nNo pair selection or exclusion decision was applied."
)

Residual-selection provenance checks:
eligible_pair_count_is_preserved: True
eligible_pair_identity_set_is_preserved: True
eligible_pair_keys_are_unique: True
all_rna_files_match_frozen_index: True
all_methylation_files_match_frozen_index: True
all_rna_files_match_qc_inventory: True
all_methylation_files_match_qc_metrics: True
case_ids_match_rna_provenance: True
case_ids_match_methylation_provenance: True
sample_ids_match_rna_provenance: True
sample_ids_match_methylation_provenance: True
platforms_match_methylation_provenance: True
rna_qc_gates_match_file_inventory: True

Residual-selection provenance audit completed.
Eligible pairs audited: 10,154
Within-sample ambiguous units: 103

Within-sample pair patterns:
within_sample_pair_pattern
multiple RNA-seq and methylation files              18
multiple RNA-seq files with one methylation file    84
one RNA-seq file with multiple methylation files     1
Name: count, dtype: int64

Methylation-file versus aliquot multiplicity:
unique_methyl

,case_submitter_id,sample_submitter_id,eligible_pair_count,unique_rna_file_count,unique_methylation_file_count,unique_methylation_aliquot_count,unique_methylation_analysis_count,methylation_missing_fraction_min,methylation_missing_fraction_max,severe_qc_signal_count_min,severe_qc_signal_count_max,within_sample_pair_pattern
3,TCGA-02-0047,TCGA-02-0047-01A,2,2,1,1,1,0.140039,0.140039,0,0,multiple RNA-seq files with one methylation file
4,TCGA-02-0055,TCGA-02-0055-01A,2,2,1,1,1,0.110740,0.110740,0,0,multiple RNA-seq files with one methylation file
7,TCGA-02-2483,TCGA-02-2483-01A,2,2,1,1,1,0.088549,0.088549,0,0,multiple RNA-seq files with one methylation file
8,TCGA-02-2485,TCGA-02-2485-01A,2,2,1,1,1,0.089056,0.089056,0,0,multiple RNA-seq files with one methylation file
68,TCGA-06-0125,TCGA-06-0125-01A,2,2,1,1,1,0.139493,0.139493,0,0,multiple RNA-seq files with one methylation file
...,...,...,...,...,...,...,...,...,...,...,...,...
3312,TCGA-B2-5635,TCGA-B2-5635-01A,4,2,2,2,2,0.140342,0.140800,0,0,multiple RNA-seq and methylation files
3846,TCGA-BK-A0CA,TCGA-BK-A0CA-01A,2,2,1,1,1,0.186770,0.186770,0,0,multiple RNA-seq files with one methylation file
3849,TCGA-BK-A0CC,TCGA-BK-A0CC-01A,2,2,1,1,1,0.162018,0.162018,0,0,multiple RNA-seq files with one methylation file
3851,TCGA-BK-A139,TCGA-BK-A139-01A,2,2,1,1,1,0.152985,0.152985,0,0,multiple RNA-seq files with one methylation file



No pair selection or exclusion decision was applied.


In [13]:
print(
    within_sample_ambiguity_audit.sort_values(
        [CASE_KEY, SAMPLE_KEY],
        kind="stable",
    ).head(100)
)

print(
    "\nNo pair selection or exclusion decision was applied."
)

     case_submitter_id sample_submitter_id  eligible_pair_count  \
3         TCGA-02-0047    TCGA-02-0047-01A                    2   
4         TCGA-02-0055    TCGA-02-0055-01A                    2   
7         TCGA-02-2483    TCGA-02-2483-01A                    2   
8         TCGA-02-2485    TCGA-02-2485-01A                    2   
68        TCGA-06-0125    TCGA-06-0125-01A                    2   
...                ...                 ...                  ...   
3312      TCGA-B2-5635    TCGA-B2-5635-01A                    4   
3846      TCGA-BK-A0CA    TCGA-BK-A0CA-01A                    2   
3849      TCGA-BK-A0CC    TCGA-BK-A0CC-01A                    2   
3851      TCGA-BK-A139    TCGA-BK-A139-01A                    2   
3855      TCGA-BK-A26L    TCGA-BK-A26L-01A                    4   

      unique_rna_file_count  unique_methylation_file_count  \
3                         2                              1   
4                         2                              1   
7        

In [14]:
# =============================================================================
# Audit residual biospecimen and technical provenance
# =============================================================================

RNA_PROVENANCE_COLUMNS = [
    RNA_FILE_KEY,
    "rna_aliquot_uuid",
    "rna_aliquot_submitter_id",
    "rna_vial",
    "rna_portion_number",
    "rna_analyte_code",
]

missing_rna_provenance_columns = sorted(
    set(RNA_PROVENANCE_COLUMNS)
    - set(rna_file_qc_inventory.columns)
)

if missing_rna_provenance_columns:
    raise KeyError(
        "RNA QC inventory is missing provenance columns: "
        + ", ".join(missing_rna_provenance_columns)
    )


rna_provenance_detail = (
    rna_file_qc_inventory[
        RNA_PROVENANCE_COLUMNS
    ]
    .copy()
)


if not rna_provenance_detail[RNA_FILE_KEY].is_unique:
    raise ValueError(
        "RNA provenance detail must contain one row per RNA-seq file."
    )


for column in RNA_PROVENANCE_COLUMNS:
    rna_provenance_detail[column] = (
        rna_provenance_detail[column].astype("string")
    )


ambiguous_pair_provenance = (
    selection_audit_input
    .merge(
        within_sample_ambiguity_audit[
            [CASE_KEY, SAMPLE_KEY]
        ],
        on=[CASE_KEY, SAMPLE_KEY],
        how="inner",
        validate="many_to_one",
        indicator="ambiguity_unit_match",
    )
    .merge(
        rna_provenance_detail,
        on=RNA_FILE_KEY,
        how="left",
        validate="many_to_one",
        indicator="rna_provenance_match",
    )
)


provenance_checks = {
    "ambiguous_unit_keys_are_unique": (
        not within_sample_ambiguity_audit[
            [CASE_KEY, SAMPLE_KEY]
        ]
        .duplicated()
        .any()
    ),
    "all_ambiguous_pairs_match_ambiguity_units": (
        ambiguous_pair_provenance[
            "ambiguity_unit_match"
        ]
        .eq("both")
        .all()
    ),
    "all_ambiguous_pairs_match_rna_provenance": (
        ambiguous_pair_provenance[
            "rna_provenance_match"
        ]
        .eq("both")
        .all()
    ),
    "ambiguous_pair_count_matches_previous_audit": (
        len(ambiguous_pair_provenance)
        == int(
            within_sample_ambiguity_audit[
                "eligible_pair_count"
            ].sum()
        )
    ),
    "rna_provenance_identifiers_are_complete": (
        rna_provenance_detail[
            [
                "rna_aliquot_uuid",
                "rna_aliquot_submitter_id",
                "rna_vial",
                "rna_portion_number",
            ]
        ]
        .notna()
        .all()
        .all()
    ),
}


print("Residual biospecimen-provenance checks:")
for check_name, check_passed in provenance_checks.items():
    print(f"{check_name}: {check_passed}")


if not all(provenance_checks.values()):
    failed_checks = [
        check_name
        for check_name, check_passed
        in provenance_checks.items()
        if not check_passed
    ]

    raise ValueError(
        "Residual biospecimen-provenance audit failed: "
        + ", ".join(failed_checks)
    )


case_sample_provenance = (
    ambiguous_pair_provenance
    .groupby(
        [CASE_KEY, SAMPLE_KEY],
        as_index=False,
        sort=True,
    )
    .agg(
        eligible_pair_count=(
            RNA_FILE_KEY,
            "size",
        ),
        unique_rna_file_count=(
            RNA_FILE_KEY,
            "nunique",
        ),
        unique_rna_aliquot_count=(
            "rna_aliquot_uuid",
            lambda values: values.nunique(dropna=False),
        ),
        unique_rna_vial_count=(
            "rna_vial",
            lambda values: values.nunique(dropna=False),
        ),
        unique_rna_portion_count=(
            "rna_portion_number",
            lambda values: values.nunique(dropna=False),
        ),
        unique_rna_analyte_count=(
            "rna_analyte_code",
            lambda values: values.nunique(dropna=False),
        ),
        unique_methylation_file_count=(
            METHYLATION_FILE_KEY,
            "nunique",
        ),
        unique_methylation_aliquot_count=(
            "methylation_aliquot_uuid",
            lambda values: values.nunique(dropna=False),
        ),
        unique_methylation_analysis_count=(
            "methylation_analysis_id",
            lambda values: values.nunique(dropna=False),
        ),
        rna_aliquot_ids=(
            "rna_aliquot_submitter_id",
            join_unique_strings,
        ),
        rna_vials=(
            "rna_vial",
            join_unique_strings,
        ),
        rna_portions=(
            "rna_portion_number",
            join_unique_strings,
        ),
        methylation_aliquot_ids=(
            "methylation_aliquot_submitter_id",
            join_unique_strings,
        ),
        methylation_analysis_ids=(
            "methylation_analysis_id",
            join_unique_strings,
        ),
    )
)


case_sample_provenance[
    "rna_provenance_pattern"
] = np.select(
    [
        case_sample_provenance[
            "unique_rna_file_count"
        ].eq(1),
        (
            case_sample_provenance[
                "unique_rna_aliquot_count"
            ].eq(1)
            & case_sample_provenance[
                "unique_rna_file_count"
            ].gt(1)
        ),
        (
            case_sample_provenance[
                "unique_rna_aliquot_count"
            ].gt(1)
            & case_sample_provenance[
                "unique_rna_vial_count"
            ].eq(1)
            & case_sample_provenance[
                "unique_rna_portion_count"
            ].eq(1)
        ),
        (
            case_sample_provenance[
                "unique_rna_vial_count"
            ].gt(1)
            | case_sample_provenance[
                "unique_rna_portion_count"
            ].gt(1)
        ),
    ],
    [
        "one RNA-seq file",
        "multiple RNA-seq files from one aliquot",
        "multiple RNA aliquots within one vial and portion",
        "multiple RNA vials or portions",
    ],
    default="unclassified RNA provenance",
)


case_sample_provenance[
    "methylation_provenance_pattern"
] = np.select(
    [
        case_sample_provenance[
            "unique_methylation_file_count"
        ].eq(1),
        (
            case_sample_provenance[
                "unique_methylation_file_count"
            ].gt(1)
            & case_sample_provenance[
                "unique_methylation_aliquot_count"
            ].eq(1)
            & case_sample_provenance[
                "unique_methylation_analysis_count"
            ].eq(1)
        ),
        (
            case_sample_provenance[
                "unique_methylation_file_count"
            ].gt(1)
            & case_sample_provenance[
                "unique_methylation_aliquot_count"
            ].eq(1)
            & case_sample_provenance[
                "unique_methylation_analysis_count"
            ].gt(1)
        ),
        case_sample_provenance[
            "unique_methylation_aliquot_count"
        ].gt(1),
    ],
    [
        "one methylation file",
        "multiple files from one aliquot and analysis",
        "multiple analyses from one aliquot",
        "multiple methylation aliquots",
    ],
    default="unclassified methylation provenance",
)


summary_checks = {
    "case_sample_provenance_keys_are_unique": (
        not case_sample_provenance[
            [CASE_KEY, SAMPLE_KEY]
        ]
        .duplicated()
        .any()
    ),
    "case_sample_count_is_preserved": (
        len(case_sample_provenance)
        == len(within_sample_ambiguity_audit)
    ),
    "pair_counts_reconstruct_ambiguous_pairs": (
        int(
            case_sample_provenance[
                "eligible_pair_count"
            ].sum()
        )
        == len(ambiguous_pair_provenance)
    ),
}


print("\nResidual provenance-summary checks:")
for check_name, check_passed in summary_checks.items():
    print(f"{check_name}: {check_passed}")


if not all(summary_checks.values()):
    failed_checks = [
        check_name
        for check_name, check_passed
        in summary_checks.items()
        if not check_passed
    ]

    raise ValueError(
        "Residual provenance summary failed: "
        + ", ".join(failed_checks)
    )


print("\nRNA provenance patterns:")
print(
    case_sample_provenance[
        "rna_provenance_pattern"
    ]
    .value_counts()
    .sort_index()
)


print("\nMethylation provenance patterns:")
print(
    case_sample_provenance[
        "methylation_provenance_pattern"
    ]
    .value_counts()
    .sort_index()
)


print(
    "\nMethylation file, aliquot, and analysis multiplicity:"
)
print(
    case_sample_provenance[
        [
            "unique_methylation_file_count",
            "unique_methylation_aliquot_count",
            "unique_methylation_analysis_count",
        ]
    ]
    .value_counts()
    .sort_index()
)


print("\nResidual biospecimen-provenance preview:")
display(
    case_sample_provenance.sort_values(
        [CASE_KEY, SAMPLE_KEY],
        kind="stable",
    ).head(100)
)


print(
    "\nNo pair selection or exclusion decision was applied."
)

Residual biospecimen-provenance checks:
ambiguous_unit_keys_are_unique: True
all_ambiguous_pairs_match_ambiguity_units: True
all_ambiguous_pairs_match_rna_provenance: True
ambiguous_pair_count_matches_previous_audit: True
rna_provenance_identifiers_are_complete: True

Residual provenance-summary checks:
case_sample_provenance_keys_are_unique: True
case_sample_count_is_preserved: True
pair_counts_reconstruct_ambiguous_pairs: True

RNA provenance patterns:
rna_provenance_pattern
multiple RNA aliquots within one vial and portion    101
multiple RNA vials or portions                         1
one RNA-seq file                                       1
Name: count, dtype: int64

Methylation provenance patterns:
methylation_provenance_pattern
multiple methylation aliquots    19
one methylation file             84
Name: count, dtype: int64

Methylation file, aliquot, and analysis multiplicity:
unique_methylation_file_count  unique_methylation_aliquot_count  unique_methylation_analysis_count
1   

,case_submitter_id,sample_submitter_id,eligible_pair_count,unique_rna_file_count,unique_rna_aliquot_count,unique_rna_vial_count,unique_rna_portion_count,unique_rna_analyte_count,unique_methylation_file_count,unique_methylation_aliquot_count,unique_methylation_analysis_count,rna_aliquot_ids,rna_vials,rna_portions,methylation_aliquot_ids,methylation_analysis_ids,rna_provenance_pattern,methylation_provenance_pattern
0,TCGA-02-0047,TCGA-02-0047-01A,2,2,2,1,1,1,1,1,1,TCGA-02-0047-01A-01R-1849-01 | TCGA-02-0047-01...,A,1,TCGA-02-0047-01A-01D-0186-05,4b164db3-4a7a-4459-9026-faac81ea6263,multiple RNA aliquots within one vial and portion,one methylation file
1,TCGA-02-0055,TCGA-02-0055-01A,2,2,2,1,1,1,1,1,1,TCGA-02-0055-01A-01R-1849-01 | TCGA-02-0055-01...,A,1,TCGA-02-0055-01A-01D-0186-05,309ce05e-288d-494f-9e7c-218e481f7157,multiple RNA aliquots within one vial and portion,one methylation file
2,TCGA-02-2483,TCGA-02-2483-01A,2,2,2,1,1,1,1,1,1,TCGA-02-2483-01A-01R-1849-01 | TCGA-02-2483-01...,A,1,TCGA-02-2483-01A-01D-0788-05,af6f90b0-32dd-48f9-9d28-9caa752ba043,multiple RNA aliquots within one vial and portion,one methylation file
3,TCGA-02-2485,TCGA-02-2485-01A,2,2,2,1,1,1,1,1,1,TCGA-02-2485-01A-01R-1849-01 | TCGA-02-2485-01...,A,1,TCGA-02-2485-01A-01D-0788-05,35c0cc64-8aad-4134-bcf4-c06ca2507551,multiple RNA aliquots within one vial and portion,one methylation file
4,TCGA-06-0125,TCGA-06-0125-01A,2,2,2,1,1,1,1,1,1,TCGA-06-0125-01A-01R-1849-01 | TCGA-06-0125-01...,A,1,TCGA-06-0125-01A-01D-A45W-05,ca9bb126-3a55-48de-aa87-f4ba6ca8b7cc,multiple RNA aliquots within one vial and portion,one methylation file
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,TCGA-B2-5635,TCGA-B2-5635-01A,4,2,2,1,1,1,2,2,2,TCGA-B2-5635-01A-01R-1541-07 | TCGA-B2-5635-01...,A,1,TCGA-B2-5635-01A-01D-1536-05 | TCGA-B2-5635-01...,2a92ecdc-27a0-41bf-bd35-b0d1580c3f6b | a56ae91...,multiple RNA aliquots within one vial and portion,multiple methylation aliquots
96,TCGA-BK-A0CA,TCGA-BK-A0CA-01A,2,2,2,1,1,1,1,1,1,TCGA-BK-A0CA-01A-21R-A118-07 | TCGA-BK-A0CA-01...,A,21,TCGA-BK-A0CA-01A-21D-A27B-05,7d103432-df62-4a00-ada5-581bd5e3d484,multiple RNA aliquots within one vial and portion,one methylation file
97,TCGA-BK-A0CC,TCGA-BK-A0CC-01A,2,2,2,1,1,1,1,1,1,TCGA-BK-A0CC-01A-21R-A16W-07 | TCGA-BK-A0CC-01...,A,21,TCGA-BK-A0CC-01A-21D-A27B-05,26a14ef3-0a17-4954-b574-2ddc0e416708,multiple RNA aliquots within one vial and portion,one methylation file
98,TCGA-BK-A139,TCGA-BK-A139-01A,2,2,2,1,1,1,1,1,1,TCGA-BK-A139-01A-11R-A118-07 | TCGA-BK-A139-01...,A,11,TCGA-BK-A139-01A-11D-A27B-05,3a2d4513-6b23-4ba7-ad7a-1bb5429ebb85,multiple RNA aliquots within one vial and portion,one methylation file



No pair selection or exclusion decision was applied.


In [15]:
# =============================================================================
# Quantify QC contrast among residual biospecimen alternatives
# =============================================================================

RNA_QC_METRIC = "gene_assigned_fraction_of_accounted"
METHYLATION_QC_METRIC = "missing_beta_fraction"

rna_qc_map = rna_file_qc_inventory[
    [RNA_FILE_KEY, RNA_QC_METRIC]
].copy()

methylation_qc_map = methylation_file_qc_metrics[
    [METHYLATION_FILE_KEY, METHYLATION_QC_METRIC]
].copy()

for table, key in [
    (rna_qc_map, RNA_FILE_KEY),
    (methylation_qc_map, METHYLATION_FILE_KEY),
]:
    table[key] = table[key].astype("string")

qc_pair_input = (
    ambiguous_pair_provenance[
        [
            CASE_KEY,
            SAMPLE_KEY,
            RNA_FILE_KEY,
            METHYLATION_FILE_KEY,
        ]
    ]
    .drop_duplicates()
    .merge(
        rna_qc_map,
        on=RNA_FILE_KEY,
        how="left",
        validate="many_to_one",
    )
    .merge(
        methylation_qc_map,
        on=METHYLATION_FILE_KEY,
        how="left",
        validate="many_to_one",
    )
)

qc_contrast_checks = {
    "pair_count_is_preserved": (
        len(qc_pair_input)
        == len(
            ambiguous_pair_provenance[
                [RNA_FILE_KEY, METHYLATION_FILE_KEY]
            ]
        )
    ),
    "rna_qc_values_are_complete": (
        qc_pair_input[RNA_QC_METRIC].notna().all()
    ),
    "methylation_qc_values_are_complete": (
        qc_pair_input[METHYLATION_QC_METRIC].notna().all()
    ),
}

print("Residual QC-contrast checks:")
for check_name, check_passed in qc_contrast_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(qc_contrast_checks.values()):
    raise ValueError("QC contrast input validation failed.")

rna_contrast = (
    qc_pair_input[
        [CASE_KEY, SAMPLE_KEY, RNA_FILE_KEY, RNA_QC_METRIC]
    ]
    .drop_duplicates()
    .groupby([CASE_KEY, SAMPLE_KEY], as_index=False)
    .agg(
        rna_file_count=(RNA_FILE_KEY, "nunique"),
        rna_qc_max=(RNA_QC_METRIC, "max"),
        rna_qc_min=(RNA_QC_METRIC, "min"),
        rna_qc_winner_count=(
            RNA_QC_METRIC,
            lambda values: int(values.eq(values.max()).sum()),
        ),
    )
)

methylation_contrast = (
    qc_pair_input[
        [
            CASE_KEY,
            SAMPLE_KEY,
            METHYLATION_FILE_KEY,
            METHYLATION_QC_METRIC,
        ]
    ]
    .drop_duplicates()
    .groupby([CASE_KEY, SAMPLE_KEY], as_index=False)
    .agg(
        methylation_file_count=(
            METHYLATION_FILE_KEY,
            "nunique",
        ),
        methylation_qc_min=(METHYLATION_QC_METRIC, "min"),
        methylation_qc_max=(METHYLATION_QC_METRIC, "max"),
        methylation_qc_winner_count=(
            METHYLATION_QC_METRIC,
            lambda values: int(values.eq(values.min()).sum()),
        ),
    )
)

qc_contrast_summary = (
    within_sample_ambiguity_audit[
        [CASE_KEY, SAMPLE_KEY, "within_sample_pair_pattern"]
    ]
    .merge(
        rna_contrast,
        on=[CASE_KEY, SAMPLE_KEY],
        validate="one_to_one",
    )
    .merge(
        methylation_contrast,
        on=[CASE_KEY, SAMPLE_KEY],
        validate="one_to_one",
    )
)

print("\nRNA-seq QC winner structure:")
print(
    qc_contrast_summary.loc[
        qc_contrast_summary["rna_file_count"].gt(1),
        "rna_qc_winner_count",
    ]
    .value_counts()
    .sort_index()
)

print("\nMethylation QC winner structure:")
print(
    qc_contrast_summary.loc[
        qc_contrast_summary["methylation_file_count"].gt(1),
        "methylation_qc_winner_count",
    ]
    .value_counts()
    .sort_index()
)

print("\nNo pair selection or exclusion decision was applied.")

display(
    qc_contrast_summary.sort_values(
        [CASE_KEY, SAMPLE_KEY],
        kind="stable",
    ).head(100)
)

Residual QC-contrast checks:
pair_count_is_preserved: True
rna_qc_values_are_complete: True
methylation_qc_values_are_complete: True

RNA-seq QC winner structure:
rna_qc_winner_count
1    102
Name: count, dtype: int64

Methylation QC winner structure:
methylation_qc_winner_count
1    19
Name: count, dtype: int64

No pair selection or exclusion decision was applied.


,case_submitter_id,sample_submitter_id,within_sample_pair_pattern,rna_file_count,rna_qc_max,rna_qc_min,rna_qc_winner_count,methylation_file_count,methylation_qc_min,methylation_qc_max,methylation_qc_winner_count
0,TCGA-02-0047,TCGA-02-0047-01A,multiple RNA-seq files with one methylation file,2,0.669034,0.504533,1,1,0.140039,0.140039,1
1,TCGA-02-0055,TCGA-02-0055-01A,multiple RNA-seq files with one methylation file,2,0.722150,0.430964,1,1,0.110740,0.110740,1
2,TCGA-02-2483,TCGA-02-2483-01A,multiple RNA-seq files with one methylation file,2,0.734270,0.472548,1,1,0.088549,0.088549,1
3,TCGA-02-2485,TCGA-02-2485-01A,multiple RNA-seq files with one methylation file,2,0.735636,0.461921,1,1,0.089056,0.089056,1
4,TCGA-06-0125,TCGA-06-0125-01A,multiple RNA-seq files with one methylation file,2,0.729172,0.446686,1,1,0.139493,0.139493,1
...,...,...,...,...,...,...,...,...,...,...,...
95,TCGA-B2-5635,TCGA-B2-5635-01A,multiple RNA-seq and methylation files,2,0.694509,0.554971,1,2,0.140342,0.140800,1
96,TCGA-BK-A0CA,TCGA-BK-A0CA-01A,multiple RNA-seq files with one methylation file,2,0.722552,0.382382,1,1,0.186770,0.186770,1
97,TCGA-BK-A0CC,TCGA-BK-A0CC-01A,multiple RNA-seq files with one methylation file,2,0.769363,0.284889,1,1,0.162018,0.162018,1
98,TCGA-BK-A139,TCGA-BK-A139-01A,multiple RNA-seq files with one methylation file,2,0.759416,0.488237,1,1,0.152985,0.152985,1


## Confirmed within-sample alternative-selection policy

This policy resolves residual alternatives within each
`case_submitter_id`–`sample_submitter_id` unit.

- Select the RNA-seq file with the highest
  `gene_assigned_fraction_of_accounted`.
- Select the methylation file with the lowest
  `missing_beta_fraction`.
- Preserve all eligible alternatives and their provenance.
- Mark ties as `unresolved`.
- Do not use UUIDs, file names, file order, or platform as tie-breakers.
- This is an operational QC selection, not a claim that the selected
  aliquot is biologically superior.
- Final one-pair-per-case selection is deferred to a later stage.

In [16]:
# =============================================================================
# Freeze the within-sample selection policy
# =============================================================================

FINAL_PAIR_GATE_KEY = "pair_eligible_after_methylation_qc"

CASE_KEY = "case_submitter_id"
SAMPLE_KEY = "sample_submitter_id"
RNA_FILE_KEY = "rna_file_id"
METHYLATION_FILE_KEY = "methylation_file_id"

RNA_QC_METRIC = "gene_assigned_fraction_of_accounted"
METHYLATION_QC_METRIC = "missing_beta_fraction"

SELECTION_UNIT_KEYS = [CASE_KEY, SAMPLE_KEY]
SELECTION_PAIR_KEYS = [
    CASE_KEY,
    SAMPLE_KEY,
    RNA_FILE_KEY,
    METHYLATION_FILE_KEY,
]

WITHIN_SAMPLE_SELECTION_POLICY_RECORD = {
    "policy_id": "tcga_multiomic_within_sample_qc_v1",
    "scope": "case_submitter_id-sample_submitter_id unit",
    "rna_metric": RNA_QC_METRIC,
    "rna_rule": "highest_value",
    "methylation_metric": METHYLATION_QC_METRIC,
    "methylation_rule": "lowest_value",
    "tie_rule": "unresolved",
    "identifier_or_file_order_tiebreaking": False,
    "final_case_level_selection": "deferred",
}

required_selection_columns = (
    SELECTION_PAIR_KEYS
    + [FINAL_PAIR_GATE_KEY]
)

missing_selection_columns = sorted(
    set(required_selection_columns)
    - set(qc_annotated_pair_inventory.columns)
)

if missing_selection_columns:
    raise KeyError(
        "Missing columns for within-sample selection: "
        + ", ".join(missing_selection_columns)
    )

if not pd.api.types.is_bool_dtype(
    qc_annotated_pair_inventory[FINAL_PAIR_GATE_KEY]
):
    raise TypeError(
        f"{FINAL_PAIR_GATE_KEY} must be boolean."
    )

eligible_candidate_pairs = (
    qc_annotated_pair_inventory.loc[
        qc_annotated_pair_inventory[FINAL_PAIR_GATE_KEY]
    ]
    .copy()
)

for column in SELECTION_PAIR_KEYS:
    eligible_candidate_pairs[column] = (
        eligible_candidate_pairs[column].astype("string")
    )

selection_policy_checks = {
    "eligible_pairs_are_not_empty": (
        not eligible_candidate_pairs.empty
    ),
    "eligible_pair_identifiers_are_complete": (
        eligible_candidate_pairs[SELECTION_PAIR_KEYS]
        .notna()
        .all()
        .all()
    ),
    "eligible_pair_identity_is_unique": (
        not eligible_candidate_pairs[
            SELECTION_PAIR_KEYS
        ].duplicated().any()
    ),
}

print("Within-sample selection-policy checks:")
for check_name, check_passed in selection_policy_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(selection_policy_checks.values()):
    raise ValueError(
        "Within-sample selection-policy input validation failed."
    )

print(
    "\nWithin-sample selection policy frozen."
)
print(
    f"Eligible candidate pairs: "
    f"{len(eligible_candidate_pairs):,}"
)

Within-sample selection-policy checks:
eligible_pairs_are_not_empty: True
eligible_pair_identifiers_are_complete: True
eligible_pair_identity_is_unique: True

Within-sample selection policy frozen.
Eligible candidate pairs: 10,154


In [17]:
# =============================================================================
# Rank RNA-seq and methylation alternatives within each sample
# =============================================================================

def build_ranked_file_candidates(
    pair_table,
    qc_table,
    file_key,
    qc_metric,
    ascending,
    prefix,
):
    candidate_columns = SELECTION_UNIT_KEYS + [file_key]

    candidates = (
        pair_table[candidate_columns]
        .drop_duplicates()
        .merge(
            qc_table[[file_key, qc_metric]],
            on=file_key,
            how="left",
            validate="many_to_one",
        )
    )

    metric_column = f"{prefix}_qc_metric"
    candidates = candidates.rename(
        columns={qc_metric: metric_column}
    )

    grouped_metric = (
        candidates
        .groupby(
            SELECTION_UNIT_KEYS,
            sort=False,
            dropna=False,
        )[metric_column]
    )

    extreme_value = grouped_metric.transform(
        "min" if ascending else "max"
    )

    winner_count = grouped_metric.transform(
        lambda values: int(
            values.eq(
                values.min() if ascending else values.max()
            ).sum()
        )
    )

    candidates[
        f"{prefix}_candidate_count_within_sample"
    ] = grouped_metric.transform("size")

    candidates[
        f"{prefix}_qc_winner_count_within_sample"
    ] = winner_count

    candidates[
        f"{prefix}_qc_rank_within_sample"
    ] = (
        candidates
        .groupby(
            SELECTION_UNIT_KEYS,
            sort=False,
            dropna=False,
        )[metric_column]
        .rank(
            method="min",
            ascending=ascending,
        )
    )

    candidates[
        f"{prefix}_qc_unique_winner_within_sample"
    ] = (
        candidates[metric_column].eq(extreme_value)
        & winner_count.eq(1)
    )

    return candidates


rna_ranked_candidates = build_ranked_file_candidates(
    pair_table=eligible_candidate_pairs,
    qc_table=rna_file_qc_inventory,
    file_key=RNA_FILE_KEY,
    qc_metric=RNA_QC_METRIC,
    ascending=False,
    prefix="rna",
)

methylation_ranked_candidates = build_ranked_file_candidates(
    pair_table=eligible_candidate_pairs,
    qc_table=methylation_file_qc_metrics,
    file_key=METHYLATION_FILE_KEY,
    qc_metric=METHYLATION_QC_METRIC,
    ascending=True,
    prefix="methylation",
)

ranking_checks = {
    "rna_qc_values_are_complete": (
        rna_ranked_candidates["rna_qc_metric"]
        .notna()
        .all()
    ),
    "methylation_qc_values_are_complete": (
        methylation_ranked_candidates[
            "methylation_qc_metric"
        ]
        .notna()
        .all()
    ),
    "rna_candidate_keys_are_unique": (
        not rna_ranked_candidates[
            SELECTION_UNIT_KEYS + [RNA_FILE_KEY]
        ].duplicated().any()
    ),
    "methylation_candidate_keys_are_unique": (
        not methylation_ranked_candidates[
            SELECTION_UNIT_KEYS + [METHYLATION_FILE_KEY]
        ].duplicated().any()
    ),
}

print("Within-sample QC-ranking checks:")
for check_name, check_passed in ranking_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(ranking_checks.values()):
    raise ValueError(
        "Within-sample QC ranking failed."
    )

Within-sample QC-ranking checks:
rna_qc_values_are_complete: True
methylation_qc_values_are_complete: True
rna_candidate_keys_are_unique: True
methylation_candidate_keys_are_unique: True


In [18]:
# =============================================================================
# Apply within-sample QC selection and retain all alternatives
# =============================================================================

rna_annotation_columns = [
    *SELECTION_UNIT_KEYS,
    RNA_FILE_KEY,
    "rna_qc_metric",
    "rna_qc_rank_within_sample",
    "rna_candidate_count_within_sample",
    "rna_qc_winner_count_within_sample",
    "rna_qc_unique_winner_within_sample",
]

methylation_annotation_columns = [
    *SELECTION_UNIT_KEYS,
    METHYLATION_FILE_KEY,
    "methylation_qc_metric",
    "methylation_qc_rank_within_sample",
    "methylation_candidate_count_within_sample",
    "methylation_qc_winner_count_within_sample",
    "methylation_qc_unique_winner_within_sample",
]

within_sample_selection_inventory = (
    eligible_candidate_pairs
    .merge(
        rna_ranked_candidates[rna_annotation_columns],
        on=SELECTION_UNIT_KEYS + [RNA_FILE_KEY],
        how="left",
        validate="many_to_one",
    )
    .merge(
        methylation_ranked_candidates[
            methylation_annotation_columns
        ],
        on=SELECTION_UNIT_KEYS + [METHYLATION_FILE_KEY],
        how="left",
        validate="many_to_one",
    )
)

within_sample_selection_inventory[
    "rna_qc_tie_within_sample"
] = (
    within_sample_selection_inventory[
        "rna_candidate_count_within_sample"
    ].gt(1)
    & within_sample_selection_inventory[
        "rna_qc_winner_count_within_sample"
    ].gt(1)
)

within_sample_selection_inventory[
    "methylation_qc_tie_within_sample"
] = (
    within_sample_selection_inventory[
        "methylation_candidate_count_within_sample"
    ].gt(1)
    & within_sample_selection_inventory[
        "methylation_qc_winner_count_within_sample"
    ].gt(1)
)

within_sample_selection_inventory[
    "within_sample_selected"
] = (
    within_sample_selection_inventory[
        "rna_qc_unique_winner_within_sample"
    ]
    & within_sample_selection_inventory[
        "methylation_qc_unique_winner_within_sample"
    ]
)

has_unresolved_tie = (
    within_sample_selection_inventory[
        "rna_qc_tie_within_sample"
    ]
    | within_sample_selection_inventory[
        "methylation_qc_tie_within_sample"
    ]
)

within_sample_selection_inventory[
    "within_sample_selection_status"
] = np.select(
    [
        within_sample_selection_inventory[
            "within_sample_selected"
        ],
        has_unresolved_tie,
    ],
    [
        "selected_within_sample",
        "unresolved_within_sample",
    ],
    default="excluded_within_sample",
)


def build_selection_reason(row):
    if row["within_sample_selected"]:
        return "selected_by_within_sample_qc_policy"

    reasons = []

    if row["rna_qc_tie_within_sample"]:
        reasons.append("unresolved_rna_qc_tie")
    elif (
        row["rna_candidate_count_within_sample"] > 1
        and not row["rna_qc_unique_winner_within_sample"]
    ):
        reasons.append("lower_rna_qc_than_unique_winner")

    if row["methylation_qc_tie_within_sample"]:
        reasons.append("unresolved_methylation_qc_tie")
    elif (
        row["methylation_candidate_count_within_sample"] > 1
        and not row[
            "methylation_qc_unique_winner_within_sample"
        ]
    ):
        reasons.append(
            "higher_methylation_missingness_than_unique_winner"
        )

    return "; ".join(reasons)


within_sample_selection_inventory[
    "within_sample_selection_reason"
] = within_sample_selection_inventory.apply(
    build_selection_reason,
    axis=1,
)

within_sample_selected_pairs = (
    within_sample_selection_inventory.loc[
        within_sample_selection_inventory[
            "within_sample_selection_status"
        ].eq("selected_within_sample")
    ]
    .copy()
)

within_sample_excluded_alternatives = (
    within_sample_selection_inventory.loc[
        within_sample_selection_inventory[
            "within_sample_selection_status"
        ].eq("excluded_within_sample")
    ]
    .copy()
)

within_sample_unresolved_pairs = (
    within_sample_selection_inventory.loc[
        within_sample_selection_inventory[
            "within_sample_selection_status"
        ].eq("unresolved_within_sample")
    ]
    .copy()
)

print("Within-sample QC selection applied.")
print(
    f"Selected pairs: "
    f"{len(within_sample_selected_pairs):,}"
)
print(
    f"Excluded alternatives: "
    f"{len(within_sample_excluded_alternatives):,}"
)
print(
    f"Unresolved pairs: "
    f"{len(within_sample_unresolved_pairs):,}"
)

Within-sample QC selection applied.
Selected pairs: 10,017
Excluded alternatives: 137
Unresolved pairs: 0


In [19]:
# =============================================================================
# Validate within-sample selection
# =============================================================================

within_sample_selection_summary = (
    within_sample_selection_inventory
    .groupby(
        SELECTION_UNIT_KEYS,
        as_index=False,
        sort=True,
    )
    .agg(
        eligible_pair_count=(
            RNA_FILE_KEY,
            "size",
        ),
        selected_pair_count=(
            "within_sample_selected",
            "sum",
        ),
        unresolved_pair_count=(
            "within_sample_selection_status",
            lambda values: int(
                values.eq("unresolved_within_sample").sum()
            ),
        ),
        unique_rna_file_count=(
            RNA_FILE_KEY,
            "nunique",
        ),
        unique_methylation_file_count=(
            METHYLATION_FILE_KEY,
            "nunique",
        ),
    )
)

within_sample_selection_summary[
    "within_sample_resolution_status"
] = np.select(
    [
        (
            within_sample_selection_summary[
                "selected_pair_count"
            ].eq(1)
            & within_sample_selection_summary[
                "unresolved_pair_count"
            ].eq(0)
        ),
        (
            within_sample_selection_summary[
                "selected_pair_count"
            ].eq(0)
            & within_sample_selection_summary[
                "unresolved_pair_count"
            ].gt(0)
        ),
    ],
    [
        "resolved_one_pair",
        "unresolved_qc_tie",
    ],
    default="invalid_resolution_state",
)

within_sample_checks = {
    "pair_count_is_preserved": (
        len(within_sample_selection_inventory)
        == len(eligible_candidate_pairs)
    ),
    "pair_identity_is_preserved": (
        set(
            map(
                tuple,
                within_sample_selection_inventory[
                    SELECTION_PAIR_KEYS
                ].itertuples(index=False, name=None),
            )
        )
        == set(
            map(
                tuple,
                eligible_candidate_pairs[
                    SELECTION_PAIR_KEYS
                ].itertuples(index=False, name=None),
            )
        )
    ),
    "one_selected_pair_per_resolved_unit": (
        within_sample_selection_summary.loc[
            within_sample_selection_summary[
                "within_sample_resolution_status"
            ].eq("resolved_one_pair"),
            "selected_pair_count",
        ].eq(1).all()
    ),
    "unresolved_units_have_no_selected_pair": (
        within_sample_selection_summary.loc[
            within_sample_selection_summary[
                "within_sample_resolution_status"
            ].eq("unresolved_qc_tie"),
            "selected_pair_count",
        ].eq(0).all()
    ),
    "selected_pair_identity_is_unique": (
        not within_sample_selected_pairs[
            SELECTION_PAIR_KEYS
        ].duplicated().any()
    ),
    "resolution_status_is_valid": (
        within_sample_selection_summary[
            "within_sample_resolution_status"
        ]
        .isin(
            [
                "resolved_one_pair",
                "unresolved_qc_tie",
            ]
        )
        .all()
    ),
}

print("Within-sample selection checks:")
for check_name, check_passed in within_sample_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(within_sample_checks.values()):
    raise ValueError(
        "Within-sample selection validation failed."
    )

print("\nWithin-sample resolution summary:")
print(
    within_sample_selection_summary[
        "within_sample_resolution_status"
    ]
    .value_counts()
    .sort_index()
)

print(
    "\nNo final one-pair-per-case selection was performed."
)

display(
    within_sample_selection_summary.head(100)
)

Within-sample selection checks:
pair_count_is_preserved: True
pair_identity_is_preserved: True
one_selected_pair_per_resolved_unit: True
unresolved_units_have_no_selected_pair: True
selected_pair_identity_is_unique: True
resolution_status_is_valid: True

Within-sample resolution summary:
within_sample_resolution_status
resolved_one_pair    10017
Name: count, dtype: int64

No final one-pair-per-case selection was performed.


,case_submitter_id,sample_submitter_id,eligible_pair_count,selected_pair_count,unresolved_pair_count,unique_rna_file_count,unique_methylation_file_count,within_sample_resolution_status
0,TCGA-02-0003,TCGA-02-0003-01A,1,1,0,1,1,resolved_one_pair
1,TCGA-02-0033,TCGA-02-0033-01A,1,1,0,1,1,resolved_one_pair
2,TCGA-02-0038,TCGA-02-0038-01A,1,1,0,1,1,resolved_one_pair
3,TCGA-02-0047,TCGA-02-0047-01A,2,1,0,2,1,resolved_one_pair
4,TCGA-02-0055,TCGA-02-0055-01A,2,1,0,2,1,resolved_one_pair
...,...,...,...,...,...,...,...,...
95,TCGA-06-1802,TCGA-06-1802-01A,1,1,0,1,1,resolved_one_pair
96,TCGA-06-1804,TCGA-06-1804-01A,2,1,0,2,1,resolved_one_pair
97,TCGA-06-1805,TCGA-06-1805-01A,1,1,0,1,1,resolved_one_pair
98,TCGA-06-1806,TCGA-06-1806-01A,1,1,0,1,1,resolved_one_pair


## Case-level residual sample multiplicity audit

The previous stage selected one operational QC-based pair within each
`case_submitter_id`–`sample_submitter_id` unit.

This stage audits whether multiple distinct samples remain within the same
TCGA case.

No case-level pair selection, exclusion, or final-cohort publication is
performed here.

In [20]:
# =============================================================================
# Validate case-level audit inputs
# =============================================================================

CASE_AUDIT_COLUMNS = [
    CASE_KEY,
    SAMPLE_KEY,
    RNA_FILE_KEY,
    METHYLATION_FILE_KEY,
]

missing_case_audit_columns = sorted(
    set(CASE_AUDIT_COLUMNS)
    - set(within_sample_selected_pairs.columns)
)

if missing_case_audit_columns:
    raise KeyError(
        "Missing case-level audit columns: "
        + ", ".join(missing_case_audit_columns)
    )

selected_case_sample_units = (
    within_sample_selected_pairs[
        [CASE_KEY, SAMPLE_KEY]
    ]
    .drop_duplicates()
)

eligible_case_sample_units = (
    eligible_candidate_pairs[
        [CASE_KEY, SAMPLE_KEY]
    ]
    .drop_duplicates()
)

case_audit_checks = {
    "selected_pairs_are_not_empty": (
        not within_sample_selected_pairs.empty
    ),
    "selected_identifiers_are_complete": (
        within_sample_selected_pairs[
            CASE_AUDIT_COLUMNS
        ]
        .notna()
        .all()
        .all()
    ),
    "selected_pair_identity_is_unique": (
        not within_sample_selected_pairs[
            CASE_AUDIT_COLUMNS
        ]
        .duplicated()
        .any()
    ),
    "one_selected_pair_per_case_sample_unit": (
        within_sample_selected_pairs
        .groupby(
            [CASE_KEY, SAMPLE_KEY],
            sort=False,
        )
        .size()
        .eq(1)
        .all()
    ),
    "case_sample_units_are_preserved": (
        set(
            map(
                tuple,
                selected_case_sample_units.itertuples(
                    index=False,
                    name=None,
                ),
            )
        )
        == set(
            map(
                tuple,
                eligible_case_sample_units.itertuples(
                    index=False,
                    name=None,
                ),
            )
        )
    ),
}

print("Case-level audit input checks:")
for check_name, check_passed in case_audit_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(case_audit_checks.values()):
    raise ValueError(
        "Case-level audit input validation failed."
    )

Case-level audit input checks:
selected_pairs_are_not_empty: True
selected_identifiers_are_complete: True
selected_pair_identity_is_unique: True
one_selected_pair_per_case_sample_unit: True
case_sample_units_are_preserved: True


In [21]:
# =============================================================================
# Audit residual sample multiplicity per case
# =============================================================================

eligible_case_summary = (
    eligible_candidate_pairs
    .groupby(CASE_KEY, as_index=False, sort=True)
    .agg(
        eligible_candidate_pair_count=(
            RNA_FILE_KEY,
            "size",
        ),
        eligible_sample_count=(
            SAMPLE_KEY,
            "nunique",
        ),
    )
)

selected_case_summary = (
    within_sample_selected_pairs
    .groupby(CASE_KEY, as_index=False, sort=True)
    .agg(
        selected_pair_count=(
            RNA_FILE_KEY,
            "size",
        ),
        selected_sample_count=(
            SAMPLE_KEY,
            "nunique",
        ),
        selected_rna_file_count=(
            RNA_FILE_KEY,
            "nunique",
        ),
        selected_methylation_file_count=(
            METHYLATION_FILE_KEY,
            "nunique",
        ),
    )
)

case_level_multiplicity_audit = (
    eligible_case_summary
    .merge(
        selected_case_summary,
        on=CASE_KEY,
        how="left",
        validate="one_to_one",
    )
)

case_level_multiplicity_audit[
    "within_sample_alternative_count"
] = (
    case_level_multiplicity_audit[
        "eligible_candidate_pair_count"
    ]
    - case_level_multiplicity_audit[
        "selected_pair_count"
    ]
)

case_level_multiplicity_audit[
    "case_level_selection_required"
] = (
    case_level_multiplicity_audit[
        "selected_sample_count"
    ]
    > 1
)

case_level_multiplicity_audit[
    "case_level_multiplicity_status"
] = np.where(
    case_level_multiplicity_audit[
        "case_level_selection_required"
    ],
    "multiple_selected_samples_pending_case_selection",
    "one_selected_sample_per_case",
)

case_level_summary_checks = {
    "case_count_is_preserved": (
        len(case_level_multiplicity_audit)
        == eligible_candidate_pairs[CASE_KEY].nunique()
    ),
    "selected_sample_counts_match_eligible_units": (
        case_level_multiplicity_audit[
            "selected_sample_count"
        ]
        .eq(
            case_level_multiplicity_audit[
                "eligible_sample_count"
            ]
        )
        .all()
    ),
    "selected_pair_counts_are_valid": (
        case_level_multiplicity_audit[
            "selected_pair_count"
        ]
        .le(
            case_level_multiplicity_audit[
                "eligible_candidate_pair_count"
            ]
        )
        .all()
    ),
    "alternative_counts_are_nonnegative": (
        case_level_multiplicity_audit[
            "within_sample_alternative_count"
        ]
        .ge(0)
        .all()
    ),
}

print("Case-level multiplicity checks:")
for check_name, check_passed in case_level_summary_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(case_level_summary_checks.values()):
    raise ValueError(
        "Case-level multiplicity audit failed."
    )

print("\nCase-level multiplicity status:")
print(
    case_level_multiplicity_audit[
        "case_level_multiplicity_status"
    ]
    .value_counts()
    .sort_index()
)

print("\nNo one-sample-per-case selection was performed.")

display(
    case_level_multiplicity_audit.head(100)
)

Case-level multiplicity checks:
case_count_is_preserved: True
selected_sample_counts_match_eligible_units: True
selected_pair_counts_are_valid: True
alternative_counts_are_nonnegative: True

Case-level multiplicity status:
case_level_multiplicity_status
multiple_selected_samples_pending_case_selection      44
one_selected_sample_per_case                        9929
Name: count, dtype: int64

No one-sample-per-case selection was performed.


,case_submitter_id,eligible_candidate_pair_count,eligible_sample_count,selected_pair_count,selected_sample_count,selected_rna_file_count,selected_methylation_file_count,within_sample_alternative_count,case_level_selection_required,case_level_multiplicity_status
0,TCGA-02-0003,1,1,1,1,1,1,0,False,one_selected_sample_per_case
1,TCGA-02-0033,1,1,1,1,1,1,0,False,one_selected_sample_per_case
2,TCGA-02-0038,1,1,1,1,1,1,0,False,one_selected_sample_per_case
3,TCGA-02-0047,2,1,1,1,1,1,1,False,one_selected_sample_per_case
4,TCGA-02-0055,2,1,1,1,1,1,1,False,one_selected_sample_per_case
...,...,...,...,...,...,...,...,...,...,...
95,TCGA-06-1802,1,1,1,1,1,1,0,False,one_selected_sample_per_case
96,TCGA-06-1804,2,1,1,1,1,1,1,False,one_selected_sample_per_case
97,TCGA-06-1805,1,1,1,1,1,1,0,False,one_selected_sample_per_case
98,TCGA-06-1806,1,1,1,1,1,1,0,False,one_selected_sample_per_case


In [22]:
# =============================================================================
# Isolate cases with multiple selected samples
# =============================================================================

cases_pending_case_selection = (
    case_level_multiplicity_audit.loc[
        case_level_multiplicity_audit[
            "case_level_selection_required"
        ]
    ]
    .copy()
)

print(
    "Cases with multiple selected samples: "
    f"{len(cases_pending_case_selection):,}"
)

print(
    "Selected pairs represented by these cases: "
    f"{cases_pending_case_selection['selected_pair_count'].sum():,}"
)

display(
    cases_pending_case_selection
    .sort_values(
        [
            "selected_sample_count",
            CASE_KEY,
        ],
        ascending=[False, True],
        kind="stable",
    )
)

Cases with multiple selected samples: 44
Selected pairs represented by these cases: 88


,case_submitter_id,eligible_candidate_pair_count,eligible_sample_count,selected_pair_count,selected_sample_count,selected_rna_file_count,selected_methylation_file_count,within_sample_alternative_count,case_level_selection_required,case_level_multiplicity_status
1052,TCGA-44-2656,3,2,2,2,2,2,1,True,multiple_selected_samples_pending_case_selection
1056,TCGA-44-2665,3,2,2,2,2,2,1,True,multiple_selected_samples_pending_case_selection
1057,TCGA-44-2666,3,2,2,2,2,2,1,True,multiple_selected_samples_pending_case_selection
1058,TCGA-44-2668,3,2,2,2,2,2,1,True,multiple_selected_samples_pending_case_selection
1061,TCGA-44-3917,2,2,2,2,2,2,0,True,multiple_selected_samples_pending_case_selection
1062,TCGA-44-3918,3,2,2,2,2,2,1,True,multiple_selected_samples_pending_case_selection
1064,TCGA-44-4112,3,2,2,2,2,2,1,True,multiple_selected_samples_pending_case_selection
1067,TCGA-44-5645,5,2,2,2,2,2,3,True,multiple_selected_samples_pending_case_selection
1069,TCGA-44-6146,5,2,2,2,2,2,3,True,multiple_selected_samples_pending_case_selection
1070,TCGA-44-6147,5,2,2,2,2,2,3,True,multiple_selected_samples_pending_case_selection


In [23]:
print(
    cases_pending_case_selection
    .sort_values(
        [
            "selected_sample_count",
            CASE_KEY,
        ],
        ascending=[False, True],
        kind="stable",
    )
)

     case_submitter_id  eligible_candidate_pair_count  eligible_sample_count  \
1052      TCGA-44-2656                              3                      2   
1056      TCGA-44-2665                              3                      2   
1057      TCGA-44-2666                              3                      2   
1058      TCGA-44-2668                              3                      2   
1061      TCGA-44-3917                              2                      2   
1062      TCGA-44-3918                              3                      2   
1064      TCGA-44-4112                              3                      2   
1067      TCGA-44-5645                              5                      2   
1069      TCGA-44-6146                              5                      2   
1070      TCGA-44-6147                              5                      2   
1073      TCGA-44-6775                              5                      2   
2154      TCGA-A6-2672                  

## Case-level sample provenance audit

The within-sample QC policy has selected one operational pair for each
`case_submitter_id`–`sample_submitter_id` unit.

This audit inspects the provenance and metadata of the 44 cases that still
contain two distinct selected sample units.

No case-level sample selection, exclusion, or final-cohort publication is
performed here.

In [24]:
# =============================================================================
# Inspect provenance of cases with multiple selected samples
# =============================================================================

pending_case_ids = set(
    cases_pending_case_selection[CASE_KEY]
    .astype("string")
)

pending_sample_rows = (
    within_sample_selected_pairs.loc[
        within_sample_selected_pairs[CASE_KEY]
        .astype("string")
        .isin(pending_case_ids)
    ]
    .copy()
)

required_case_sample_columns = [
    CASE_KEY,
    SAMPLE_KEY,
    RNA_FILE_KEY,
    METHYLATION_FILE_KEY,
    "rna_qc_metric",
    "methylation_qc_metric",
]

missing_required_columns = sorted(
    set(required_case_sample_columns)
    - set(pending_sample_rows.columns)
)

if missing_required_columns:
    raise KeyError(
        "Missing case-level provenance columns: "
        + ", ".join(missing_required_columns)
    )

optional_metadata_tokens = (
    "sample_type",
    "tissue",
    "aliquot",
    "vial",
    "portion",
    "analyte",
    "project",
    "ffpe",
    "platform",
    "center",
)

optional_metadata_columns = [
    column
    for column in pending_sample_rows.columns
    if any(
        token in column.lower()
        for token in optional_metadata_tokens
    )
]

case_level_provenance_columns = list(
    dict.fromkeys(
        required_case_sample_columns
        + optional_metadata_columns
    )
)

case_level_sample_provenance = (
    pending_sample_rows[
        case_level_provenance_columns
    ]
    .sort_values(
        [CASE_KEY, SAMPLE_KEY],
        kind="stable",
    )
    .reset_index(drop=True)
)

case_level_provenance_checks = {
    "pending_case_count_is_preserved": (
        case_level_sample_provenance[CASE_KEY].nunique()
        == len(cases_pending_case_selection)
    ),
    "pending_pair_count_is_preserved": (
        len(case_level_sample_provenance)
        == int(
            cases_pending_case_selection[
                "selected_pair_count"
            ].sum()
        )
    ),
    "case_sample_identifiers_are_complete": (
        case_level_sample_provenance[
            [CASE_KEY, SAMPLE_KEY]
        ]
        .notna()
        .all()
        .all()
    ),
    "one_selected_pair_per_case_sample": (
        case_level_sample_provenance
        .groupby(
            [CASE_KEY, SAMPLE_KEY],
            sort=False,
        )
        .size()
        .eq(1)
        .all()
    ),
    "exactly_two_samples_per_pending_case": (
        case_level_sample_provenance
        .groupby(CASE_KEY)[SAMPLE_KEY]
        .nunique()
        .eq(2)
        .all()
    ),
}

print("Case-level provenance checks:")
for check_name, check_passed in case_level_provenance_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(case_level_provenance_checks.values()):
    raise ValueError(
        "Case-level provenance audit validation failed."
    )

print(
    "\nOptional provenance columns available:"
)
print(
    [
        column
        for column in optional_metadata_columns
        if column not in required_case_sample_columns
    ]
)

print(
    "\nNo case-level selection or exclusion was performed."
)

display(case_level_sample_provenance)

Case-level provenance checks:
pending_case_count_is_preserved: True
pending_pair_count_is_preserved: True
case_sample_identifiers_are_complete: True
one_selected_pair_per_case_sample: True
exactly_two_samples_per_pending_case: True

Optional provenance columns available:
['project_id', 'rna_aliquot_uuid', 'rna_aliquot_submitter_id', 'methylation_aliquot_uuid', 'methylation_aliquot_submitter_id', 'methylation_platform', 'rna_project', 'rna_sample_type_code', 'rna_vial', 'rna_portion_number', 'rna_analyte_code', 'rna_center_code', 'rna_sample_submitter_id_from_aliquot', 'methylation_project', 'methylation_sample_type_code', 'methylation_vial', 'methylation_portion_number', 'methylation_analyte_code', 'methylation_center_code', 'methylation_sample_submitter_id_from_aliquot', 'same_biospecimen_portion', 'analyte_pair', 'project_multidomain_review_candidate', 'same_portion_analyte_comparator_count', 'representative_comparator_aliquot_id', 'extreme_beta_center']

No case-level selection or e

,case_submitter_id,sample_submitter_id,rna_file_id,methylation_file_id,rna_qc_metric,methylation_qc_metric,project_id,rna_aliquot_uuid,rna_aliquot_submitter_id,methylation_aliquot_uuid,...,methylation_portion_number,methylation_analyte_code,methylation_center_code,methylation_sample_submitter_id_from_aliquot,same_biospecimen_portion,analyte_pair,project_multidomain_review_candidate,same_portion_analyte_comparator_count,representative_comparator_aliquot_id,extreme_beta_center
0,TCGA-44-2656,TCGA-44-2656-01A,77819b24-ea90-427b-8a99-c9a90417c903,8d5e1a2d-96b2-42ec-9087-3ea11cabd5e6,0.754029,0.150800,TCGA-LUAD,7b9dc578-8869-4322-a705-bfb9942d9b65,TCGA-44-2656-01A-02R-0946-07,e3df8a4b-8ff5-4723-ba74-e300c4e5fc13,...,2,D,5,TCGA-44-2656-01A,True,R → D,False,0,NaN,False
1,TCGA-44-2656,TCGA-44-2656-01B,c4bcc6b0-96a1-452b-9169-b4a16bcb9403,c280687c-5ca6-47e1-93c4-b1c71d2a76d8,0.258579,0.151531,TCGA-LUAD,8349b70a-dad2-492a-a7d1-95e734a96a5e,TCGA-44-2656-01B-06R-A277-07,e835b92b-461b-426f-b837-1c8c51f4c52b,...,6,D,5,TCGA-44-2656-01B,True,R → D,True,0,NaN,False
2,TCGA-44-2665,TCGA-44-2665-01A,ba8aebd8-07de-46c0-a28f-b2408c36f07f,bfe5caf9-c16b-4a48-9906-7020be48d009,0.776572,0.143890,TCGA-LUAD,9a6962a2-0a32-44d5-a2bf-c28c5f1a9cd0,TCGA-44-2665-01A-01R-0946-07,0b8fd8b7-3d18-4825-b672-c05d18e490e5,...,1,D,5,TCGA-44-2665-01A,True,R → D,False,0,NaN,False
3,TCGA-44-2665,TCGA-44-2665-01B,fc732523-d387-4a1d-ab81-34e44e869bac,0f59f51f-a4f6-40d7-b116-145053db1e85,0.077411,0.160334,TCGA-LUAD,ca010055-253d-4506-a55b-ae36d10805a1,TCGA-44-2665-01B-06R-A277-07,b39628cd-fd8c-469b-b0b0-e100611f37a2,...,6,D,5,TCGA-44-2665-01B,True,R → D,True,0,NaN,False
4,TCGA-44-2666,TCGA-44-2666-01A,52363b7b-f157-4fbe-8388-169b6152c9fe,8f61503a-ee54-4b4b-99da-97564fd126b0,0.772877,0.160242,TCGA-LUAD,bfad522f-6130-424d-8497-99cac9714966,TCGA-44-2666-01A-01R-0946-07,d1538d07-1565-443e-92a3-f39d9dcb0aed,...,1,D,5,TCGA-44-2666-01A,True,R → D,False,0,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,TCGA-HC-8258,TCGA-HC-8258-01B,7fdc3295-81f3-4cfe-9a71-ec28cf13ee8f,b8c3d07a-a322-472a-ba72-bdd96968ef3d,0.301473,0.151936,TCGA-PRAD,783062c7-cf22-426d-8cd1-52ecdd28b43d,TCGA-HC-8258-01B-05R-2302-07,c2d663c1-a355-4215-b4f4-a586f53f7234,...,5,D,5,TCGA-HC-8258-01B,True,R → D,False,0,NaN,False
84,TCGA-HC-8261,TCGA-HC-8261-01A,2b5963dc-6bbd-4541-a6ed-19fcc046fc10,05b64f4d-2622-40af-8afe-983d3f2662ad,0.775531,0.140412,TCGA-PRAD,4a54f172-9435-4e46-8396-405634425db2,TCGA-HC-8261-01A-11R-2263-07,ec9f0a58-ce6f-43ec-b55b-bbb5789b77fa,...,11,D,5,TCGA-HC-8261-01A,True,R → D,False,0,NaN,False
85,TCGA-HC-8261,TCGA-HC-8261-01B,653d54b0-3a4b-4d64-acb0-219b18b44b7d,c799680f-7ed8-4921-9db6-ff561b915230,0.448470,0.139832,TCGA-PRAD,3476ee8f-8cbe-4c6e-a3f7-f53facfe585a,TCGA-HC-8261-01B-05R-2302-07,b1944a95-71da-4197-b862-87519fa802e7,...,5,D,5,TCGA-HC-8261-01B,True,R → D,True,0,NaN,False
86,TCGA-HC-8265,TCGA-HC-8265-01A,cc7216dc-6b4a-4e8e-a69e-fa32c570c3a3,a0a9cefd-e118-4f0f-8c5b-df795279f393,0.736834,0.146156,TCGA-PRAD,cefe4277-778b-42b3-8e05-5df8327f50be,TCGA-HC-8265-01A-11R-2263-07,01d8e95f-61ab-41d8-9953-30d6b85a038f,...,11,D,5,TCGA-HC-8265-01A,True,R → D,False,0,NaN,False


In [25]:
print(case_level_sample_provenance)

   case_submitter_id sample_submitter_id  \
0       TCGA-44-2656    TCGA-44-2656-01A   
1       TCGA-44-2656    TCGA-44-2656-01B   
2       TCGA-44-2665    TCGA-44-2665-01A   
3       TCGA-44-2665    TCGA-44-2665-01B   
4       TCGA-44-2666    TCGA-44-2666-01A   
..               ...                 ...   
83      TCGA-HC-8258    TCGA-HC-8258-01B   
84      TCGA-HC-8261    TCGA-HC-8261-01A   
85      TCGA-HC-8261    TCGA-HC-8261-01B   
86      TCGA-HC-8265    TCGA-HC-8265-01A   
87      TCGA-HC-8265    TCGA-HC-8265-01B   

                             rna_file_id  \
0   77819b24-ea90-427b-8a99-c9a90417c903   
1   c4bcc6b0-96a1-452b-9169-b4a16bcb9403   
2   ba8aebd8-07de-46c0-a28f-b2408c36f07f   
3   fc732523-d387-4a1d-ab81-34e44e869bac   
4   52363b7b-f157-4fbe-8388-169b6152c9fe   
..                                   ...   
83  7fdc3295-81f3-4cfe-9a71-ec28cf13ee8f   
84  2b5963dc-6bbd-4541-a6ed-19fcc046fc10   
85  653d54b0-3a4b-4d64-acb0-219b18b44b7d   
86  cc7216dc-6b4a-4e8e-a69e-fa3

## Case-level joint QC dominance audit

The previous stage selected one operational RNA-seq–methylation pair within
each `case_submitter_id`–`sample_submitter_id` unit.

Forty-four cases retain two distinct selected sample units. The
`same_biospecimen_portion` flag refers to RNA–methylation matching within each
selected pair; it does not establish equivalence between distinct samples of
the same case.

This audit compares the two selected sample-level pairs using the already
frozen QC metrics:

- higher `gene_assigned_fraction_of_accounted` is preferred;
- lower `missing_beta_fraction` is preferred.

A sample is considered jointly QC-dominant only when it is no worse on either
metric and strictly better on at least one metric.

QC trade-offs and exact ties remain unresolved. No barcode, UUID, filename,
file order, platform, or project-review flag is used as a tie-breaker.

No final case-level selection or cohort publication is performed here.

In [26]:
# =============================================================================
# Prepare case-level joint QC comparison
# =============================================================================

CASE_LEVEL_RNA_METRIC = "rna_qc_metric"
CASE_LEVEL_METHYLATION_METRIC = "methylation_qc_metric"

CASE_LEVEL_SELECTION_POLICY_RECORD = {
    "policy_id": "tcga_multiomic_case_level_qc_dominance_v1",
    "scope": "cases with multiple selected sample-level pairs",
    "rna_metric": CASE_LEVEL_RNA_METRIC,
    "rna_rule": "higher_is_better",
    "methylation_metric": CASE_LEVEL_METHYLATION_METRIC,
    "methylation_rule": "lower_is_better",
    "dominance_rule": (
        "no worse on either metric and strictly better on at least one"
    ),
    "tradeoff_rule": "unresolved",
    "tie_rule": "unresolved",
    "identifier_or_file_order_tiebreaking": False,
    "platform_or_project_flag_tiebreaking": False,
    "biological_interpretation": "operational QC preference only",
    "final_case_level_selection": "deferred",
}

pending_case_ids = set(
    cases_pending_case_selection[CASE_KEY]
    .astype("string")
)

pending_case_pairs = (
    within_sample_selected_pairs.loc[
        within_sample_selected_pairs[CASE_KEY]
        .astype("string")
        .isin(pending_case_ids)
    ]
    .copy()
)

required_case_level_columns = [
    CASE_KEY,
    SAMPLE_KEY,
    RNA_FILE_KEY,
    METHYLATION_FILE_KEY,
    CASE_LEVEL_RNA_METRIC,
    CASE_LEVEL_METHYLATION_METRIC,
]

missing_case_level_columns = sorted(
    set(required_case_level_columns)
    - set(pending_case_pairs.columns)
)

if missing_case_level_columns:
    raise KeyError(
        "Missing case-level QC columns: "
        + ", ".join(missing_case_level_columns)
    )

case_level_input_checks = {
    "pending_case_pairs_are_not_empty": (
        not pending_case_pairs.empty
    ),
    "pending_case_count_is_preserved": (
        pending_case_pairs[CASE_KEY].nunique()
        == len(cases_pending_case_selection)
    ),
    "pending_pair_count_is_preserved": (
        len(pending_case_pairs)
        == int(
            cases_pending_case_selection[
                "selected_pair_count"
            ].sum()
        )
    ),
    "exactly_two_samples_per_case": (
        pending_case_pairs
        .groupby(CASE_KEY)[SAMPLE_KEY]
        .nunique()
        .eq(2)
        .all()
    ),
    "case_sample_units_are_unique": (
        not pending_case_pairs[
            [CASE_KEY, SAMPLE_KEY]
        ].duplicated().any()
    ),
    "qc_metrics_are_complete": (
        pending_case_pairs[
            [
                CASE_LEVEL_RNA_METRIC,
                CASE_LEVEL_METHYLATION_METRIC,
            ]
        ]
        .notna()
        .all()
        .all()
    ),
    "qc_metrics_are_numeric": (
        pd.api.types.is_numeric_dtype(
            pending_case_pairs[CASE_LEVEL_RNA_METRIC]
        )
        and pd.api.types.is_numeric_dtype(
            pending_case_pairs[CASE_LEVEL_METHYLATION_METRIC]
        )
    ),
}

print("Case-level joint-QC input checks:")
for check_name, check_passed in case_level_input_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(case_level_input_checks.values()):
    raise ValueError(
        "Case-level joint-QC input validation failed."
    )

print("\nCase-level joint-QC comparison prepared.")
print(f"Pending cases: {pending_case_pairs[CASE_KEY].nunique():,}")
print(f"Pending selected pairs: {len(pending_case_pairs):,}")

Case-level joint-QC input checks:
pending_case_pairs_are_not_empty: True
pending_case_count_is_preserved: True
pending_pair_count_is_preserved: True
exactly_two_samples_per_case: True
case_sample_units_are_unique: True
qc_metrics_are_complete: True
qc_metrics_are_numeric: True

Case-level joint-QC comparison prepared.
Pending cases: 44
Pending selected pairs: 88


In [27]:
# =============================================================================
# Identify jointly QC-dominant sample-level pairs
# =============================================================================

case_level_qc_comparison = pending_case_pairs.copy()

case_group = case_level_qc_comparison.groupby(
    CASE_KEY,
    sort=False,
)

case_level_qc_comparison["case_rna_qc_max"] = (
    case_group[CASE_LEVEL_RNA_METRIC]
    .transform("max")
)

case_level_qc_comparison["case_rna_qc_min"] = (
    case_group[CASE_LEVEL_RNA_METRIC]
    .transform("min")
)

case_level_qc_comparison["case_methylation_qc_min"] = (
    case_group[CASE_LEVEL_METHYLATION_METRIC]
    .transform("min")
)

case_level_qc_comparison["case_methylation_qc_max"] = (
    case_group[CASE_LEVEL_METHYLATION_METRIC]
    .transform("max")
)

case_level_qc_comparison[
    "case_level_pareto_dominant"
] = (
    case_level_qc_comparison[CASE_LEVEL_RNA_METRIC]
    .eq(case_level_qc_comparison["case_rna_qc_max"])
    & case_level_qc_comparison[CASE_LEVEL_METHYLATION_METRIC]
    .eq(case_level_qc_comparison["case_methylation_qc_min"])
    & (
        case_level_qc_comparison[CASE_LEVEL_RNA_METRIC]
        .gt(case_level_qc_comparison["case_rna_qc_min"])
        | case_level_qc_comparison[
            CASE_LEVEL_METHYLATION_METRIC
        ]
        .lt(case_level_qc_comparison["case_methylation_qc_max"])
    )
)

case_level_qc_comparison[
    "case_level_dominant_pair_count"
] = (
    case_level_qc_comparison
    .groupby(CASE_KEY, sort=False)[
        "case_level_pareto_dominant"
    ]
    .transform("sum")
)

case_level_qc_comparison[
    "case_level_selection_status"
] = np.select(
    [
        case_level_qc_comparison[
            "case_level_dominant_pair_count"
        ].eq(1),
        case_level_qc_comparison[
            "case_level_dominant_pair_count"
        ].eq(0),
    ],
    [
        "resolved_by_joint_qc_dominance",
        "unresolved_qc_tradeoff_or_tie",
    ],
    default="invalid_case_level_state",
)

case_level_comparison_summary = (
    case_level_qc_comparison
    .groupby(CASE_KEY, as_index=False, sort=True)
    .agg(
        selected_sample_count=(
            SAMPLE_KEY,
            "nunique",
        ),
        dominant_pair_count=(
            "case_level_pareto_dominant",
            "sum",
        ),
        rna_qc_range=(
            CASE_LEVEL_RNA_METRIC,
            lambda values: float(values.max() - values.min()),
        ),
        methylation_missingness_range=(
            CASE_LEVEL_METHYLATION_METRIC,
            lambda values: float(values.max() - values.min()),
        ),
        case_level_selection_status=(
            "case_level_selection_status",
            "first",
        ),
    )
)

case_level_comparison_checks = {
    "one_or_zero_dominant_pairs_per_case": (
        case_level_comparison_summary[
            "dominant_pair_count"
        ].le(1).all()
    ),
    "selection_status_is_valid": (
        case_level_comparison_summary[
            "case_level_selection_status"
        ]
        .isin(
            [
                "resolved_by_joint_qc_dominance",
                "unresolved_qc_tradeoff_or_tie",
            ]
        )
        .all()
    ),
    "summary_case_count_is_preserved": (
        len(case_level_comparison_summary)
        == len(cases_pending_case_selection)
    ),
}

print("Case-level joint-QC comparison checks:")
for check_name, check_passed in case_level_comparison_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(case_level_comparison_checks.values()):
    raise ValueError(
        "Case-level joint-QC comparison failed."
    )

print("\nCase-level comparison status:")
print(
    case_level_comparison_summary[
        "case_level_selection_status"
    ]
    .value_counts()
    .sort_index()
)

print("\nNo final case-level selection was performed.")

display(case_level_comparison_summary)

Case-level joint-QC comparison checks:
one_or_zero_dominant_pairs_per_case: True
selection_status_is_valid: True
summary_case_count_is_preserved: True

Case-level comparison status:
case_level_selection_status
resolved_by_joint_qc_dominance    36
unresolved_qc_tradeoff_or_tie      8
Name: count, dtype: int64

No final case-level selection was performed.


,case_submitter_id,selected_sample_count,dominant_pair_count,rna_qc_range,methylation_missingness_range,case_level_selection_status
0,TCGA-44-2656,2,1,0.495449,0.000732,resolved_by_joint_qc_dominance
1,TCGA-44-2665,2,1,0.699161,0.016444,resolved_by_joint_qc_dominance
2,TCGA-44-2666,2,0,0.538744,0.007343,unresolved_qc_tradeoff_or_tie
3,TCGA-44-2668,2,0,0.520259,0.001400,unresolved_qc_tradeoff_or_tie
4,TCGA-44-3917,2,1,0.096236,0.003933,resolved_by_joint_qc_dominance
5,TCGA-44-3918,2,1,0.503859,0.012053,resolved_by_joint_qc_dominance
6,TCGA-44-4112,2,1,0.525878,0.000465,resolved_by_joint_qc_dominance
7,TCGA-44-5645,2,1,0.551157,0.007933,resolved_by_joint_qc_dominance
8,TCGA-44-6146,2,1,0.498631,0.016173,resolved_by_joint_qc_dominance
9,TCGA-44-6147,2,1,0.542890,0.015558,resolved_by_joint_qc_dominance


## Confirmed case-level selection policy

The preceding within-sample stage selected one operational RNA-seq–methylation
pair for each `case_submitter_id`–`sample_submitter_id` unit.

For the 44 cases with two selected samples, the confirmed policy is:

- retain the sample-level pair only when it is no worse on both frozen QC metrics
  and strictly better on at least one;
- prefer higher `gene_assigned_fraction_of_accounted`;
- prefer lower `missing_beta_fraction`;
- classify QC trade-offs and exact ties as unresolved;
- select the jointly QC-dominant pair in the 36 resolved cases;
- exclude the 8 unresolved cases from the primary one-pair-per-case cohort; and
- retain both selected pairs from unresolved cases in the complete audit inventory.

The exclusion label is `unresolved_case_level_qc_tradeoff`. This represents
selection ambiguity, not file-level QC failure.

No barcode suffix, UUID, filename, file order, platform, or project flag is used
as a tie-breaker.

In [28]:
# =============================================================================
# Freeze the confirmed case-level selection policy
# =============================================================================

CASE_KEY = "case_submitter_id"
SAMPLE_KEY = "sample_submitter_id"
RNA_FILE_KEY = "rna_file_id"
METHYLATION_FILE_KEY = "methylation_file_id"

RNA_QC_METRIC = "rna_qc_metric"
METHYLATION_QC_METRIC = "methylation_qc_metric"

RESOLVED_CASE_STATUS = "resolved_by_joint_qc_dominance"
UNRESOLVED_CASE_STATUS = "unresolved_qc_tradeoff_or_tie"
UNRESOLVED_CASE_LABEL = "unresolved_case_level_qc_tradeoff"

EXPECTED_SOURCE_CASE_COUNT = 9_973
EXPECTED_PENDING_CASE_COUNT = 44
EXPECTED_PENDING_PAIR_COUNT = 88
EXPECTED_RESOLVED_CASE_COUNT = 36
EXPECTED_UNRESOLVED_CASE_COUNT = 8
EXPECTED_PRIMARY_CASE_COUNT = 9_965

CASE_LEVEL_PAIR_SELECTION_INVENTORY_PATH = (
    INTEGRATION_METADATA_OUTPUT_DIR
    / "tcga_primary_tumor_rnaseq_methylation_"
      "case_level_pair_selection_inventory.csv"
)

FINAL_MULTIOMIC_PAIR_MAPPING_PATH = (
    INTEGRATION_METADATA_OUTPUT_DIR
    / "tcga_primary_tumor_rnaseq_methylation_final_pair_mapping.csv"
)

CASE_LEVEL_SELECTION_POLICY_PATH = (
    INTEGRATION_METADATA_OUTPUT_DIR
    / "tcga_primary_tumor_rnaseq_methylation_"
      "case_level_selection_policy.json"
)

CASE_LEVEL_SELECTION_POLICY_RECORD = {
    "policy_id": "tcga_case_level_joint_qc_dominance_v1",
    "biological_unit": CASE_KEY,
    "sample_unit": SAMPLE_KEY,
    "input_scope": "one operational pair per selected case-sample unit",
    "rna_metric": RNA_QC_METRIC,
    "rna_metric_definition": (
        "gene_assigned_fraction_of_accounted; higher is better"
    ),
    "methylation_metric": METHYLATION_QC_METRIC,
    "methylation_metric_definition": (
        "missing_beta_fraction; lower is better"
    ),
    "dominance_rule": (
        "no worse on either metric and strictly better on at least one"
    ),
    "tradeoff_rule": "unresolved",
    "exact_tie_rule": "unresolved",
    "primary_cohort_rule": (
        "retain one jointly dominant pair per case; "
        "exclude unresolved cases"
    ),
    "unresolved_case_label": UNRESOLVED_CASE_LABEL,
    "arbitrary_tiebreakers_used": False,
    "arbitrary_tiebreakers_excluded": [
        "barcode suffix",
        "UUID",
        "filename",
        "file order",
        "platform",
        "project flag",
    ],
    "interpretation": (
        "operational QC preference only; "
        "not a biological equivalence claim"
    ),
    "sensitivity_analysis_policy": (
        "retain both pairs from unresolved cases in the audit inventory"
    ),
}

print("Confirmed case-level selection policy frozen.")
print(
    "Policy ID:",
    CASE_LEVEL_SELECTION_POLICY_RECORD["policy_id"],
)

Confirmed case-level selection policy frozen.
Policy ID: tcga_case_level_joint_qc_dominance_v1


In [29]:
# =============================================================================
# Validate the preceding within-sample selection boundary
# =============================================================================

required_case_level_objects = [
    "within_sample_selected_pairs",
    "case_level_qc_comparison",
]

missing_case_level_objects = [
    name
    for name in required_case_level_objects
    if name not in globals()
]

if missing_case_level_objects:
    raise RuntimeError(
        "Run the preceding within-sample and joint-QC stages first. "
        "Missing objects: "
        + ", ".join(missing_case_level_objects)
    )

required_comparison_columns = [
    CASE_KEY,
    SAMPLE_KEY,
    RNA_FILE_KEY,
    METHYLATION_FILE_KEY,
    RNA_QC_METRIC,
    METHYLATION_QC_METRIC,
    "case_level_pareto_dominant",
    "case_level_selection_status",
]

missing_comparison_columns = sorted(
    set(required_comparison_columns)
    - set(case_level_qc_comparison.columns)
)

if missing_comparison_columns:
    raise KeyError(
        "Missing columns in case_level_qc_comparison: "
        + ", ".join(missing_comparison_columns)
    )

comparison_pairs = case_level_qc_comparison[
    required_comparison_columns
].copy()

for column in [
    CASE_KEY,
    SAMPLE_KEY,
    RNA_FILE_KEY,
    METHYLATION_FILE_KEY,
]:
    comparison_pairs[column] = comparison_pairs[column].astype("string")

comparison_case_summary = (
    comparison_pairs
    .groupby(CASE_KEY, as_index=False, sort=True)
    .agg(
        selected_sample_count=(
            SAMPLE_KEY,
            "nunique",
        ),
        dominant_pair_count=(
            "case_level_pareto_dominant",
            "sum",
        ),
        case_level_selection_status=(
            "case_level_selection_status",
            "first",
        ),
    )
)

case_level_status_counts = (
    comparison_case_summary[
        "case_level_selection_status"
    ]
    .value_counts()
)

case_level_input_checks = {
    "comparison_pairs_are_not_empty": (
        not comparison_pairs.empty
    ),
    "comparison_case_sample_keys_are_unique": (
        not comparison_pairs[
            [CASE_KEY, SAMPLE_KEY]
        ]
        .duplicated()
        .any()
    ),
    "comparison_case_count_is_44": (
        len(comparison_case_summary)
        == EXPECTED_PENDING_CASE_COUNT
    ),
    "comparison_pair_count_is_88": (
        len(comparison_pairs)
        == EXPECTED_PENDING_PAIR_COUNT
    ),
    "all_pending_cases_have_two_samples": (
        comparison_case_summary[
            "selected_sample_count"
        ]
        .eq(2)
        .all()
    ),
    "dominant_pair_count_is_zero_or_one": (
        comparison_case_summary[
            "dominant_pair_count"
        ]
        .isin([0, 1])
        .all()
    ),
    "comparison_statuses_are_valid": (
        comparison_case_summary[
            "case_level_selection_status"
        ]
        .isin(
            [
                RESOLVED_CASE_STATUS,
                UNRESOLVED_CASE_STATUS,
            ]
        )
        .all()
    ),
    "resolved_case_count_is_36": (
        int(
            case_level_status_counts.get(
                RESOLVED_CASE_STATUS,
                0,
            )
        )
        == EXPECTED_RESOLVED_CASE_COUNT
    ),
    "unresolved_case_count_is_8": (
        int(
            case_level_status_counts.get(
                UNRESOLVED_CASE_STATUS,
                0,
            )
        )
        == EXPECTED_UNRESOLVED_CASE_COUNT
    ),
}

print("Case-level policy input checks:")
for check_name, check_passed in case_level_input_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(case_level_input_checks.values()):
    raise ValueError(
        "Confirmed case-level policy input validation failed."
    )

print()
print(
    "Resolved cases:",
    int(
        case_level_status_counts.get(
            RESOLVED_CASE_STATUS,
            0,
        )
    ),
)
print(
    "Unresolved cases:",
    int(
        case_level_status_counts.get(
            UNRESOLVED_CASE_STATUS,
            0,
        )
    ),
)

Case-level policy input checks:
comparison_pairs_are_not_empty: True
comparison_case_sample_keys_are_unique: True
comparison_case_count_is_44: True
comparison_pair_count_is_88: True
all_pending_cases_have_two_samples: True
dominant_pair_count_is_zero_or_one: True
comparison_statuses_are_valid: True
resolved_case_count_is_36: True
unresolved_case_count_is_8: True

Resolved cases: 36
Unresolved cases: 8


In [30]:
# =============================================================================
# Build the complete case-level pair-selection inventory
# =============================================================================

source_pair_inventory = (
    qc_annotated_pair_inventory
    .copy()
    .assign(
        _source_candidate_pair_order=np.arange(
            len(qc_annotated_pair_inventory)
        )
    )
)

for column in [
    CASE_KEY,
    SAMPLE_KEY,
    RNA_FILE_KEY,
    METHYLATION_FILE_KEY,
]:
    source_pair_inventory[column] = (
        source_pair_inventory[column]
        .astype("string")
    )


selected_pair_keys = (
    within_sample_selected_pairs[
        [RNA_FILE_KEY, METHYLATION_FILE_KEY]
    ]
    .copy()
)

for column in [
    RNA_FILE_KEY,
    METHYLATION_FILE_KEY,
]:
    selected_pair_keys[column] = (
        selected_pair_keys[column]
        .astype("string")
    )

if selected_pair_keys.duplicated().any():
    raise ValueError(
        "Within-sample selected pair keys are not unique."
    )

selected_pair_keys["within_sample_pair_selected"] = True


comparison_decisions = (
    comparison_pairs[
        [
            CASE_KEY,
            SAMPLE_KEY,
            "case_level_pareto_dominant",
            "case_level_selection_status",
        ]
    ]
    .copy()
)


dominant_sample_by_case = (
    comparison_decisions.loc[
        comparison_decisions[
            "case_level_pareto_dominant"
        ],
        [CASE_KEY, SAMPLE_KEY],
    ]
    .set_index(CASE_KEY)[SAMPLE_KEY]
)


rna_metric_lookup = (
    rna_file_qc_inventory[
        [
            RNA_FILE_KEY,
            "gene_assigned_fraction_of_accounted",
        ]
    ]
    .copy()
)

rna_metric_lookup[RNA_FILE_KEY] = (
    rna_metric_lookup[RNA_FILE_KEY]
    .astype("string")
)

if not rna_metric_lookup[RNA_FILE_KEY].is_unique:
    raise ValueError(
        "RNA-seq QC metric lookup is not unique by file."
    )

rna_metric_lookup = (
    rna_metric_lookup
    .set_index(RNA_FILE_KEY)
    ["gene_assigned_fraction_of_accounted"]
)


methylation_metric_lookup = (
    methylation_file_qc_metrics[
        [
            METHYLATION_FILE_KEY,
            "missing_beta_fraction",
        ]
    ]
    .copy()
)

methylation_metric_lookup[METHYLATION_FILE_KEY] = (
    methylation_metric_lookup[METHYLATION_FILE_KEY]
    .astype("string")
)

if not methylation_metric_lookup[METHYLATION_FILE_KEY].is_unique:
    raise ValueError(
        "Methylation QC metric lookup is not unique by file."
    )

methylation_metric_lookup = (
    methylation_metric_lookup
    .set_index(METHYLATION_FILE_KEY)
    ["missing_beta_fraction"]
)


source_pair_inventory[RNA_QC_METRIC] = (
    source_pair_inventory[RNA_FILE_KEY]
    .map(rna_metric_lookup)
)

source_pair_inventory[METHYLATION_QC_METRIC] = (
    source_pair_inventory[METHYLATION_FILE_KEY]
    .map(methylation_metric_lookup)
)


source_pair_eligible = (
    source_pair_inventory[
        "pair_eligible_after_methylation_qc"
    ]
    .astype("boolean")
)

if source_pair_eligible.isna().any():
    raise ValueError(
        "Pair-level QC eligibility contains missing values."
    )

source_pair_eligible = source_pair_eligible.astype(bool)


if source_pair_inventory.loc[
    source_pair_eligible,
    [
        RNA_QC_METRIC,
        METHYLATION_QC_METRIC,
    ],
].isna().any().any():
    raise ValueError(
        "QC metrics are incomplete for eligible pairs."
    )


case_level_pair_selection_inventory = (
    source_pair_inventory
    .merge(
        selected_pair_keys,
        on=[
            RNA_FILE_KEY,
            METHYLATION_FILE_KEY,
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        comparison_decisions,
        on=[
            CASE_KEY,
            SAMPLE_KEY,
        ],
        how="left",
        validate="many_to_one",
    )
    .sort_values(
        "_source_candidate_pair_order",
        kind="stable",
    )
    .reset_index(drop=True)
)


case_level_pair_selection_inventory[
    "within_sample_pair_selected"
] = (
    case_level_pair_selection_inventory[
        "within_sample_pair_selected"
    ]
    .fillna(False)
    .astype(bool)
)


case_level_pair_selection_inventory[
    "case_level_selection_status"
] = (
    case_level_pair_selection_inventory[
        "case_level_selection_status"
    ]
    .astype("string")
    .fillna("single_selected_sample")
)


# -------------------------------------------------------------------------
# Convert all logical masks to ordinary bool Series.
# This prevents pandas.NA from reaching np.select().
# -------------------------------------------------------------------------

pair_eligible = (
    case_level_pair_selection_inventory[
        "pair_eligible_after_methylation_qc"
    ]
    .astype("boolean")
)

if pair_eligible.isna().any():
    raise ValueError(
        "Merged pair-level QC eligibility contains missing values."
    )

pair_eligible = pair_eligible.astype(bool)


dominant_sample = (
    case_level_pair_selection_inventory[
        CASE_KEY
    ]
    .astype("string")
    .map(dominant_sample_by_case)
    .astype("string")
)


dominant_sample_match = (
    case_level_pair_selection_inventory[
        SAMPLE_KEY
    ]
    .astype("string")
    .eq(dominant_sample)
    .fillna(False)
    .astype(bool)
)


resolved_case = (
    case_level_pair_selection_inventory[
        "case_level_selection_status"
    ]
    .eq(RESOLVED_CASE_STATUS)
    .fillna(False)
    .astype(bool)
)


unresolved_case = (
    case_level_pair_selection_inventory[
        "case_level_selection_status"
    ]
    .eq(UNRESOLVED_CASE_STATUS)
    .fillna(False)
    .astype(bool)
)


within_sample_selected = (
    case_level_pair_selection_inventory[
        "within_sample_pair_selected"
    ]
    .fillna(False)
    .astype(bool)
)


case_level_pair_selection_inventory[
    "case_level_row_decision"
] = np.select(
    [
        ~pair_eligible,
        pair_eligible & ~within_sample_selected,
        (
            pair_eligible
            & within_sample_selected
            & ~resolved_case
            & ~unresolved_case
        ),
        (
            pair_eligible
            & within_sample_selected
            & unresolved_case
        ),
        (
            pair_eligible
            & within_sample_selected
            & resolved_case
            & dominant_sample_match
        ),
        (
            pair_eligible
            & within_sample_selected
            & resolved_case
            & ~dominant_sample_match
        ),
    ],
    [
        "excluded_upstream_qc",
        "excluded_within_sample_alternative",
        "selected_unique_sample_case",
        "excluded_unresolved_case_level_qc_tradeoff",
        "selected_case_level_qc_dominant",
        "excluded_non_dominant_case_level_alternative",
    ],
    default="invalid_case_level_selection_state",
)


case_level_pair_selection_inventory[
    "case_level_primary_inclusion"
] = (
    case_level_pair_selection_inventory[
        "case_level_row_decision"
    ]
    .isin(
        [
            "selected_unique_sample_case",
            "selected_case_level_qc_dominant",
        ]
    )
)


case_level_pair_selection_inventory[
    "case_level_exclusion_reason"
] = np.select(
    [
        case_level_pair_selection_inventory[
            "case_level_row_decision"
        ].eq("excluded_upstream_qc"),
        case_level_pair_selection_inventory[
            "case_level_row_decision"
        ].eq("excluded_within_sample_alternative"),
        case_level_pair_selection_inventory[
            "case_level_row_decision"
        ].eq(
            "excluded_unresolved_case_level_qc_tradeoff"
        ),
        case_level_pair_selection_inventory[
            "case_level_row_decision"
        ].eq(
            "excluded_non_dominant_case_level_alternative"
        ),
    ],
    [
        "Ineligible after upstream RNA-seq or methylation QC",
        "Alternative pair excluded by within-sample selection",
        UNRESOLVED_CASE_LABEL,
        "Non-dominant sample-level alternative under joint QC",
    ],
    default="",
)


selection_state_checks = {
    "all_source_rows_are_retained": (
        len(case_level_pair_selection_inventory)
        == len(source_pair_inventory)
    ),
    "candidate_pair_keys_remain_unique": (
        not case_level_pair_selection_inventory[
            [RNA_FILE_KEY, METHYLATION_FILE_KEY]
        ]
        .duplicated()
        .any()
    ),
    "selected_pair_count_matches_previous_stage": (
        int(
            case_level_pair_selection_inventory[
                "within_sample_pair_selected"
            ]
            .sum()
        )
        == len(within_sample_selected_pairs)
    ),
    "selected_pairs_are_qc_eligible": (
        pair_eligible.loc[
            case_level_pair_selection_inventory[
                "within_sample_pair_selected"
            ]
        ]
        .all()
    ),
    "row_decisions_are_valid": (
        case_level_pair_selection_inventory[
            "case_level_row_decision"
        ]
        .isin(
            [
                "excluded_upstream_qc",
                "excluded_within_sample_alternative",
                "selected_unique_sample_case",
                "excluded_unresolved_case_level_qc_tradeoff",
                "selected_case_level_qc_dominant",
                "excluded_non_dominant_case_level_alternative",
            ]
        )
        .all()
    ),
}


print("Case-level pair-selection inventory checks:")

for check_name, check_passed in selection_state_checks.items():
    print(f"{check_name}: {check_passed}")


if not all(selection_state_checks.values()):
    raise ValueError(
        "Case-level pair-selection inventory construction failed."
    )

Case-level pair-selection inventory checks:
all_source_rows_are_retained: True
candidate_pair_keys_remain_unique: True
selected_pair_count_matches_previous_stage: True
selected_pairs_are_qc_eligible: True
row_decisions_are_valid: True


In [31]:
# =============================================================================
# Construct and validate the primary one-pair-per-case mapping
# =============================================================================

final_multiomic_pair_mapping = (
    case_level_pair_selection_inventory.loc[
        case_level_pair_selection_inventory[
            "case_level_primary_inclusion"
        ]
    ]
    .sort_values(
        [CASE_KEY, SAMPLE_KEY],
        kind="stable",
    )
    .reset_index(drop=True)
)

primary_case_ids = (
    final_multiomic_pair_mapping[CASE_KEY]
    .astype("string")
)

unresolved_case_ids = set(
    comparison_case_summary.loc[
        comparison_case_summary[
            "case_level_selection_status"
        ].eq(UNRESOLVED_CASE_STATUS),
        CASE_KEY,
    ]
    .astype("string")
)

mapping_checks = {
    "primary_mapping_is_not_empty": (
        not final_multiomic_pair_mapping.empty
    ),
    "one_row_per_primary_case": (
        primary_case_ids.is_unique
    ),
    "primary_mapping_pairs_are_qc_eligible": (
        final_multiomic_pair_mapping[
            "pair_eligible_after_methylation_qc"
        ]
        .all()
    ),
    "unresolved_cases_are_absent": (
        unresolved_case_ids.isdisjoint(
            set(primary_case_ids)
        )
    ),
    "unresolved_pairs_remain_in_audit_inventory": (
        case_level_pair_selection_inventory.loc[
            case_level_pair_selection_inventory[
                CASE_KEY
            ]
            .astype("string")
            .isin(unresolved_case_ids)
            & case_level_pair_selection_inventory[
                "within_sample_pair_selected"
            ],
            "case_level_row_decision",
        ]
        .eq(
            "excluded_unresolved_case_level_qc_tradeoff"
        )
        .all()
    ),
    "primary_case_count_is_9965": (
        len(final_multiomic_pair_mapping)
        == EXPECTED_PRIMARY_CASE_COUNT
    ),
}

print("Primary multi-omic mapping checks:")
for check_name, check_passed in mapping_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(mapping_checks.values()):
    raise ValueError(
        "Primary one-pair-per-case mapping validation failed."
    )

print()
print(
    "Complete audit inventory rows:",
    f"{len(case_level_pair_selection_inventory):,}",
)
print(
    "Primary cases:",
    f"{len(final_multiomic_pair_mapping):,}",
)
print(
    "Unresolved cases excluded:",
    f"{len(unresolved_case_ids):,}",
)

Primary multi-omic mapping checks:
primary_mapping_is_not_empty: True
one_row_per_primary_case: True
primary_mapping_pairs_are_qc_eligible: True
unresolved_cases_are_absent: True
unresolved_pairs_remain_in_audit_inventory: True
primary_case_count_is_9965: True

Complete audit inventory rows: 10,162
Primary cases: 9,965
Unresolved cases excluded: 8


In [32]:
# =============================================================================
# Publish and round-trip-validate case-level selection artifacts
# =============================================================================

INTEGRATION_METADATA_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

policy_output = dict(
    CASE_LEVEL_SELECTION_POLICY_RECORD
)

policy_output["observed_counts"] = {
    "source_candidate_pair_rows": int(
        len(qc_annotated_pair_inventory)
    ),
    "source_case_count": int(
        qc_annotated_pair_inventory[
            CASE_KEY
        ].nunique()
    ),
    "within_sample_selected_pair_count": int(
        len(within_sample_selected_pairs)
    ),
    "pending_case_count": int(
        len(comparison_case_summary)
    ),
    "resolved_case_count": int(
        case_level_status_counts.get(
            RESOLVED_CASE_STATUS,
            0,
        )
    ),
    "unresolved_case_count": int(
        case_level_status_counts.get(
            UNRESOLVED_CASE_STATUS,
            0,
        )
    ),
    "primary_case_count": int(
        len(final_multiomic_pair_mapping)
    ),
}

policy_output["unresolved_case_ids"] = sorted(
    unresolved_case_ids
)

policy_output["input_artifacts"] = {
    "pair_inventory": {
        "path": project_relative_path(
            METHYLATION_QC_ANNOTATED_PAIR_INVENTORY_PATH
        ),
        "sha256": calculate_sha256(
            METHYLATION_QC_ANNOTATED_PAIR_INVENTORY_PATH
        ),
    },
    "rna_file_qc_inventory": {
        "path": project_relative_path(
            RNA_FILE_QC_INVENTORY_PATH
        ),
        "sha256": calculate_sha256(
            RNA_FILE_QC_INVENTORY_PATH
        ),
    },
    "methylation_file_qc_metrics": {
        "path": project_relative_path(
            METHYLATION_FILE_QC_METRICS_PATH
        ),
        "sha256": calculate_sha256(
            METHYLATION_FILE_QC_METRICS_PATH
        ),
    },
}

temporary_inventory_path = (
    CASE_LEVEL_PAIR_SELECTION_INVENTORY_PATH.with_name(
        CASE_LEVEL_PAIR_SELECTION_INVENTORY_PATH.name
        + ".tmp"
    )
)

temporary_mapping_path = (
    FINAL_MULTIOMIC_PAIR_MAPPING_PATH.with_name(
        FINAL_MULTIOMIC_PAIR_MAPPING_PATH.name
        + ".tmp"
    )
)

temporary_policy_path = (
    CASE_LEVEL_SELECTION_POLICY_PATH.with_name(
        CASE_LEVEL_SELECTION_POLICY_PATH.name
        + ".tmp"
    )
)

case_level_pair_selection_inventory.to_csv(
    temporary_inventory_path,
    index=False,
)

final_multiomic_pair_mapping.to_csv(
    temporary_mapping_path,
    index=False,
)

with temporary_policy_path.open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        policy_output,
        handle,
        indent=2,
        ensure_ascii=False,
    )
    handle.write("\n")

written_inventory = pd.read_csv(
    temporary_inventory_path,
    low_memory=False,
)

written_mapping = pd.read_csv(
    temporary_mapping_path,
    low_memory=False,
)

with temporary_policy_path.open(
    "r",
    encoding="utf-8",
) as handle:
    written_policy = json.load(handle)

written_artifact_checks = {
    "temporary_inventory_exists": (
        temporary_inventory_path.is_file()
    ),
    "temporary_mapping_exists": (
        temporary_mapping_path.is_file()
    ),
    "temporary_policy_exists": (
        temporary_policy_path.is_file()
    ),
    "written_inventory_shape_matches_memory": (
        written_inventory.shape
        == case_level_pair_selection_inventory.shape
    ),
    "written_inventory_columns_match_memory": (
        written_inventory.columns.tolist()
        == case_level_pair_selection_inventory.columns.tolist()
    ),
    "written_mapping_shape_matches_memory": (
        written_mapping.shape
        == final_multiomic_pair_mapping.shape
    ),
    "written_mapping_columns_match_memory": (
        written_mapping.columns.tolist()
        == final_multiomic_pair_mapping.columns.tolist()
    ),
    "written_mapping_case_order_matches_memory": (
        written_mapping[CASE_KEY]
        .astype("string")
        .tolist()
        == final_multiomic_pair_mapping[
            CASE_KEY
        ]
        .astype("string")
        .tolist()
    ),
    "written_policy_id_matches_memory": (
        written_policy["policy_id"]
        == CASE_LEVEL_SELECTION_POLICY_RECORD[
            "policy_id"
        ]
    ),
    "written_policy_counts_match_memory": (
        written_policy["observed_counts"]
        == policy_output["observed_counts"]
    ),
    "written_unresolved_cases_match_memory": (
        written_policy["unresolved_case_ids"]
        == policy_output["unresolved_case_ids"]
    ),
}

print("Case-level selection written-artifact checks:")
for check_name, check_passed in written_artifact_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(written_artifact_checks.values()):
    temporary_inventory_path.unlink(
        missing_ok=True
    )
    temporary_mapping_path.unlink(
        missing_ok=True
    )
    temporary_policy_path.unlink(
        missing_ok=True
    )
    raise IOError(
        "Case-level selection artifacts failed validation."
    )

temporary_inventory_path.replace(
    CASE_LEVEL_PAIR_SELECTION_INVENTORY_PATH
)

temporary_mapping_path.replace(
    FINAL_MULTIOMIC_PAIR_MAPPING_PATH
)

temporary_policy_path.replace(
    CASE_LEVEL_SELECTION_POLICY_PATH
)

print()
print("Case-level selection artifacts published.")
print(
    "Audit inventory:",
    project_relative_path(
        CASE_LEVEL_PAIR_SELECTION_INVENTORY_PATH
    ),
)
print(
    "Primary mapping:",
    project_relative_path(
        FINAL_MULTIOMIC_PAIR_MAPPING_PATH
    ),
)
print(
    "Policy record:",
    project_relative_path(
        CASE_LEVEL_SELECTION_POLICY_PATH
    ),
)

Case-level selection written-artifact checks:
temporary_inventory_exists: True
temporary_mapping_exists: True
temporary_policy_exists: True
written_inventory_shape_matches_memory: True
written_inventory_columns_match_memory: True
written_mapping_shape_matches_memory: True
written_mapping_columns_match_memory: True
written_mapping_case_order_matches_memory: True
written_policy_id_matches_memory: True
written_policy_counts_match_memory: True
written_unresolved_cases_match_memory: True

Case-level selection artifacts published.
Audit inventory: data/interim/metadata/tcga_primary_tumor_rnaseq_methylation_case_level_pair_selection_inventory.csv
Primary mapping: data/interim/metadata/tcga_primary_tumor_rnaseq_methylation_final_pair_mapping.csv
Policy record: data/interim/metadata/tcga_primary_tumor_rnaseq_methylation_case_level_selection_policy.json


## Final case-level multi-omic consumables

The confirmed case-level selection is now frozen.

This stage constructs the downstream-consumable representation layer:

- one final RNA-seq–methylation observation per primary case;
- the cohort-frozen raw unstranded RNA-seq representation;
- platform-specific HM27 and HM450 methylation spaces;
- an explicitly aligned shared HM27/HM450 methylation space;
- final sample and probe mappings;
- explicit matrix-column and matrix-row orders.

No cohort-dependent RNA-seq filtering, normalization, or transformation is applied here. These operations remain deferred to downstream preprocessing after the definitive paired cohort has been established.

HM27 and HM450 remain separate analytical representations. The shared space is restricted to probes passing the frozen eligibility policy on both platforms and does not imply naive platform pooling.

In [33]:
# =============================================================================
# Define final case-level multi-omic consumable paths
# =============================================================================

HM27_PLATFORM = "Illumina Human Methylation 27"
HM450_PLATFORM = "Illumina Human Methylation 450"

SHARED_METHYLATION_PLATFORM = "shared_hm27_hm450"

FINAL_PRIMARY_CASE_COUNT = EXPECTED_PRIMARY_CASE_COUNT

FINAL_RNA_MATRIX_PATH = (
    INTEGRATION_EXPRESSION_OUTPUT_DIR
    / "tcga_primary_tumor_rnaseq_final_case_level_unstranded_raw_counts.npy"
)

FINAL_RNA_FEATURE_INDEX_PATH = (
    INTEGRATION_EXPRESSION_OUTPUT_DIR
    / "tcga_primary_tumor_rnaseq_final_case_level_gene_feature_index.csv"
)

FINAL_HM27_MATRIX_PATH = (
    INTEGRATION_METHYLATION_OUTPUT_DIR
    / "tcga_primary_tumor_methylation_hm27_final_case_level_beta_values.npy"
)

FINAL_HM450_MATRIX_PATH = (
    INTEGRATION_METHYLATION_OUTPUT_DIR
    / "tcga_primary_tumor_methylation_hm450_final_case_level_beta_values.npy"
)

FINAL_SHARED_METHYLATION_MATRIX_PATH = (
    INTEGRATION_METHYLATION_OUTPUT_DIR
    / (
        "tcga_primary_tumor_methylation_shared_hm27_hm450_"
        "final_case_level_beta_values.npy"
    )
)

FINAL_SAMPLE_MAPPING_PATH = (
    INTEGRATION_METADATA_OUTPUT_DIR
    / "tcga_primary_tumor_multiomic_final_case_level_sample_mapping.csv"
)

FINAL_PROBE_MAPPING_PATH = (
    INTEGRATION_METADATA_OUTPUT_DIR
    / "tcga_primary_tumor_methylation_final_probe_mapping.csv"
)

FINAL_CONSUMABLES_METADATA_PATH = (
    INTEGRATION_METADATA_OUTPUT_DIR
    / "tcga_primary_tumor_multiomic_final_consumables_metadata.json"
)


FINAL_CONSUMABLE_PATHS = {
    "rna_matrix": FINAL_RNA_MATRIX_PATH,
    "rna_feature_index": FINAL_RNA_FEATURE_INDEX_PATH,
    "hm27_matrix": FINAL_HM27_MATRIX_PATH,
    "hm450_matrix": FINAL_HM450_MATRIX_PATH,
    "shared_methylation_matrix": FINAL_SHARED_METHYLATION_MATRIX_PATH,
    "sample_mapping": FINAL_SAMPLE_MAPPING_PATH,
    "probe_mapping": FINAL_PROBE_MAPPING_PATH,
    "metadata": FINAL_CONSUMABLES_METADATA_PATH,
}


def integration_inprogress_path(output_path):
    return output_path.with_name(
        f"{output_path.stem}.inprogress{output_path.suffix}"
    )


FINAL_CONSUMABLE_TEMP_PATHS = {
    artifact_name: integration_inprogress_path(output_path)
    for artifact_name, output_path in FINAL_CONSUMABLE_PATHS.items()
}


print("Final multi-omic consumable paths configured.")

for artifact_name, output_path in FINAL_CONSUMABLE_PATHS.items():
    print(f"{artifact_name}: {project_relative_path(output_path)}")

Final multi-omic consumable paths configured.
rna_matrix: data/interim/expression/tcga_primary_tumor_rnaseq_final_case_level_unstranded_raw_counts.npy
rna_feature_index: data/interim/expression/tcga_primary_tumor_rnaseq_final_case_level_gene_feature_index.csv
hm27_matrix: data/interim/methylation/tcga_primary_tumor_methylation_hm27_final_case_level_beta_values.npy
hm450_matrix: data/interim/methylation/tcga_primary_tumor_methylation_hm450_final_case_level_beta_values.npy
shared_methylation_matrix: data/interim/methylation/tcga_primary_tumor_methylation_shared_hm27_hm450_final_case_level_beta_values.npy
sample_mapping: data/interim/metadata/tcga_primary_tumor_multiomic_final_case_level_sample_mapping.csv
probe_mapping: data/interim/metadata/tcga_primary_tumor_methylation_final_probe_mapping.csv
metadata: data/interim/metadata/tcga_primary_tumor_multiomic_final_consumables_metadata.json


In [34]:
# =============================================================================
# Construct the final primary-cohort sample mapping
# =============================================================================

required_final_mapping_columns = [
    "project_id",
    CASE_KEY,
    SAMPLE_KEY,
    "rna_case_uuid",
    "rna_aliquot_uuid",
    "rna_aliquot_submitter_id",
    RNA_FILE_KEY,
    "rna_file_name",
    "methylation_case_uuid",
    "methylation_aliquot_uuid",
    "methylation_aliquot_submitter_id",
    METHYLATION_FILE_KEY,
    "methylation_file_name",
    "methylation_platform",
    "methylation_qc_eligible_for_downstream_selection",
    "pair_eligible_after_methylation_qc",
    "case_level_row_decision",
]

missing_final_mapping_columns = sorted(
    set(required_final_mapping_columns)
    - set(final_multiomic_pair_mapping.columns)
)

if missing_final_mapping_columns:
    raise KeyError(
        "The primary mapping is missing required columns: "
        + ", ".join(missing_final_mapping_columns)
    )


primary_mapping_mask = (
    final_multiomic_pair_mapping[
        "case_level_primary_inclusion"
    ]
    .fillna(False)
    .astype(bool)
)


selected_pair_mapping = (
    final_multiomic_pair_mapping.loc[
        primary_mapping_mask,
        required_final_mapping_columns,
    ]
    .copy()
)


identifier_columns = [
    "project_id",
    CASE_KEY,
    SAMPLE_KEY,
    "rna_case_uuid",
    "rna_aliquot_uuid",
    "rna_aliquot_submitter_id",
    RNA_FILE_KEY,
    "rna_file_name",
    "methylation_case_uuid",
    "methylation_aliquot_uuid",
    "methylation_aliquot_submitter_id",
    METHYLATION_FILE_KEY,
    "methylation_file_name",
    "methylation_platform",
]

for column in identifier_columns:
    selected_pair_mapping[column] = (
        selected_pair_mapping[column].astype("string")
    )


# -----------------------------------------------------------------------------
# Add RNA-seq file-level QC and candidate-matrix coordinates
# -----------------------------------------------------------------------------

rna_context_columns = [
    RNA_FILE_KEY,
    "matrix_column_index",
    "rna_local_path",
    "rna_qc_eligible_for_downstream_selection",
    "gene_assigned_fraction_of_accounted",
]

rna_context = (
    rna_file_qc_inventory[
        rna_context_columns
    ]
    .copy()
    .rename(
        columns={
            "matrix_column_index": (
                "rna_candidate_matrix_column_index"
            ),
        }
    )
)

rna_context[RNA_FILE_KEY] = (
    rna_context[RNA_FILE_KEY].astype("string")
)

if not rna_context[RNA_FILE_KEY].is_unique:
    raise ValueError(
        "RNA-seq QC context is not unique by rna_file_id."
    )


# -----------------------------------------------------------------------------
# Add methylation file-level QC
# -----------------------------------------------------------------------------

methylation_context_columns = [
    METHYLATION_FILE_KEY,
    "missing_beta_fraction",
]

methylation_context = (
    methylation_file_qc_metrics[
        methylation_context_columns
    ]
    .copy()
)

methylation_context[METHYLATION_FILE_KEY] = (
    methylation_context[METHYLATION_FILE_KEY].astype("string")
)

if not methylation_context[METHYLATION_FILE_KEY].is_unique:
    raise ValueError(
        "Methylation QC context is not unique by methylation_file_id."
    )


# -----------------------------------------------------------------------------
# Add frozen RNA-seq file-index provenance
# -----------------------------------------------------------------------------

rna_index_context = (
    rna_file_index[
        [
            "file_id",
            "file_name",
            "project_id",
            "case_id",
            "sample_id",
        ]
    ]
    .copy()
    .rename(
        columns={
            "file_id": RNA_FILE_KEY,
            "file_name": "rna_index_file_name",
            "project_id": "rna_index_project_id",
            "case_id": "rna_index_case_submitter_id",
            "sample_id": "rna_index_sample_submitter_id",
        }
    )
)

for column in [
    RNA_FILE_KEY,
    "rna_index_file_name",
    "rna_index_project_id",
    "rna_index_case_submitter_id",
    "rna_index_sample_submitter_id",
]:
    rna_index_context[column] = (
        rna_index_context[column].astype("string")
    )

if not rna_index_context[RNA_FILE_KEY].is_unique:
    raise ValueError(
        "Frozen RNA-seq file index is not unique by file ID."
    )


# -----------------------------------------------------------------------------
# Add frozen methylation file-index provenance
# -----------------------------------------------------------------------------

methylation_index_context = (
    methylation_file_index[
        [
            "file_id",
            "file_name",
            "platform",
            "project_id",
            "case_uuid",
            "case_submitter_id",
            "sample_submitter_id",
            "aliquot_uuid",
            "aliquot_submitter_id",
        ]
    ]
    .copy()
    .rename(
        columns={
            "file_id": METHYLATION_FILE_KEY,
            "file_name": "methylation_index_file_name",
            "platform": "methylation_index_platform",
            "project_id": "methylation_index_project_id",
            "case_uuid": "methylation_index_case_uuid",
            "case_submitter_id": (
                "methylation_index_case_submitter_id"
            ),
            "sample_submitter_id": (
                "methylation_index_sample_submitter_id"
            ),
            "aliquot_uuid": (
                "methylation_index_aliquot_uuid"
            ),
            "aliquot_submitter_id": (
                "methylation_index_aliquot_submitter_id"
            ),
        }
    )
)

methylation_index_context[METHYLATION_FILE_KEY] = (
    methylation_index_context[METHYLATION_FILE_KEY]
    .astype("string")
)

if not methylation_index_context[METHYLATION_FILE_KEY].is_unique:
    raise ValueError(
        "Frozen methylation file index is not unique by file ID."
    )


# -----------------------------------------------------------------------------
# Merge the contexts
# -----------------------------------------------------------------------------

final_sample_mapping = (
    selected_pair_mapping
    .merge(
        rna_context,
        on=RNA_FILE_KEY,
        how="left",
        validate="one_to_one",
    )
    .merge(
        methylation_context,
        on=METHYLATION_FILE_KEY,
        how="left",
        validate="one_to_one",
    )
    .merge(
        rna_index_context,
        on=RNA_FILE_KEY,
        how="left",
        validate="one_to_one",
    )
    .merge(
        methylation_index_context,
        on=METHYLATION_FILE_KEY,
        how="left",
        validate="one_to_one",
    )
)


# -----------------------------------------------------------------------------
# Resolve project-relative methylation payload paths
# -----------------------------------------------------------------------------

methylation_payload_paths = [
    (
        METHYLATION_DOWNLOAD_DIR
        / str(file_id)
        / str(file_name)
    )
    for file_id, file_name in zip(
        final_sample_mapping[METHYLATION_FILE_KEY],
        final_sample_mapping["methylation_file_name"],
    )
]

final_sample_mapping["methylation_local_path"] = [
    project_relative_path(path)
    for path in methylation_payload_paths
]

print("Final primary-cohort sample mapping constructed.")
print(f"Rows: {len(final_sample_mapping):,}")

Final primary-cohort sample mapping constructed.
Rows: 9,965


In [35]:
# =============================================================================
# Assign final matrix coordinates and validate sample identity
# =============================================================================

final_sample_mapping = (
    final_sample_mapping
    .sort_values(
        [
            CASE_KEY,
            SAMPLE_KEY,
            RNA_FILE_KEY,
            METHYLATION_FILE_KEY,
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)


index_columns_to_rebuild = [
    "final_sample_column_index",
    "rna_final_matrix_column_index",
    "hm27_matrix_column_index",
    "hm450_matrix_column_index",
    "shared_matrix_column_index",
]

final_sample_mapping = final_sample_mapping.drop(
    columns=[
        column
        for column in index_columns_to_rebuild
        if column in final_sample_mapping.columns
    ]
)


final_sample_mapping["final_sample_column_index"] = (
    np.arange(
        len(final_sample_mapping),
        dtype=np.int64,
    )
)

final_sample_mapping["rna_final_matrix_column_index"] = (
    final_sample_mapping["final_sample_column_index"]
)

final_sample_mapping["shared_matrix_column_index"] = (
    final_sample_mapping["final_sample_column_index"]
)


for platform, column_name in [
    (HM27_PLATFORM, "hm27_matrix_column_index"),
    (HM450_PLATFORM, "hm450_matrix_column_index"),
]:
    platform_index = pd.Series(
        pd.NA,
        index=final_sample_mapping.index,
        dtype="Int64",
    )

    platform_mask = (
        final_sample_mapping["methylation_platform"]
        .eq(platform)
    )

    platform_index.loc[platform_mask] = np.arange(
        int(platform_mask.sum()),
        dtype=np.int64,
    )

    final_sample_mapping[column_name] = platform_index


# -----------------------------------------------------------------------------
# Check payload availability
# -----------------------------------------------------------------------------

final_sample_mapping["rna_payload_exists"] = (
    final_sample_mapping["rna_local_path"]
    .map(
        lambda path: (
            PROJECT_ROOT / str(path)
        ).is_file()
    )
)

final_sample_mapping["methylation_payload_exists"] = (
    final_sample_mapping["methylation_local_path"]
    .map(
        lambda path: (
            PROJECT_ROOT / str(path)
        ).is_file()
    )
)


# -----------------------------------------------------------------------------
# Validate the final mapping
# -----------------------------------------------------------------------------

final_mapping_checks = {
    "final_mapping_is_not_empty": (
        not final_sample_mapping.empty
    ),
    "final_case_count_matches_policy": (
        len(final_sample_mapping)
        == FINAL_PRIMARY_CASE_COUNT
    ),
    "one_row_per_case": (
        final_sample_mapping[CASE_KEY].is_unique
    ),
    "candidate_pair_keys_are_unique": (
        not final_sample_mapping[
            [RNA_FILE_KEY, METHYLATION_FILE_KEY]
        ]
        .duplicated()
        .any()
    ),
    "final_sample_column_indices_are_complete": (
        final_sample_mapping[
            "final_sample_column_index"
        ]
        .tolist()
        == list(range(len(final_sample_mapping)))
    ),
    "RNA_candidate_matrix_indices_are_complete": (
        final_sample_mapping[
            "rna_candidate_matrix_column_index"
        ]
        .notna()
        .all()
    ),
    "RNA_candidate_matrix_indices_are_unique": (
        final_sample_mapping[
            "rna_candidate_matrix_column_index"
        ]
        .is_unique
    ),
    "all_RNA_QC_gates_are_true": (
        final_sample_mapping[
            "rna_qc_eligible_for_downstream_selection"
        ]
        .astype("boolean")
        .fillna(False)
        .all()
    ),
    "all_methylation_QC_gates_are_true": (
        final_sample_mapping[
            "methylation_qc_eligible_for_downstream_selection"
        ]
        .astype("boolean")
        .fillna(False)
        .all()
    ),
    "all_pair_QC_gates_are_true": (
        final_sample_mapping[
            "pair_eligible_after_methylation_qc"
        ]
        .astype("boolean")
        .fillna(False)
        .all()
    ),
    "expected_platforms_are_present": (
        set(
            final_sample_mapping["methylation_platform"]
        )
        == {
            HM27_PLATFORM,
            HM450_PLATFORM,
        }
    ),
    "RNA_file_names_match_frozen_index": (
        final_sample_mapping["rna_file_name"]
        .eq(final_sample_mapping["rna_index_file_name"])
        .all()
    ),
    "RNA_projects_match_frozen_index": (
        final_sample_mapping["project_id"]
        .eq(final_sample_mapping["rna_index_project_id"])
        .all()
    ),
    "RNA_case_submitter_IDs_match_frozen_index": (
        final_sample_mapping[CASE_KEY]
        .eq(
            final_sample_mapping[
                "rna_index_case_submitter_id"
            ]
        )
        .fillna(False)
        .all()
    ),

    "RNA_sample_IDs_match_frozen_index": (
        final_sample_mapping[SAMPLE_KEY]
        .eq(
            final_sample_mapping[
                "rna_index_sample_submitter_id"
            ]
        )
        .fillna(False)
        .all()
    ),
    "methylation_file_names_match_frozen_index": (
        final_sample_mapping["methylation_file_name"]
        .eq(
            final_sample_mapping[
                "methylation_index_file_name"
            ]
        )
        .all()
    ),
    "methylation_platforms_match_frozen_index": (
        final_sample_mapping["methylation_platform"]
        .eq(
            final_sample_mapping[
                "methylation_index_platform"
            ]
        )
        .all()
    ),
    "methylation_case_UUIDs_match_frozen_index": (
        final_sample_mapping["methylation_case_uuid"]
        .eq(
            final_sample_mapping[
                "methylation_index_case_uuid"
            ]
        )
        .all()
    ),
    "methylation_sample_IDs_match_frozen_index": (
        final_sample_mapping[SAMPLE_KEY]
        .eq(
            final_sample_mapping[
                "methylation_index_sample_submitter_id"
            ]
        )
        .all()
    ),
    "methylation_aliquots_match_frozen_index": (
        final_sample_mapping[
            "methylation_aliquot_uuid"
        ]
        .eq(
            final_sample_mapping[
                "methylation_index_aliquot_uuid"
            ]
        )
        .all()
    ),
    "all_RNA_payloads_are_available": (
        final_sample_mapping[
            "rna_payload_exists"
        ].all()
    ),
    "all_methylation_payloads_are_available": (
        final_sample_mapping[
            "methylation_payload_exists"
        ].all()
    ),
}


print("Final sample-mapping checks:")

for check_name, check_passed in final_mapping_checks.items():
    print(f"{check_name}: {check_passed}")


if not all(final_mapping_checks.values()):
    failed_checks = [
        check_name
        for check_name, check_passed
        in final_mapping_checks.items()
        if not check_passed
    ]

    raise ValueError(
        "Final sample-mapping validation failed: "
        + ", ".join(failed_checks)
    )


final_sample_mapping_output_columns = [
    "final_sample_column_index",
    CASE_KEY,
    SAMPLE_KEY,
    "project_id",
    "rna_case_uuid",
    "rna_aliquot_uuid",
    "rna_aliquot_submitter_id",
    RNA_FILE_KEY,
    "rna_file_name",
    "rna_candidate_matrix_column_index",
    "rna_final_matrix_column_index",
    "rna_qc_eligible_for_downstream_selection",
    "gene_assigned_fraction_of_accounted",
    "methylation_case_uuid",
    "methylation_aliquot_uuid",
    "methylation_aliquot_submitter_id",
    METHYLATION_FILE_KEY,
    "methylation_file_name",
    "methylation_platform",
    "hm27_matrix_column_index",
    "hm450_matrix_column_index",
    "shared_matrix_column_index",
    "missing_beta_fraction",
    "methylation_qc_eligible_for_downstream_selection",
    "pair_eligible_after_methylation_qc",
    "case_level_row_decision",
    "rna_local_path",
    "methylation_local_path",
    "rna_payload_exists",
    "methylation_payload_exists",
]

final_sample_mapping = final_sample_mapping[
    final_sample_mapping_output_columns
].copy()


print()
print("Final sample mapping ready for publication.")
print(f"Primary cases: {len(final_sample_mapping):,}")
print()
print("Selected methylation files by platform:")
print(
    final_sample_mapping[
        "methylation_platform"
    ].value_counts()
)

Final sample-mapping checks:
final_mapping_is_not_empty: True
final_case_count_matches_policy: True
one_row_per_case: True
candidate_pair_keys_are_unique: True
final_sample_column_indices_are_complete: True
RNA_candidate_matrix_indices_are_complete: True
RNA_candidate_matrix_indices_are_unique: True
all_RNA_QC_gates_are_true: True
all_methylation_QC_gates_are_true: True
all_pair_QC_gates_are_true: True
expected_platforms_are_present: True
RNA_file_names_match_frozen_index: True
RNA_projects_match_frozen_index: True
RNA_case_submitter_IDs_match_frozen_index: True
RNA_sample_IDs_match_frozen_index: True
methylation_file_names_match_frozen_index: True
methylation_platforms_match_frozen_index: True
methylation_case_UUIDs_match_frozen_index: True
methylation_sample_IDs_match_frozen_index: True
methylation_aliquots_match_frozen_index: True
all_RNA_payloads_are_available: True
all_methylation_payloads_are_available: True

Final sample mapping ready for publication.
Primary cases: 9,965

Selec

In [36]:
# =============================================================================
# Prepare eligible platform-specific and shared probe mappings
# =============================================================================

probe_mapping_source = probe_level_qc_metrics.copy()
shared_probe_mapping_source = shared_probe_qc_inventory.copy()


probe_mapping_source["methylation_platform"] = (
    probe_mapping_source["methylation_platform"]
    .astype("string")
)

probe_mapping_source["probe_id"] = (
    probe_mapping_source["probe_id"]
    .astype("string")
)

probe_mapping_source["platform_probe_order"] = (
    pd.to_numeric(
        probe_mapping_source["platform_probe_order"],
        errors="raise",
    )
    .astype(np.int64)
)

probe_mapping_source[
    "probe_qc_eligible_within_platform"
] = (
    probe_mapping_source[
        "probe_qc_eligible_within_platform"
    ]
    .astype("boolean")
)

if (
    probe_mapping_source[
        "probe_qc_eligible_within_platform"
    ]
    .isna()
    .any()
):
    raise ValueError(
        "Platform-specific probe eligibility contains missing values."
    )

probe_mapping_source[
    "probe_qc_eligible_within_platform"
] = (
    probe_mapping_source[
        "probe_qc_eligible_within_platform"
    ]
    .astype(bool)
)


shared_probe_mapping_source["probe_id"] = (
    shared_probe_mapping_source["probe_id"]
    .astype("string")
)

shared_probe_mapping_source["shared_probe_order"] = (
    pd.to_numeric(
        shared_probe_mapping_source["shared_probe_order"],
        errors="raise",
    )
    .astype(np.int64)
)

shared_probe_mapping_source[
    "shared_probe_qc_eligible"
] = (
    shared_probe_mapping_source[
        "shared_probe_qc_eligible"
    ]
    .astype("boolean")
)

if (
    shared_probe_mapping_source[
        "shared_probe_qc_eligible"
    ]
    .isna()
    .any()
):
    raise ValueError(
        "Shared-probe eligibility contains missing values."
    )

shared_probe_mapping_source[
    "shared_probe_qc_eligible"
] = (
    shared_probe_mapping_source[
        "shared_probe_qc_eligible"
    ]
    .astype(bool)
)


platform_probe_tables = {}
platform_probe_ids = {}
platform_eligible_positions = {}

for platform in [HM27_PLATFORM, HM450_PLATFORM]:
    platform_table = (
        probe_mapping_source.loc[
            probe_mapping_source["methylation_platform"]
            .eq(platform)
        ]
        .sort_values(
            "platform_probe_order",
            kind="stable",
        )
        .reset_index(drop=True)
    )

    expected_orders = list(range(len(platform_table)))

    platform_checks = {
        f"{platform}_probe_table_is_not_empty": (
            not platform_table.empty
        ),
        f"{platform}_probe_ids_are_unique": (
            platform_table["probe_id"].is_unique
        ),
        f"{platform}_probe_order_is_complete": (
            platform_table["platform_probe_order"].tolist()
            == expected_orders
        ),
    }

    if not all(platform_checks.values()):
        raise ValueError(
            "Platform-specific probe-order validation failed: "
            + str(platform_checks)
        )

    platform_probe_tables[platform] = platform_table
    platform_probe_ids[platform] = pd.Index(
        platform_table["probe_id"],
        name="probe_id",
    )
    platform_eligible_positions[platform] = (
        platform_table[
            "probe_qc_eligible_within_platform"
        ]
        .to_numpy(dtype=bool)
        .nonzero()[0]
    )


shared_probe_table = (
    shared_probe_mapping_source
    .sort_values(
        "shared_probe_order",
        kind="stable",
    )
    .reset_index(drop=True)
)

shared_probe_checks = {
    "shared_probe_table_is_not_empty": (
        not shared_probe_table.empty
    ),
    "shared_probe_ids_are_unique": (
        shared_probe_table["probe_id"].is_unique
    ),
    "shared_probe_order_is_complete": (
        shared_probe_table["shared_probe_order"].tolist()
        == list(range(len(shared_probe_table)))
    ),
}

if not all(shared_probe_checks.values()):
    raise ValueError(
        "Shared-probe mapping validation failed: "
        + str(shared_probe_checks)
    )


shared_probe_ids = pd.Index(
    shared_probe_table.loc[
        shared_probe_table["shared_probe_qc_eligible"],
        "probe_id",
    ],
    name="probe_id",
)

shared_probe_positions_by_platform = {}

for platform, platform_ids in platform_probe_ids.items():
    positions = platform_ids.get_indexer(shared_probe_ids)

    if (positions < 0).any():
        raise ValueError(
            f"Shared probes are absent from {platform} probe order."
        )

    shared_probe_positions_by_platform[platform] = positions

# =============================================================================
# Define methylation representation labels
# =============================================================================

HM27_PLATFORM = "Illumina Human Methylation 27"
HM450_PLATFORM = "Illumina Human Methylation 450"
SHARED_METHYLATION_REPRESENTATION = "shared_hm27_hm450"

# -----------------------------------------------------------------------------
# Build a single explicit probe mapping for all final representations
# -----------------------------------------------------------------------------

probe_mapping_frames = []

for representation, platform in [
    ("hm27", HM27_PLATFORM),
    ("hm450", HM450_PLATFORM),
]:
    platform_table = platform_probe_tables[platform]
    eligible_mask = (
        platform_table[
            "probe_qc_eligible_within_platform"
        ]
    )
    eligible_table = platform_table.loc[
        eligible_mask
    ].reset_index(drop=True)

    platform_frame = pd.DataFrame(
        {
            "representation": representation,
            "methylation_platform": platform,
            "matrix_row_index": np.arange(
                len(eligible_table),
                dtype=np.int64,
            ),
            "probe_id": eligible_table["probe_id"].to_numpy(
                dtype=object
            ),
            "source_platform_probe_order": (
                eligible_table[
                    "platform_probe_order"
                ]
                .to_numpy(dtype=np.int64)
            ),
            "source_shared_probe_order": pd.Series(
                pd.NA,
                index=eligible_table.index,
                dtype="Int64",
            ),
            "probe_missing_beta_fraction": (
                eligible_table[
                    "missing_beta_fraction"
                ]
                .to_numpy(dtype=float)
            ),
            "probe_qc_eligibility_scope": (
                "platform_specific"
            ),
            "probe_qc_eligible": True,
            "hm27_missing_beta_fraction": (
                eligible_table[
                    "missing_beta_fraction"
                ].to_numpy(dtype=float)
                if platform == HM27_PLATFORM
                else np.nan
            ),
            "hm450_missing_beta_fraction": (
                eligible_table[
                    "missing_beta_fraction"
                ].to_numpy(dtype=float)
                if platform == HM450_PLATFORM
                else np.nan
            ),
            "hm27_probe_qc_eligible": (
                True
                if platform == HM27_PLATFORM
                else pd.NA
            ),
            "hm450_probe_qc_eligible": (
                True
                if platform == HM450_PLATFORM
                else pd.NA
            ),
            "shared_probe_qc_eligible": pd.NA,
        }
    )

    probe_mapping_frames.append(platform_frame)


eligible_shared_table = shared_probe_table.loc[
    shared_probe_table["shared_probe_qc_eligible"]
].reset_index(drop=True)

shared_frame = pd.DataFrame(
    {
        "representation": (
            SHARED_METHYLATION_REPRESENTATION
        ),
        "methylation_platform": (
            SHARED_METHYLATION_REPRESENTATION
        ),
        "matrix_row_index": np.arange(
            len(eligible_shared_table),
            dtype=np.int64,
        ),
        "probe_id": eligible_shared_table["probe_id"].to_numpy(
            dtype=object
        ),
        "source_platform_probe_order": pd.Series(
            pd.NA,
            index=eligible_shared_table.index,
            dtype="Int64",
        ),
        "source_shared_probe_order": (
            eligible_shared_table[
                "shared_probe_order"
            ]
            .to_numpy(dtype=np.int64)
        ),
        "probe_missing_beta_fraction": np.nan,
        "probe_qc_eligibility_scope": (
            "shared_both_platforms"
        ),
        "probe_qc_eligible": True,
        "hm27_missing_beta_fraction": (
            eligible_shared_table[
                "hm27_missing_beta_fraction"
            ]
            .to_numpy(dtype=float)
        ),
        "hm450_missing_beta_fraction": (
            eligible_shared_table[
                "hm450_missing_beta_fraction"
            ]
            .to_numpy(dtype=float)
        ),
        "hm27_probe_qc_eligible": (
            eligible_shared_table[
                "hm27_probe_qc_eligible"
            ]
            .to_numpy(dtype=bool)
        ),
        "hm450_probe_qc_eligible": (
            eligible_shared_table[
                "hm450_probe_qc_eligible"
            ]
            .to_numpy(dtype=bool)
        ),
        "shared_probe_qc_eligible": True,
    }
)

probe_mapping_frames.append(shared_frame)

final_probe_mapping = pd.concat(
    probe_mapping_frames,
    ignore_index=True,
)


probe_representation_counts = (
    final_probe_mapping
    .groupby(
        "representation",
        sort=False,
    )
    .size()
)

probe_mapping_checks = {
    "final_probe_mapping_is_not_empty": (
        not final_probe_mapping.empty
    ),
    "hm27_count_matches_policy": (
        int(
            probe_representation_counts.get("hm27", 0)
        )
        == len(platform_eligible_positions[HM27_PLATFORM])
    ),
    "hm450_count_matches_policy": (
        int(
            probe_representation_counts.get("hm450", 0)
        )
        == len(platform_eligible_positions[HM450_PLATFORM])
    ),
    "shared_count_matches_policy": (
        int(
            probe_representation_counts.get(
                SHARED_METHYLATION_REPRESENTATION,
                0,
            )
        )
        == int(
            shared_probe_table[
                "shared_probe_qc_eligible"
            ].sum()
        )
    ),
    "representation_row_indices_are_unique": (
        not final_probe_mapping.duplicated(
            [
                "representation",
                "matrix_row_index",
            ]
        ).any()
    ),
    "all_final_probe_rows_are_eligible": (
        final_probe_mapping[
            "probe_qc_eligible"
        ]
        .astype("boolean")
        .fillna(False)
        .all()
    ),
}

print("Final probe-mapping checks:")

for check_name, check_passed in probe_mapping_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(probe_mapping_checks.values()):
    raise ValueError(
        "Final probe-mapping construction failed."
    )

print()
print("Final probe mapping prepared.")
print(probe_representation_counts)
print()
print("Eligible shared probes:", len(shared_probe_ids))

Final probe-mapping checks:
final_probe_mapping_is_not_empty: True
hm27_count_matches_policy: True
hm450_count_matches_policy: True
shared_count_matches_policy: True
representation_row_indices_are_unique: True
all_final_probe_rows_are_eligible: True

Final probe mapping prepared.
representation
hm27                  24303
hm450                407696
shared_hm27_hm450     23356
dtype: int64

Eligible shared probes: 23356


In [37]:
# =============================================================================
# Construct and validate the final case-level RNA-seq matrix
# =============================================================================

RNA_MATRIX_CHUNK_COLUMNS = 512

TEMP_FINAL_RNA_MATRIX_PATH = FINAL_RNA_MATRIX_PATH.with_name(
    FINAL_RNA_MATRIX_PATH.stem + ".tmp" + FINAL_RNA_MATRIX_PATH.suffix
)

TEMP_FINAL_RNA_FEATURE_INDEX_PATH = (
    FINAL_RNA_FEATURE_INDEX_PATH.with_name(
        FINAL_RNA_FEATURE_INDEX_PATH.stem
        + ".tmp"
        + FINAL_RNA_FEATURE_INDEX_PATH.suffix
    )
)

for temporary_path in [
    TEMP_FINAL_RNA_MATRIX_PATH,
    TEMP_FINAL_RNA_FEATURE_INDEX_PATH,
]:
    temporary_path.unlink(missing_ok=True)


# -----------------------------------------------------------------------------
# Load the candidate RNA-seq matrix and canonical feature index
# -----------------------------------------------------------------------------

candidate_rna_matrix = np.load(
    RAW_COUNT_MATRIX_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

candidate_gene_feature_index = pd.read_csv(
    RNA_GENE_FEATURE_INDEX_PATH,
    low_memory=False,
)


# -----------------------------------------------------------------------------
# Resolve final sample columns from the candidate matrix
# -----------------------------------------------------------------------------

final_rna_source_indices = (
    final_sample_mapping[
        "rna_candidate_matrix_column_index"
    ]
    .astype("int64")
    .to_numpy()
)

final_sample_indices = (
    final_sample_mapping[
        "final_sample_column_index"
    ]
    .astype("int64")
    .to_numpy()
)

expected_final_sample_indices = np.arange(
    len(final_sample_mapping),
    dtype=np.int64,
)


rna_matrix_input_checks = {
    "candidate_matrix_is_two_dimensional": (
        candidate_rna_matrix.ndim == 2
    ),
    "candidate_matrix_is_fortran_contiguous": (
        candidate_rna_matrix.flags.f_contiguous
    ),
    "candidate_matrix_dtype_is_uint32": (
        candidate_rna_matrix.dtype == np.dtype("uint32")
    ),
    "candidate_feature_count_matches_matrix": (
        len(candidate_gene_feature_index)
        == candidate_rna_matrix.shape[0]
    ),
    "final_sample_indices_are_complete": (
        np.array_equal(
            final_sample_indices,
            expected_final_sample_indices,
        )
    ),
    "final_RNA_source_indices_are_unique": (
        len(final_rna_source_indices)
        == len(np.unique(final_rna_source_indices))
    ),
    "final_RNA_source_indices_are_in_range": (
        final_rna_source_indices.min(initial=0)
        >= 0
        and final_rna_source_indices.max(initial=0)
        < candidate_rna_matrix.shape[1]
    ),
}


print("Final RNA-matrix input checks:")

for check_name, check_passed in rna_matrix_input_checks.items():
    print(f"{check_name}: {check_passed}")


if not all(rna_matrix_input_checks.values()):
    raise ValueError(
        "Final RNA-matrix input validation failed."
    )


# -----------------------------------------------------------------------------
# Construct the final matrix in final sample order
# -----------------------------------------------------------------------------

final_rna_matrix_shape = (
    candidate_rna_matrix.shape[0],
    len(final_sample_mapping),
)

final_rna_matrix = np.lib.format.open_memmap(
    TEMP_FINAL_RNA_MATRIX_PATH,
    mode="w+",
    dtype=candidate_rna_matrix.dtype,
    shape=final_rna_matrix_shape,
    fortran_order=True,
)

for start in range(
    0,
    len(final_sample_mapping),
    RNA_MATRIX_CHUNK_COLUMNS,
):
    stop = min(
        start + RNA_MATRIX_CHUNK_COLUMNS,
        len(final_sample_mapping),
    )

    source_columns = final_rna_source_indices[start:stop]

    final_rna_matrix[:, start:stop] = (
        candidate_rna_matrix[:, source_columns]
    )

final_rna_matrix.flush()
del final_rna_matrix


# -----------------------------------------------------------------------------
# Preserve the canonical gene-feature index for the final matrix
# -----------------------------------------------------------------------------

candidate_gene_feature_index.to_csv(
    TEMP_FINAL_RNA_FEATURE_INDEX_PATH,
    index=False,
    lineterminator="\n",
)


# -----------------------------------------------------------------------------
# Validate temporary artifacts
# -----------------------------------------------------------------------------

temporary_final_rna_matrix = np.load(
    TEMP_FINAL_RNA_MATRIX_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

temporary_final_rna_feature_index = pd.read_csv(
    TEMP_FINAL_RNA_FEATURE_INDEX_PATH,
    low_memory=False,
)

values_match_selected_candidate_columns = True

for start in range(
    0,
    len(final_sample_mapping),
    RNA_MATRIX_CHUNK_COLUMNS,
):
    stop = min(
        start + RNA_MATRIX_CHUNK_COLUMNS,
        len(final_sample_mapping),
    )

    expected_values = candidate_rna_matrix[
        :,
        final_rna_source_indices[start:stop],
    ]

    observed_values = temporary_final_rna_matrix[
        :,
        start:stop,
    ]

    if not np.array_equal(
        observed_values,
        expected_values,
    ):
        values_match_selected_candidate_columns = False
        break


final_rna_matrix_checks = {
    "temporary_matrix_exists": (
        TEMP_FINAL_RNA_MATRIX_PATH.is_file()
    ),
    "temporary_feature_index_exists": (
        TEMP_FINAL_RNA_FEATURE_INDEX_PATH.is_file()
    ),
    "temporary_matrix_shape_matches_mapping": (
        temporary_final_rna_matrix.shape
        == final_rna_matrix_shape
    ),
    "temporary_matrix_dtype_matches_candidate": (
        temporary_final_rna_matrix.dtype
        == candidate_rna_matrix.dtype
    ),
    "temporary_matrix_is_fortran_contiguous": (
        temporary_final_rna_matrix.flags.f_contiguous
    ),
    "temporary_feature_count_matches_matrix": (
        len(temporary_final_rna_feature_index)
        == temporary_final_rna_matrix.shape[0]
    ),
    "temporary_feature_index_matches_candidate": (
        temporary_final_rna_feature_index
        .astype("string")
        .equals(
            candidate_gene_feature_index
            .astype("string")
        )
    ),
    "temporary_values_match_selected_candidate_columns": (
        values_match_selected_candidate_columns
    ),
}


print()
print("Final RNA-matrix temporary-artifact checks:")

for check_name, check_passed in final_rna_matrix_checks.items():
    print(f"{check_name}: {check_passed}")


if not all(final_rna_matrix_checks.values()):
    TEMP_FINAL_RNA_MATRIX_PATH.unlink(missing_ok=True)
    TEMP_FINAL_RNA_FEATURE_INDEX_PATH.unlink(missing_ok=True)

    raise ValueError(
        "Final RNA matrix failed temporary-artifact validation."
    )


# -----------------------------------------------------------------------------
# Publish the validated final RNA artifacts
# -----------------------------------------------------------------------------

# Release memory-mapped files before Windows publication
for variable_name in [
    "observed_values",
    "expected_values",
    "temporary_final_rna_matrix",
    "candidate_rna_matrix",
    "final_rna_matrix",
    "published_final_rna_matrix",
]:
    globals().pop(variable_name, None)

gc.collect()

TEMP_FINAL_RNA_MATRIX_PATH.replace(
    FINAL_RNA_MATRIX_PATH
)

TEMP_FINAL_RNA_FEATURE_INDEX_PATH.replace(
    FINAL_RNA_FEATURE_INDEX_PATH
)

published_final_rna_matrix = np.load(
    FINAL_RNA_MATRIX_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

published_final_rna_checks = {
    "published_matrix_exists": (
        FINAL_RNA_MATRIX_PATH.is_file()
    ),
    "published_feature_index_exists": (
        FINAL_RNA_FEATURE_INDEX_PATH.is_file()
    ),
    "published_matrix_shape_matches_mapping": (
        published_final_rna_matrix.shape
        == final_rna_matrix_shape
    ),
    "published_matrix_dtype_is_uint32": (
        published_final_rna_matrix.dtype
        == np.dtype("uint32")
    ),
    "published_matrix_is_fortran_contiguous": (
        published_final_rna_matrix.flags.f_contiguous
    ),
}

print()
print("Published final RNA-matrix checks:")

for check_name, check_passed in published_final_rna_checks.items():
    print(f"{check_name}: {check_passed}")


if not all(published_final_rna_checks.values()):
    raise ValueError(
        "Published final RNA matrix validation failed."
    )


print()
print("Final case-level RNA-seq matrix published.")
print(
    "Shape: "
    f"{published_final_rna_matrix.shape[0]:,} genes × "
    f"{published_final_rna_matrix.shape[1]:,} cases"
)
print(
    "Matrix: "
    f"{project_relative_path(FINAL_RNA_MATRIX_PATH)}"
)
print(
    "Feature index: "
    f"{project_relative_path(FINAL_RNA_FEATURE_INDEX_PATH)}"
)

Final RNA-matrix input checks:
candidate_matrix_is_two_dimensional: True
candidate_matrix_is_fortran_contiguous: True
candidate_matrix_dtype_is_uint32: True
candidate_feature_count_matches_matrix: True
final_sample_indices_are_complete: True
final_RNA_source_indices_are_unique: True
final_RNA_source_indices_are_in_range: True

Final RNA-matrix temporary-artifact checks:
temporary_matrix_exists: True
temporary_feature_index_exists: True
temporary_matrix_shape_matches_mapping: True
temporary_matrix_dtype_matches_candidate: True
temporary_matrix_is_fortran_contiguous: True
temporary_feature_count_matches_matrix: True
temporary_feature_index_matches_candidate: True
temporary_values_match_selected_candidate_columns: True

Published final RNA-matrix checks:
published_matrix_exists: True
published_feature_index_exists: True
published_matrix_shape_matches_mapping: True
published_matrix_dtype_is_uint32: True
published_matrix_is_fortran_contiguous: True

Final case-level RNA-seq matrix published

In [38]:
# =============================================================================
# Define the canonical methylation-payload reader
# =============================================================================

METHYLATION_PAYLOAD_COLUMNS = [
    "probe_id",
    "beta_value",
]


def read_methylation_payload(file_path):
    payload = pd.read_csv(
        file_path,
        sep="\t",
        header=None,
        names=METHYLATION_PAYLOAD_COLUMNS,
        dtype="string",
        keep_default_na=False,
    )

    payload["probe_id"] = payload["probe_id"].str.strip()

    beta_text = payload["beta_value"].str.strip()

    missing_beta = (
        beta_text.eq("")
        | beta_text.str.upper().isin(["NA", "NAN"])
    )

    beta_numeric = pd.to_numeric(
        beta_text.mask(missing_beta),
        errors="coerce",
    )

    invalid_beta = (
        ~missing_beta
        & beta_numeric.isna()
    )

    if invalid_beta.any():
        raise ValueError(
            f"Non-numeric beta values detected in: {file_path}"
        )

    payload["beta_value"] = beta_numeric

    return payload

In [39]:
# =============================================================================
# Configure and validate the final HM27 matrix mapping
# =============================================================================

HM27_PLATFORM = "Illumina Human Methylation 27"
HM27_REPRESENTATION = "hm27"
HM27_MATRIX_DTYPE = np.dtype("float32")

FINAL_HM27_MATRIX_PATH = (
    Paths.methylation
    / "tcga_primary_tumor_methylation_hm27_final_case_level_beta_values.npy"
)

TEMP_FINAL_HM27_MATRIX_PATH = (
    FINAL_HM27_MATRIX_PATH.with_name(
        FINAL_HM27_MATRIX_PATH.stem
        + ".tmp"
        + FINAL_HM27_MATRIX_PATH.suffix
    )
)


hm27_sample_mapping = (
    final_sample_mapping.loc[
        final_sample_mapping["methylation_platform"]
        .eq(HM27_PLATFORM)
    ]
    .sort_values(
        "hm27_matrix_column_index",
        kind="stable",
    )
    .reset_index(drop=True)
)

hm27_probe_mapping = (
    final_probe_mapping.loc[
        final_probe_mapping["representation"]
        .eq(HM27_REPRESENTATION)
    ]
    .sort_values(
        "matrix_row_index",
        kind="stable",
    )
    .reset_index(drop=True)
)

hm27_reference_probe_ids = platform_probe_ids[HM27_PLATFORM]
hm27_eligible_probe_positions = (
    platform_eligible_positions[HM27_PLATFORM]
)

expected_hm27_probe_ids = (
    hm27_reference_probe_ids
    .take(hm27_eligible_probe_positions)
)

observed_hm27_probe_ids = pd.Index(
    hm27_probe_mapping["probe_id"].astype("string"),
    name="probe_id",
)

hm27_sample_column_indices = (
    hm27_sample_mapping[
        "hm27_matrix_column_index"
    ]
    .astype("int64")
    .to_numpy()
)

hm27_probe_row_indices = (
    hm27_probe_mapping["matrix_row_index"]
    .astype("int64")
    .to_numpy()
)

hm27_mapping_checks = {
    "HM27_sample_mapping_is_not_empty": (
        not hm27_sample_mapping.empty
    ),
    "HM27_sample_count_matches_policy": (
        len(hm27_sample_mapping) == 1_620
    ),
    "HM27_sample_indices_are_complete": (
        np.array_equal(
            hm27_sample_column_indices,
            np.arange(
                len(hm27_sample_mapping),
                dtype=np.int64,
            ),
        )
    ),
    "HM27_methylation_file_ids_are_unique": (
        hm27_sample_mapping[
            "methylation_file_id"
        ]
        .astype("string")
        .is_unique
    ),
    "HM27_payload_paths_are_complete": (
        hm27_sample_mapping[
            "methylation_local_path"
        ]
        .notna()
        .all()
    ),
    "HM27_payloads_are_available": (
        hm27_sample_mapping[
            "methylation_payload_exists"
        ]
        .astype("boolean")
        .fillna(False)
        .all()
    ),
    "HM27_probe_mapping_is_not_empty": (
        not hm27_probe_mapping.empty
    ),
    "HM27_probe_count_matches_policy": (
        len(hm27_probe_mapping)
        == len(hm27_eligible_probe_positions)
    ),
    "HM27_probe_indices_are_complete": (
        np.array_equal(
            hm27_probe_row_indices,
            np.arange(
                len(hm27_probe_mapping),
                dtype=np.int64,
            ),
        )
    ),
    "HM27_source_probe_order_matches_mapping": (
        np.array_equal(
            hm27_probe_mapping[
                "source_platform_probe_order"
            ]
            .astype("int64")
            .to_numpy(),
            hm27_eligible_probe_positions,
        )
    ),
    "HM27_probe_ids_match_mapping": (
        observed_hm27_probe_ids.equals(
            expected_hm27_probe_ids
        )
    ),
    "HM27_all_probe_rows_are_eligible": (
        hm27_probe_mapping[
            "probe_qc_eligible"
        ]
        .astype("boolean")
        .fillna(False)
        .all()
    ),
}


print("HM27 mapping checks:")

for check_name, check_passed in hm27_mapping_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(hm27_mapping_checks.values()):
    failed_checks = [
        check_name
        for check_name, check_passed
        in hm27_mapping_checks.items()
        if not check_passed
    ]

    raise ValueError(
        "HM27 mapping validation failed: "
        + ", ".join(failed_checks)
    )

print()
print("HM27 mapping ready.")
print(
    f"Expected matrix shape: "
    f"{len(hm27_probe_mapping):,} probes × "
    f"{len(hm27_sample_mapping):,} samples"
)

HM27 mapping checks:
HM27_sample_mapping_is_not_empty: True
HM27_sample_count_matches_policy: True
HM27_sample_indices_are_complete: True
HM27_methylation_file_ids_are_unique: True
HM27_payload_paths_are_complete: True
HM27_payloads_are_available: True
HM27_probe_mapping_is_not_empty: True
HM27_probe_count_matches_policy: True
HM27_probe_indices_are_complete: True
HM27_source_probe_order_matches_mapping: True
HM27_probe_ids_match_mapping: True
HM27_all_probe_rows_are_eligible: True

HM27 mapping ready.
Expected matrix shape: 24,303 probes × 1,620 samples


# ========================================================
# Construct the temporary final HM27 beta-value matrix
# ========================================================

import gc


for variable_name in [
    "temporary_hm27_matrix",
    "published_hm27_matrix",
    "flat_hm27_values",
]:
    globals().pop(variable_name, None)

gc.collect()

TEMP_FINAL_HM27_MATRIX_PATH.unlink(missing_ok=True)


def build_hm27_temporary_matrix():
    temporary_matrix = None
    current_file_id = None
    current_file_path = None

    selected_missing_beta_count = 0

    try:
        temporary_matrix = np.lib.format.open_memmap(
            TEMP_FINAL_HM27_MATRIX_PATH,
            mode="w+",
            dtype=HM27_MATRIX_DTYPE,
            shape=(
                len(hm27_probe_mapping),
                len(hm27_sample_mapping),
            ),
            fortran_order=True,
        )

        temporary_matrix[:] = np.nan

        for sample_position, record in enumerate(
            hm27_sample_mapping.itertuples(index=False),
            start=1,
        ):
            current_file_id = str(
                record.methylation_file_id
            )

            current_file_path = (
                PROJECT_ROOT
                / str(record.methylation_local_path)
            )

            payload = read_methylation_payload(
                current_file_path
            )

            observed_probe_ids = pd.Index(
                payload["probe_id"],
                name="probe_id",
            )

            if not observed_probe_ids.equals(
                hm27_reference_probe_ids
            ):
                raise ValueError(
                    "HM27 payload probe identity or order "
                    "does not match the canonical reference."
                )

            beta_values = payload[
                "beta_value"
            ].to_numpy(
                dtype=HM27_MATRIX_DTYPE
            )

            missing_mask = np.isnan(beta_values)
            numeric_beta_values = beta_values[
                ~missing_mask
            ]

            if not np.isfinite(
                numeric_beta_values
            ).all():
                raise ValueError(
                    "HM27 payload contains non-finite "
                    "non-missing beta-values."
                )

            if not np.logical_and(
                numeric_beta_values >= 0,
                numeric_beta_values <= 1,
            ).all():
                raise ValueError(
                    "HM27 payload contains beta-values "
                    "outside [0, 1]."
                )

            expected_missing_fraction = float(
                record.missing_beta_fraction
            )

            observed_missing_fraction = float(
                missing_mask.mean()
            )

            if not np.isclose(
                observed_missing_fraction,
                expected_missing_fraction,
                rtol=0,
                atol=1e-12,
            ):
                raise ValueError(
                    "HM27 payload missingness does not match "
                    f"the frozen QC metric for {current_file_id}."
                )

            expected_column = beta_values[
                hm27_eligible_probe_positions
            ]

            matrix_column_index = int(
                record.hm27_matrix_column_index
            )

            if matrix_column_index != sample_position - 1:
                raise ValueError(
                    "HM27 sample-column order is not complete."
                )

            temporary_matrix[
                :,
                matrix_column_index,
            ] = expected_column

            if not np.array_equal(
                temporary_matrix[
                    :,
                    matrix_column_index,
                ],
                expected_column,
                equal_nan=True,
            ):
                raise ValueError(
                    "Written HM27 column does not match "
                    f"the source payload {current_file_id}."
                )

            selected_missing_beta_count += int(
                np.isnan(expected_column).sum()
            )

            if (
                sample_position % 100 == 0
                or sample_position
                == len(hm27_sample_mapping)
            ):
                print(
                    f"Processed HM27 payloads: "
                    f"{sample_position:,}/"
                    f"{len(hm27_sample_mapping):,}"
                )

        temporary_matrix.flush()

        return {
            "processed_file_count": len(
                hm27_sample_mapping
            ),
            "selected_missing_beta_count": (
                selected_missing_beta_count
            ),
        }

    except Exception as error:
        if temporary_matrix is not None:
            temporary_matrix.flush()

        temporary_matrix = None
        gc.collect()

        TEMP_FINAL_HM27_MATRIX_PATH.unlink(
            missing_ok=True
        )

        raise RuntimeError(
            "HM27 temporary-matrix construction failed "
            f"at file {current_file_id}: "
            f"{current_file_path}"
        ) from error

    finally:
        if temporary_matrix is not None:
            temporary_matrix.flush()

        temporary_matrix = None
        gc.collect()


hm27_build_summary = build_hm27_temporary_matrix()

print()
print("HM27 temporary matrix constructed.")
print(
    f"Processed payloads: "
    f"{hm27_build_summary['processed_file_count']:,}"
)
print(
    f"Selected missing beta-values: "
    f"{hm27_build_summary['selected_missing_beta_count']:,}"
)
print(
    f"Temporary matrix: "
    f"{project_relative_path(TEMP_FINAL_HM27_MATRIX_PATH)}"
)

# ========================================================
# Validate and publish the final HM27 beta-value matrix
# ========================================================

if not TEMP_FINAL_HM27_MATRIX_PATH.is_file():
    raise FileNotFoundError(
        "Validated HM27 temporary matrix was not found: "
        f"{TEMP_FINAL_HM27_MATRIX_PATH}"
    )


temporary_hm27_matrix = np.load(
    TEMP_FINAL_HM27_MATRIX_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

flat_hm27_values = temporary_hm27_matrix.ravel(
    order="K"
)

nonmissing_hm27_values = flat_hm27_values[
    ~np.isnan(flat_hm27_values)
]

temporary_hm27_checks = {
    "temporary_matrix_exists": (
        TEMP_FINAL_HM27_MATRIX_PATH.is_file()
    ),
    "temporary_matrix_shape_matches_mapping": (
        temporary_hm27_matrix.shape
        == (
            len(hm27_probe_mapping),
            len(hm27_sample_mapping),
        )
    ),
    "temporary_matrix_dtype_is_float32": (
        temporary_hm27_matrix.dtype
        == HM27_MATRIX_DTYPE
    ),
    "temporary_matrix_is_fortran_contiguous": (
        temporary_hm27_matrix.flags.f_contiguous
    ),
    "temporary_values_are_finite": (
        np.isfinite(nonmissing_hm27_values).all()
    ),
    "temporary_beta_values_are_in_range": (
        np.logical_and(
            nonmissing_hm27_values >= 0,
            nonmissing_hm27_values <= 1,
        ).all()
    ),
    "temporary_missing_count_matches_written_columns": (
        int(np.isnan(flat_hm27_values).sum())
        == hm27_build_summary[
            "selected_missing_beta_count"
        ]
    ),
}


print("HM27 temporary-artifact checks:")

for check_name, check_passed in (
    temporary_hm27_checks.items()
):
    print(f"{check_name}: {check_passed}")

temporary_hm27_is_valid = all(
    temporary_hm27_checks.values()
)


del nonmissing_hm27_values
del flat_hm27_values
del temporary_hm27_matrix

gc.collect()


if not temporary_hm27_is_valid:
    TEMP_FINAL_HM27_MATRIX_PATH.unlink(
        missing_ok=True
    )

    raise ValueError(
        "HM27 temporary-artifact validation failed."
    )


for variable_name in [
    "temporary_hm27_matrix",
    "published_hm27_matrix",
    "flat_hm27_values",
]:
    globals().pop(variable_name, None)

gc.collect()


TEMP_FINAL_HM27_MATRIX_PATH.replace(
    FINAL_HM27_MATRIX_PATH
)


published_hm27_matrix = np.load(
    FINAL_HM27_MATRIX_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

published_hm27_checks = {
    "published_matrix_exists": (
        FINAL_HM27_MATRIX_PATH.is_file()
    ),
    "published_matrix_shape_matches_mapping": (
        published_hm27_matrix.shape
        == (
            len(hm27_probe_mapping),
            len(hm27_sample_mapping),
        )
    ),
    "published_matrix_dtype_is_float32": (
        published_hm27_matrix.dtype
        == HM27_MATRIX_DTYPE
    ),
    "published_matrix_is_fortran_contiguous": (
        published_hm27_matrix.flags.f_contiguous
    ),
}


print()
print("Published HM27-matrix checks:")

for check_name, check_passed in (
    published_hm27_checks.items()
):
    print(f"{check_name}: {check_passed}")


if not all(published_hm27_checks.values()):
    raise ValueError(
        "Published HM27 matrix validation failed."
    )


print()
print("Final HM27 methylation matrix published.")
print(
    f"Shape: {published_hm27_matrix.shape[0]:,} probes × "
    f"{published_hm27_matrix.shape[1]:,} samples"
)
print(
    "Matrix: "
    f"{project_relative_path(FINAL_HM27_MATRIX_PATH)}"
)


del published_hm27_matrix
gc.collect()

In [40]:
# =============================================================================
# Reload the completed HM27 matrix state from persisted artifacts
# =============================================================================

hm27_matrix_shape = (
    len(hm27_probe_mapping),
    len(hm27_sample_mapping),
)

hm27_reuse_checks = {
    "hm27_final_matrix_exists": (
        FINAL_HM27_MATRIX_PATH.is_file()
    ),
}

if not all(hm27_reuse_checks.values()):
    raise FileNotFoundError(
        "HM27 final matrix was not found. Run the one-time "
        "HM27 construction and publication cells before skipping them."
    )

published_hm27_matrix = np.load(
    FINAL_HM27_MATRIX_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

hm27_reuse_checks.update(
    {
        "hm27_shape_matches_mapping": (
            published_hm27_matrix.shape == hm27_matrix_shape
        ),
        "hm27_dtype_is_float32": (
            published_hm27_matrix.dtype == HM27_MATRIX_DTYPE
        ),
        "hm27_matrix_is_fortran_contiguous": (
            published_hm27_matrix.flags.f_contiguous
        ),
    }
)

print("Reusable HM27 artifact checks:")

for check_name, check_passed in hm27_reuse_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(hm27_reuse_checks.values()):
    raise ValueError(
        "Reusable HM27 artifact validation failed."
    )

print()
print("HM27 matrix state reloaded.")
print(
    f"Shape: {published_hm27_matrix.shape[0]:,} probes × "
    f"{published_hm27_matrix.shape[1]:,} samples"
)
print(
    "Matrix: "
    f"{project_relative_path(FINAL_HM27_MATRIX_PATH)}"
)

del published_hm27_matrix
gc.collect()

Reusable HM27 artifact checks:
hm27_final_matrix_exists: True
hm27_shape_matches_mapping: True
hm27_dtype_is_float32: True
hm27_matrix_is_fortran_contiguous: True

HM27 matrix state reloaded.
Shape: 24,303 probes × 1,620 samples
Matrix: data/interim/methylation/tcga_primary_tumor_methylation_hm27_final_case_level_beta_values.npy


11

In [41]:
# =============================================================================
# Configure and validate the final HM450 matrix mapping
# =============================================================================

HM450_PLATFORM = "Illumina Human Methylation 450"
HM450_REPRESENTATION = "hm450"
HM450_MATRIX_DTYPE = np.dtype("float32")

HM450_EXPECTED_SAMPLE_COUNT = 8_345
HM450_EXPECTED_PROBE_COUNT = 407_696


FINAL_HM450_MATRIX_PATH = (
    Paths.methylation
    / "tcga_primary_tumor_methylation_hm450_final_case_level_beta_values.npy"
)

TEMP_FINAL_HM450_MATRIX_PATH = (
    FINAL_HM450_MATRIX_PATH.with_name(
        FINAL_HM450_MATRIX_PATH.stem
        + ".tmp"
        + FINAL_HM450_MATRIX_PATH.suffix
    )
)


# -----------------------------------------------------------------------------
# Resolve HM450 sample and probe mappings
# -----------------------------------------------------------------------------

hm450_sample_mapping = (
    final_sample_mapping.loc[
        final_sample_mapping["methylation_platform"]
        .eq(HM450_PLATFORM)
    ]
    .sort_values(
        "hm450_matrix_column_index",
        kind="stable",
    )
    .reset_index(drop=True)
)

hm450_probe_mapping = (
    final_probe_mapping.loc[
        final_probe_mapping["representation"]
        .eq(HM450_REPRESENTATION)
    ]
    .sort_values(
        "matrix_row_index",
        kind="stable",
    )
    .reset_index(drop=True)
)


hm450_reference_probe_ids = platform_probe_ids[HM450_PLATFORM]

hm450_eligible_probe_positions = (
    platform_eligible_positions[HM450_PLATFORM]
)

expected_hm450_probe_ids = (
    hm450_reference_probe_ids
    .take(hm450_eligible_probe_positions)
)

observed_hm450_probe_ids = pd.Index(
    hm450_probe_mapping["probe_id"].astype("string"),
    name="probe_id",
)


hm450_sample_column_indices = (
    hm450_sample_mapping[
        "hm450_matrix_column_index"
    ]
    .astype("int64")
    .to_numpy()
)

hm450_probe_row_indices = (
    hm450_probe_mapping["matrix_row_index"]
    .astype("int64")
    .to_numpy()
)


# -----------------------------------------------------------------------------
# Validate mappings
# -----------------------------------------------------------------------------

hm450_mapping_checks = {
    "HM450_sample_mapping_is_not_empty": (
        not hm450_sample_mapping.empty
    ),
    "HM450_sample_count_matches_policy": (
        len(hm450_sample_mapping)
        == HM450_EXPECTED_SAMPLE_COUNT
    ),
    "HM450_sample_indices_are_complete": (
        np.array_equal(
            hm450_sample_column_indices,
            np.arange(
                len(hm450_sample_mapping),
                dtype=np.int64,
            ),
        )
    ),
    "HM450_methylation_file_ids_are_unique": (
        hm450_sample_mapping[
            "methylation_file_id"
        ]
        .astype("string")
        .is_unique
    ),
    "HM450_payload_paths_are_complete": (
        hm450_sample_mapping[
            "methylation_local_path"
        ]
        .notna()
        .all()
    ),
    "HM450_payloads_are_available": (
        hm450_sample_mapping[
            "methylation_payload_exists"
        ]
        .astype("boolean")
        .fillna(False)
        .all()
    ),
    "HM450_probe_mapping_is_not_empty": (
        not hm450_probe_mapping.empty
    ),
    "HM450_probe_count_matches_policy": (
        len(hm450_probe_mapping)
        == HM450_EXPECTED_PROBE_COUNT
    ),
    "HM450_probe_indices_are_complete": (
        np.array_equal(
            hm450_probe_row_indices,
            np.arange(
                len(hm450_probe_mapping),
                dtype=np.int64,
            ),
        )
    ),
    "HM450_source_probe_order_matches_mapping": (
        np.array_equal(
            hm450_probe_mapping[
                "source_platform_probe_order"
            ]
            .astype("int64")
            .to_numpy(),
            hm450_eligible_probe_positions,
        )
    ),
    "HM450_probe_ids_match_mapping": (
        observed_hm450_probe_ids.equals(
            expected_hm450_probe_ids
        )
    ),
    "HM450_all_probe_rows_are_eligible": (
        hm450_probe_mapping[
            "probe_qc_eligible"
        ]
        .astype("boolean")
        .fillna(False)
        .all()
    ),
}


print("HM450 mapping checks:")

for check_name, check_passed in hm450_mapping_checks.items():
    print(f"{check_name}: {check_passed}")


if not all(hm450_mapping_checks.values()):
    failed_checks = [
        check_name
        for check_name, check_passed
        in hm450_mapping_checks.items()
        if not check_passed
    ]

    raise ValueError(
        "HM450 mapping validation failed: "
        + ", ".join(failed_checks)
    )


hm450_matrix_shape = (
    len(hm450_probe_mapping),
    len(hm450_sample_mapping),
)

hm450_required_bytes = (
    int(np.prod(hm450_matrix_shape))
    * HM450_MATRIX_DTYPE.itemsize
)

hm450_free_bytes = shutil.disk_usage(
    FINAL_HM450_MATRIX_PATH.parent
).free

print()
print("HM450 mapping ready.")
print(
    f"Expected matrix shape: "
    f"{hm450_matrix_shape[0]:,} probes × "
    f"{hm450_matrix_shape[1]:,} samples"
)
print(
    f"Expected matrix size: "
    f"{hm450_required_bytes / (1024**3):,.2f} GiB"
)
print(
    f"Available disk space: "
    f"{hm450_free_bytes / (1024**3):,.2f} GiB"
)

if hm450_free_bytes < hm450_required_bytes:
    raise OSError(
        "Insufficient free disk space for the HM450 matrix."
    )

HM450 mapping checks:
HM450_sample_mapping_is_not_empty: True
HM450_sample_count_matches_policy: True
HM450_sample_indices_are_complete: True
HM450_methylation_file_ids_are_unique: True
HM450_payload_paths_are_complete: True
HM450_payloads_are_available: True
HM450_probe_mapping_is_not_empty: True
HM450_probe_count_matches_policy: True
HM450_probe_indices_are_complete: True
HM450_source_probe_order_matches_mapping: True
HM450_probe_ids_match_mapping: True
HM450_all_probe_rows_are_eligible: True

HM450 mapping ready.
Expected matrix shape: 407,696 probes × 8,345 samples
Expected matrix size: 12.67 GiB
Available disk space: 398.83 GiB


# =============================================================================
# Construct the temporary final HM450 beta-value matrix
# =============================================================================

for variable_name in [
    "temporary_hm450_matrix",
    "published_hm450_matrix",
    "block",
    "missing_mask",
    "nonmissing_hm450_values",
]:
    globals().pop(variable_name, None)

gc.collect()

TEMP_FINAL_HM450_MATRIX_PATH.unlink(
    missing_ok=True
)


def build_hm450_temporary_matrix():

    temporary_matrix = None
    current_file_id = None
    current_file_path = None

    selected_missing_beta_count = 0

    try:
        temporary_matrix = np.lib.format.open_memmap(
            TEMP_FINAL_HM450_MATRIX_PATH,
            mode="w+",
            dtype=HM450_MATRIX_DTYPE,
            shape=hm450_matrix_shape,
            fortran_order=True,
        )

        for sample_position, record in enumerate(
            hm450_sample_mapping.itertuples(index=False),
            start=1,
        ):

            current_file_id = str(
                record.methylation_file_id
            )

            current_file_path = (
                PROJECT_ROOT
                / str(record.methylation_local_path)
            )

            payload = read_methylation_payload(
                current_file_path
            )

            observed_probe_ids = pd.Index(
                payload["probe_id"],
                name="probe_id",
            )

            if not observed_probe_ids.equals(
                hm450_reference_probe_ids
            ):
                raise ValueError(
                    "HM450 payload probe identity or order "
                    "does not match the canonical reference."
                )

            beta_values = payload[
                "beta_value"
            ].to_numpy(
                dtype=HM450_MATRIX_DTYPE
            )

            missing_mask = np.isnan(beta_values)

            numeric_beta_values = beta_values[
                ~missing_mask
            ]

            if not np.isfinite(
                numeric_beta_values
            ).all():
                raise ValueError(
                    "HM450 payload contains non-finite "
                    "non-missing beta-values."
                )

            if not np.logical_and(
                numeric_beta_values >= 0,
                numeric_beta_values <= 1,
            ).all():
                raise ValueError(
                    "HM450 payload contains beta-values "
                    "outside [0, 1]."
                )

            expected_missing_fraction = float(
                record.missing_beta_fraction
            )

            observed_missing_fraction = float(
                missing_mask.mean()
            )

            if not np.isclose(
                observed_missing_fraction,
                expected_missing_fraction,
                rtol=0,
                atol=1e-12,
            ):
                raise ValueError(
                    "HM450 payload missingness does not match "
                    f"the frozen QC metric for {current_file_id}."
                )

            expected_column = beta_values[
                hm450_eligible_probe_positions
            ]

            matrix_column_index = int(
                record.hm450_matrix_column_index
            )

            if matrix_column_index != sample_position - 1:
                raise ValueError(
                    "HM450 sample-column order is not complete."
                )

            temporary_matrix[
                :,
                matrix_column_index,
            ] = expected_column

            if not np.array_equal(
                temporary_matrix[
                    :,
                    matrix_column_index,
                ],
                expected_column,
                equal_nan=True,
            ):
                raise ValueError(
                    "Written HM450 column does not match "
                    f"the source payload {current_file_id}."
                )

            selected_missing_beta_count += int(
                np.isnan(expected_column).sum()
            )

            if (
                sample_position % 100 == 0
                or sample_position
                == len(hm450_sample_mapping)
            ):
                print(
                    f"Processed HM450 payloads: "
                    f"{sample_position:,}/"
                    f"{len(hm450_sample_mapping):,}"
                )

        temporary_matrix.flush()

        return {
            "processed_file_count": len(
                hm450_sample_mapping
            ),
            "selected_missing_beta_count": (
                selected_missing_beta_count
            ),
        }

    except Exception as error:

        if temporary_matrix is not None:
            temporary_matrix.flush()

        temporary_matrix = None
        gc.collect()

        TEMP_FINAL_HM450_MATRIX_PATH.unlink(
            missing_ok=True
        )

        raise RuntimeError(
            "HM450 temporary-matrix construction failed "
            f"at file {current_file_id}: "
            f"{current_file_path}"
        ) from error

    finally:

        if temporary_matrix is not None:
            temporary_matrix.flush()

        temporary_matrix = None
        gc.collect()


hm450_build_summary = (
    build_hm450_temporary_matrix()
)

print()
print("HM450 temporary matrix constructed.")
print(
    f"Processed payloads: "
    f"{hm450_build_summary['processed_file_count']:,}"
)
print(
    f"Selected missing beta-values: "
    f"{hm450_build_summary['selected_missing_beta_count']:,}"
)
print(
    "Temporary matrix: "
    f"{project_relative_path(TEMP_FINAL_HM450_MATRIX_PATH)}"
)

# =============================================================================
# Persist the completed HM450 construction metadata
# =============================================================================

import json
from datetime import datetime, timezone


HM450_BUILD_METADATA_PATH = (
    FINAL_HM450_MATRIX_PATH.with_name(
        FINAL_HM450_MATRIX_PATH.stem
        + ".build_metadata.json"
    )
)

HM450_BUILD_METADATA_TEMP_PATH = (
    HM450_BUILD_METADATA_PATH.with_name(
        HM450_BUILD_METADATA_PATH.stem
        + ".tmp"
        + HM450_BUILD_METADATA_PATH.suffix
    )
)


if not TEMP_FINAL_HM450_MATRIX_PATH.is_file():
    raise FileNotFoundError(
        "The completed HM450 temporary matrix was not found."
    )

if "hm450_build_summary" not in globals():
    raise RuntimeError(
        "hm450_build_summary is not available. "
        "The construction cell may not have completed successfully."
    )


if (
    int(hm450_build_summary["processed_file_count"])
    != len(hm450_sample_mapping)
):
    raise ValueError(
        "Processed HM450 payload count does not match the mapping."
    )


hm450_build_metadata = {
    "artifact_type": (
        "final_hm450_case_level_beta_value_matrix"
    ),
    "status": "constructed",
    "recorded_at_utc": (
        datetime.now(timezone.utc).isoformat()
    ),
    "platform": HM450_PLATFORM,
    "representation": HM450_REPRESENTATION,
    "matrix_shape": [
        int(hm450_matrix_shape[0]),
        int(hm450_matrix_shape[1]),
    ],
    "matrix_dtype": HM450_MATRIX_DTYPE.name,
    "fortran_order": True,
    "processed_file_count": int(
        hm450_build_summary["processed_file_count"]
    ),
    "selected_missing_beta_count": int(
        hm450_build_summary[
            "selected_missing_beta_count"
        ]
    ),
    "selected_nonmissing_beta_count": int(
        np.prod(hm450_matrix_shape)
        - hm450_build_summary[
            "selected_missing_beta_count"
        ]
    ),
    "temporary_matrix_path": project_relative_path(
        TEMP_FINAL_HM450_MATRIX_PATH
    ),
    "final_matrix_path": project_relative_path(
        FINAL_HM450_MATRIX_PATH
    ),
    "beta_value_policy": (
        "raw beta-values; eligible probes only; "
        "NaN retained; no imputation, normalization, "
        "or batch correction"
    ),
}


HM450_BUILD_METADATA_TEMP_PATH.unlink(
    missing_ok=True
)

HM450_BUILD_METADATA_TEMP_PATH.write_text(
    json.dumps(
        hm450_build_metadata,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

HM450_BUILD_METADATA_TEMP_PATH.replace(
    HM450_BUILD_METADATA_PATH
)


print("HM450 construction metadata persisted.")
print(
    "Metadata: "
    f"{project_relative_path(HM450_BUILD_METADATA_PATH)}"
)

In [42]:
# =============================================================================
# Reload the completed HM450 matrix state from persisted artifacts
# =============================================================================

HM450_BUILD_METADATA_PATH = (
    FINAL_HM450_MATRIX_PATH.with_name(
        FINAL_HM450_MATRIX_PATH.stem
        + ".build_metadata.json"
    )
)


hm450_reuse_checks = {
    "hm450_build_metadata_exists": (
        HM450_BUILD_METADATA_PATH.is_file()
    ),
    "hm450_final_matrix_exists": (
        FINAL_HM450_MATRIX_PATH.is_file()
    ),
}

if not all(hm450_reuse_checks.values()):
    for check_name, check_passed in hm450_reuse_checks.items():
        print(f"{check_name}: {check_passed}")

    raise FileNotFoundError(
        "HM450 cannot be reused yet. Run the one-time "
        "construction, metadata, and publication cells once before "
        "skipping them in Run All."
    )


with HM450_BUILD_METADATA_PATH.open(
    "r",
    encoding="utf-8",
) as metadata_file:
    hm450_build_metadata = json.load(metadata_file)


hm450_build_summary = {
    "processed_file_count": int(
        hm450_build_metadata["processed_file_count"]
    ),
    "selected_missing_beta_count": int(
        hm450_build_metadata["selected_missing_beta_count"]
    ),
}


published_hm450_matrix = np.load(
    FINAL_HM450_MATRIX_PATH,
    mmap_mode="r",
    allow_pickle=False,
)


hm450_reuse_checks.update(
    {
        "hm450_processed_count_matches_mapping": (
            hm450_build_summary["processed_file_count"]
            == len(hm450_sample_mapping)
        ),
        "hm450_shape_matches_mapping": (
            published_hm450_matrix.shape
            == hm450_matrix_shape
        ),
        "hm450_shape_matches_metadata": (
            list(published_hm450_matrix.shape)
            == hm450_build_metadata["matrix_shape"]
        ),
        "hm450_dtype_is_float32": (
            published_hm450_matrix.dtype
            == HM450_MATRIX_DTYPE
        ),
        "hm450_matrix_is_fortran_contiguous": (
            published_hm450_matrix.flags.f_contiguous
        ),
    }
)


print("Reusable HM450 artifact checks:")

for check_name, check_passed in hm450_reuse_checks.items():
    print(f"{check_name}: {check_passed}")


if not all(hm450_reuse_checks.values()):
    raise ValueError(
        "Reusable HM450 artifact validation failed."
    )


print()
print("HM450 matrix state reloaded.")
print(
    f"Shape: {published_hm450_matrix.shape[0]:,} probes × "
    f"{published_hm450_matrix.shape[1]:,} samples"
)
print(
    "Matrix: "
    f"{project_relative_path(FINAL_HM450_MATRIX_PATH)}"
)
print(
    "Metadata: "
    f"{project_relative_path(HM450_BUILD_METADATA_PATH)}"
)


del published_hm450_matrix
gc.collect()

Reusable HM450 artifact checks:
hm450_build_metadata_exists: True
hm450_final_matrix_exists: True
hm450_processed_count_matches_mapping: True
hm450_shape_matches_mapping: True
hm450_shape_matches_metadata: True
hm450_dtype_is_float32: True
hm450_matrix_is_fortran_contiguous: True

HM450 matrix state reloaded.
Shape: 407,696 probes × 8,345 samples
Matrix: data/interim/methylation/tcga_primary_tumor_methylation_hm450_final_case_level_beta_values.npy
Metadata: data/interim/methylation/tcga_primary_tumor_methylation_hm450_final_case_level_beta_values.build_metadata.json


11

In [43]:
# =============================================================================
# Configure and validate the shared HM27/HM450 matrix mapping
# =============================================================================

SHARED_METHYLATION_REPRESENTATION = "shared_hm27_hm450"
SHARED_METHYLATION_MATRIX_DTYPE = np.dtype("float32")

SHARED_MATRIX_CHUNK_COLUMNS = 128

TEMP_FINAL_SHARED_METHYLATION_MATRIX_PATH = (
    FINAL_SHARED_METHYLATION_MATRIX_PATH.with_name(
        FINAL_SHARED_METHYLATION_MATRIX_PATH.stem
        + ".tmp"
        + FINAL_SHARED_METHYLATION_MATRIX_PATH.suffix
    )
)

SHARED_BUILD_METADATA_PATH = (
    FINAL_SHARED_METHYLATION_MATRIX_PATH.with_name(
        FINAL_SHARED_METHYLATION_MATRIX_PATH.stem
        + ".build_metadata.json"
    )
)


shared_probe_mapping = (
    final_probe_mapping.loc[
        final_probe_mapping["representation"]
        .eq(SHARED_METHYLATION_REPRESENTATION)
    ]
    .sort_values("matrix_row_index", kind="stable")
    .reset_index(drop=True)
)

shared_sample_mapping = (
    final_sample_mapping
    .sort_values("shared_matrix_column_index", kind="stable")
    .reset_index(drop=True)
)

shared_probe_ids_for_matrix = pd.Index(
    shared_probe_mapping["probe_id"].astype("string"),
    name="probe_id",
)

hm27_shared_source_probe_positions = (
    platform_probe_ids[HM27_PLATFORM]
    .get_indexer(shared_probe_ids_for_matrix)
)

hm450_shared_source_probe_positions = (
    platform_probe_ids[HM450_PLATFORM]
    .get_indexer(shared_probe_ids_for_matrix)
)


hm27_row_by_source_probe_order = pd.Series(
    hm27_probe_mapping["matrix_row_index"]
    .astype("int64")
    .to_numpy(),
    index=hm27_probe_mapping["source_platform_probe_order"]
    .astype("int64")
    .to_numpy(),
)

hm450_row_by_source_probe_order = pd.Series(
    hm450_probe_mapping["matrix_row_index"]
    .astype("int64")
    .to_numpy(),
    index=hm450_probe_mapping["source_platform_probe_order"]
    .astype("int64")
    .to_numpy(),
)


shared_hm27_matrix_row_indices = (
    hm27_row_by_source_probe_order
    .reindex(hm27_shared_source_probe_positions)
)

shared_hm450_matrix_row_indices = (
    hm450_row_by_source_probe_order
    .reindex(hm450_shared_source_probe_positions)
)


hm27_shared_sample_mask = (
    shared_sample_mapping["methylation_platform"]
    .eq(HM27_PLATFORM)
)

hm450_shared_sample_mask = (
    shared_sample_mapping["methylation_platform"]
    .eq(HM450_PLATFORM)
)


shared_hm27_target_column_indices = (
    shared_sample_mapping.loc[
        hm27_shared_sample_mask,
        "shared_matrix_column_index",
    ]
    .astype("int64")
    .to_numpy()
)

shared_hm450_target_column_indices = (
    shared_sample_mapping.loc[
        hm450_shared_sample_mask,
        "shared_matrix_column_index",
    ]
    .astype("int64")
    .to_numpy()
)

shared_hm27_source_column_indices = (
    shared_sample_mapping.loc[
        hm27_shared_sample_mask,
        "hm27_matrix_column_index",
    ]
    .astype("int64")
    .to_numpy()
)

shared_hm450_source_column_indices = (
    shared_sample_mapping.loc[
        hm450_shared_sample_mask,
        "hm450_matrix_column_index",
    ]
    .astype("int64")
    .to_numpy()
)


shared_matrix_shape = (
    len(shared_probe_mapping),
    len(shared_sample_mapping),
)

shared_required_bytes = (
    int(np.prod(shared_matrix_shape))
    * SHARED_METHYLATION_MATRIX_DTYPE.itemsize
)

shared_free_bytes = shutil.disk_usage(
    FINAL_SHARED_METHYLATION_MATRIX_PATH.parent
).free


shared_mapping_checks = {
    "shared_probe_mapping_is_not_empty": (
        not shared_probe_mapping.empty
    ),
    "shared_probe_indices_are_complete": (
        shared_probe_mapping["matrix_row_index"].tolist()
        == list(range(len(shared_probe_mapping)))
    ),
    "shared_sample_indices_are_complete": (
        shared_sample_mapping["shared_matrix_column_index"].tolist()
        == list(range(len(shared_sample_mapping)))
    ),
    "shared_HM27_probes_exist_in_platform_matrix": (
        not shared_hm27_matrix_row_indices.isna().any()
    ),
    "shared_HM450_probes_exist_in_platform_matrix": (
        not shared_hm450_matrix_row_indices.isna().any()
    ),
    "shared_HM27_columns_cover_platform_samples": (
        np.array_equal(
            np.sort(shared_hm27_source_column_indices),
            np.arange(len(hm27_sample_mapping)),
        )
    ),
    "shared_HM450_columns_cover_platform_samples": (
        np.array_equal(
            np.sort(shared_hm450_source_column_indices),
            np.arange(len(hm450_sample_mapping)),
        )
    ),
    "HM27_final_matrix_exists": FINAL_HM27_MATRIX_PATH.is_file(),
    "HM450_final_matrix_exists": FINAL_HM450_MATRIX_PATH.is_file(),
}

print("Shared HM27/HM450 mapping checks:")

for check_name, check_passed in shared_mapping_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(shared_mapping_checks.values()):
    failed_checks = [
        check_name
        for check_name, check_passed in shared_mapping_checks.items()
        if not check_passed
    ]

    raise ValueError(
        "Shared methylation mapping validation failed: "
        + ", ".join(failed_checks)
    )


shared_hm27_matrix_row_indices = (
    shared_hm27_matrix_row_indices.astype("int64").to_numpy()
)

shared_hm450_matrix_row_indices = (
    shared_hm450_matrix_row_indices.astype("int64").to_numpy()
)


print()
print("Shared HM27/HM450 mapping ready.")
print(
    f"Expected matrix shape: "
    f"{shared_matrix_shape[0]:,} probes × "
    f"{shared_matrix_shape[1]:,} cases"
)
print(
    f"Expected matrix size: "
    f"{shared_required_bytes / (1024**3):,.2f} GiB"
)
print(
    f"Available disk space: "
    f"{shared_free_bytes / (1024**3):,.2f} GiB"
)

if shared_free_bytes < shared_required_bytes:
    raise OSError(
        "Insufficient free disk space for the shared methylation matrix."
    )

Shared HM27/HM450 mapping checks:
shared_probe_mapping_is_not_empty: True
shared_probe_indices_are_complete: True
shared_sample_indices_are_complete: True
shared_HM27_probes_exist_in_platform_matrix: True
shared_HM450_probes_exist_in_platform_matrix: True
shared_HM27_columns_cover_platform_samples: True
shared_HM450_columns_cover_platform_samples: True
HM27_final_matrix_exists: True
HM450_final_matrix_exists: True

Shared HM27/HM450 mapping ready.
Expected matrix shape: 23,356 probes × 9,965 cases
Expected matrix size: 0.87 GiB
Available disk space: 398.83 GiB


In [44]:
# =============================================================================
# Construct or reload the shared HM27/HM450 beta-value matrix state
# =============================================================================

import gc
import json
from datetime import datetime, timezone


SHARED_BUILD_METADATA_TEMP_PATH = (
    SHARED_BUILD_METADATA_PATH.with_name(
        SHARED_BUILD_METADATA_PATH.stem
        + ".tmp"
        + SHARED_BUILD_METADATA_PATH.suffix
    )
)


if FINAL_SHARED_METHYLATION_MATRIX_PATH.is_file():
    print(
        "The final shared HM27/HM450 methylation matrix already exists. "
        "Skipping temporary construction and reloading the persisted artifact."
    )

    if not SHARED_BUILD_METADATA_PATH.is_file():
        raise FileNotFoundError(
            "The shared HM27/HM450 methylation metadata was not found."
        )

    with SHARED_BUILD_METADATA_PATH.open("r", encoding="utf-8") as metadata_file:
        shared_build_metadata = json.load(metadata_file)

    shared_matrix_shape = tuple(
        int(value) for value in shared_build_metadata["matrix_shape"]
    )

    published_shared_matrix = np.load(
        FINAL_SHARED_METHYLATION_MATRIX_PATH,
        mmap_mode="r",
        allow_pickle=False,
    )

    published_structural_checks = {
        "final_matrix_exists": FINAL_SHARED_METHYLATION_MATRIX_PATH.is_file(),
        "published_shape_matches_metadata": (
            published_shared_matrix.shape == shared_matrix_shape
        ),
        "published_dtype_is_float32": (
            published_shared_matrix.dtype == SHARED_METHYLATION_MATRIX_DTYPE
        ),
        "published_matrix_is_fortran_contiguous": (
            published_shared_matrix.flags.f_contiguous
        ),
    }

    print("Published shared matrix structural checks:")

    for check_name, check_passed in published_structural_checks.items():
        print(f"{check_name}: {check_passed}")

    if not all(published_structural_checks.values()):
        raise ValueError(
            "Published shared methylation matrix structural validation failed."
        )

    if shared_build_metadata.get("status") != "published":
        if shared_build_metadata.get("status") != "constructed":
            raise ValueError(
                "The final shared methylation matrix exists, but metadata status "
                f"is unexpected: {shared_build_metadata.get('status')!r}."
            )

        shared_build_metadata = dict(shared_build_metadata)
        shared_build_metadata["status"] = "published"
        shared_build_metadata["published_at_utc"] = datetime.now(
            timezone.utc
        ).isoformat()
        shared_build_metadata[
            "publication_recovered_from_existing_final_matrix"
        ] = True

        if (
            "observed_missing_beta_count" not in shared_build_metadata
            and "selected_missing_beta_count" in shared_build_metadata
        ):
            shared_build_metadata["observed_missing_beta_count"] = int(
                shared_build_metadata["selected_missing_beta_count"]
            )

        if (
            "observed_nonmissing_beta_count" not in shared_build_metadata
            and "selected_nonmissing_beta_count" in shared_build_metadata
        ):
            shared_build_metadata["observed_nonmissing_beta_count"] = int(
                shared_build_metadata["selected_nonmissing_beta_count"]
            )

        SHARED_BUILD_METADATA_TEMP_PATH.write_text(
            json.dumps(
                shared_build_metadata,
                indent=2,
                sort_keys=True,
            )
            + "\n",
            encoding="utf-8",
        )

        SHARED_BUILD_METADATA_TEMP_PATH.replace(SHARED_BUILD_METADATA_PATH)

        print()
        print(
            "Shared HM27/HM450 metadata status was repaired from "
            "'constructed' to 'published'."
        )

    published_checks = {
        "final_matrix_exists": FINAL_SHARED_METHYLATION_MATRIX_PATH.is_file(),
        "metadata_status_is_published": (
            shared_build_metadata.get("status") == "published"
        ),
        "published_shape_matches_metadata": (
            published_shared_matrix.shape == shared_matrix_shape
        ),
        "published_dtype_is_float32": (
            published_shared_matrix.dtype == SHARED_METHYLATION_MATRIX_DTYPE
        ),
        "published_matrix_is_fortran_contiguous": (
            published_shared_matrix.flags.f_contiguous
        ),
    }

    print()
    print("Published shared matrix checks:")

    for check_name, check_passed in published_checks.items():
        print(f"{check_name}: {check_passed}")

    if not all(published_checks.values()):
        raise ValueError(
            "Published shared methylation matrix validation failed."
        )

    print()
    print("Shared HM27/HM450 matrix state reloaded.")
    print(
        f"Shape: {shared_matrix_shape[0]:,} probes x "
        f"{shared_matrix_shape[1]:,} cases"
    )
    print(
        "Matrix: "
        f"{project_relative_path(FINAL_SHARED_METHYLATION_MATRIX_PATH)}"
    )
    print(
        "Metadata: "
        f"{project_relative_path(SHARED_BUILD_METADATA_PATH)}"
    )

    del published_shared_matrix
    gc.collect()

else:
    print(
        "The final shared HM27/HM450 methylation matrix does not exist yet. "
        "Proceeding with temporary construction."
    )

    combined_shared_target_columns = np.concatenate(
        [
            shared_hm27_target_column_indices,
            shared_hm450_target_column_indices,
        ]
    )

    if not np.array_equal(
        np.sort(combined_shared_target_columns),
        np.arange(len(shared_sample_mapping)),
    ):
        raise ValueError(
            "Shared target columns do not cover the final sample mapping exactly once."
        )

    TEMP_FINAL_SHARED_METHYLATION_MATRIX_PATH.unlink(missing_ok=True)
    SHARED_BUILD_METADATA_TEMP_PATH.unlink(missing_ok=True)

    def copy_platform_to_shared_matrix(
        *,
        platform_name,
        source_matrix,
        source_row_indices,
        source_column_indices,
        target_column_indices,
        temporary_matrix,
    ):
        processed_column_count = 0
        selected_missing_beta_count = 0

        for start in range(
            0,
            len(source_column_indices),
            SHARED_MATRIX_CHUNK_COLUMNS,
        ):
            stop = min(
                start + SHARED_MATRIX_CHUNK_COLUMNS,
                len(source_column_indices),
            )

            source_columns = source_column_indices[start:stop]
            target_columns = target_column_indices[start:stop]

            source_block = np.asarray(
                source_matrix[
                    np.ix_(
                        source_row_indices,
                        source_columns,
                    )
                ],
                dtype=SHARED_METHYLATION_MATRIX_DTYPE,
            )

            temporary_matrix[:, target_columns] = source_block

            if not np.array_equal(
                temporary_matrix[:, target_columns],
                source_block,
                equal_nan=True,
            ):
                raise ValueError(
                    f"Written shared {platform_name} block does not match "
                    "the source matrix."
                )

            selected_missing_beta_count += int(np.isnan(source_block).sum())
            processed_column_count += len(source_columns)

            print(
                f"Processed shared {platform_name} columns: "
                f"{processed_column_count:,}/"
                f"{len(source_column_indices):,}"
            )

            del source_block

        return {
            "processed_column_count": int(processed_column_count),
            "selected_missing_beta_count": int(selected_missing_beta_count),
        }

    temporary_shared_matrix = None
    hm27_final_matrix = None
    hm450_final_matrix = None

    try:
        hm27_final_matrix = np.load(
            FINAL_HM27_MATRIX_PATH,
            mmap_mode="r",
            allow_pickle=False,
        )

        hm450_final_matrix = np.load(
            FINAL_HM450_MATRIX_PATH,
            mmap_mode="r",
            allow_pickle=False,
        )

        temporary_shared_matrix = np.lib.format.open_memmap(
            TEMP_FINAL_SHARED_METHYLATION_MATRIX_PATH,
            mode="w+",
            dtype=SHARED_METHYLATION_MATRIX_DTYPE,
            shape=shared_matrix_shape,
            fortran_order=True,
        )

        hm27_shared_build_summary = copy_platform_to_shared_matrix(
            platform_name=HM27_PLATFORM,
            source_matrix=hm27_final_matrix,
            source_row_indices=shared_hm27_matrix_row_indices,
            source_column_indices=shared_hm27_source_column_indices,
            target_column_indices=shared_hm27_target_column_indices,
            temporary_matrix=temporary_shared_matrix,
        )

        hm450_shared_build_summary = copy_platform_to_shared_matrix(
            platform_name=HM450_PLATFORM,
            source_matrix=hm450_final_matrix,
            source_row_indices=shared_hm450_matrix_row_indices,
            source_column_indices=shared_hm450_source_column_indices,
            target_column_indices=shared_hm450_target_column_indices,
            temporary_matrix=temporary_shared_matrix,
        )

        temporary_shared_matrix.flush()

        shared_build_summary = {
            "processed_column_count": int(
                hm27_shared_build_summary["processed_column_count"]
                + hm450_shared_build_summary["processed_column_count"]
            ),
            "selected_missing_beta_count": int(
                hm27_shared_build_summary["selected_missing_beta_count"]
                + hm450_shared_build_summary["selected_missing_beta_count"]
            ),
            "hm27_processed_column_count": int(
                hm27_shared_build_summary["processed_column_count"]
            ),
            "hm27_selected_missing_beta_count": int(
                hm27_shared_build_summary["selected_missing_beta_count"]
            ),
            "hm450_processed_column_count": int(
                hm450_shared_build_summary["processed_column_count"]
            ),
            "hm450_selected_missing_beta_count": int(
                hm450_shared_build_summary["selected_missing_beta_count"]
            ),
        }

        shared_build_metadata = {
            "artifact_type": "final_shared_hm27_hm450_case_level_beta_value_matrix",
            "status": "constructed",
            "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
            "representation": SHARED_METHYLATION_REPRESENTATION,
            "matrix_shape": [
                int(shared_matrix_shape[0]),
                int(shared_matrix_shape[1]),
            ],
            "matrix_dtype": SHARED_METHYLATION_MATRIX_DTYPE.name,
            "fortran_order": True,
            "processed_column_count": int(
                shared_build_summary["processed_column_count"]
            ),
            "selected_missing_beta_count": int(
                shared_build_summary["selected_missing_beta_count"]
            ),
            "selected_nonmissing_beta_count": int(
                np.prod(shared_matrix_shape)
                - shared_build_summary["selected_missing_beta_count"]
            ),
            "hm27_processed_column_count": int(
                shared_build_summary["hm27_processed_column_count"]
            ),
            "hm27_selected_missing_beta_count": int(
                shared_build_summary["hm27_selected_missing_beta_count"]
            ),
            "hm450_processed_column_count": int(
                shared_build_summary["hm450_processed_column_count"]
            ),
            "hm450_selected_missing_beta_count": int(
                shared_build_summary["hm450_selected_missing_beta_count"]
            ),
            "source_hm27_matrix_path": project_relative_path(FINAL_HM27_MATRIX_PATH),
            "source_hm450_matrix_path": project_relative_path(FINAL_HM450_MATRIX_PATH),
            "temporary_matrix_path": project_relative_path(
                TEMP_FINAL_SHARED_METHYLATION_MATRIX_PATH
            ),
            "final_matrix_path": project_relative_path(
                FINAL_SHARED_METHYLATION_MATRIX_PATH
            ),
            "beta_value_policy": (
                "raw beta-values; shared HM27/HM450 probes only; "
                "NaN retained; no imputation, normalization, or batch correction"
            ),
        }

        SHARED_BUILD_METADATA_TEMP_PATH.write_text(
            json.dumps(
                shared_build_metadata,
                indent=2,
                sort_keys=True,
            )
            + "\n",
            encoding="utf-8",
        )

        SHARED_BUILD_METADATA_TEMP_PATH.replace(SHARED_BUILD_METADATA_PATH)

    except Exception as error:
        if temporary_shared_matrix is not None:
            temporary_shared_matrix.flush()

        temporary_shared_matrix = None
        hm27_final_matrix = None
        hm450_final_matrix = None
        gc.collect()

        TEMP_FINAL_SHARED_METHYLATION_MATRIX_PATH.unlink(missing_ok=True)
        SHARED_BUILD_METADATA_TEMP_PATH.unlink(missing_ok=True)

        raise RuntimeError(
            "Shared HM27/HM450 temporary-matrix construction failed."
        ) from error

    finally:
        if temporary_shared_matrix is not None:
            temporary_shared_matrix.flush()

        temporary_shared_matrix = None
        hm27_final_matrix = None
        hm450_final_matrix = None
        gc.collect()

    print()
    print("Shared HM27/HM450 temporary matrix constructed.")
    print(
        f"Processed columns: "
        f"{shared_build_summary['processed_column_count']:,}"
    )
    print(
        f"Selected missing beta-values: "
        f"{shared_build_summary['selected_missing_beta_count']:,}"
    )
    print(
        "Temporary matrix: "
        f"{project_relative_path(TEMP_FINAL_SHARED_METHYLATION_MATRIX_PATH)}"
    )
    print(
        "Metadata: "
        f"{project_relative_path(SHARED_BUILD_METADATA_PATH)}"
    )

The final shared HM27/HM450 methylation matrix already exists. Skipping temporary construction and reloading the persisted artifact.
Published shared matrix structural checks:
final_matrix_exists: True
published_shape_matches_metadata: True
published_dtype_is_float32: True
published_matrix_is_fortran_contiguous: True

Published shared matrix checks:
final_matrix_exists: True
metadata_status_is_published: True
published_shape_matches_metadata: True
published_dtype_is_float32: True
published_matrix_is_fortran_contiguous: True

Shared HM27/HM450 matrix state reloaded.
Shape: 23,356 probes x 9,965 cases
Matrix: data/interim/methylation/tcga_primary_tumor_methylation_shared_hm27_hm450_final_case_level_beta_values.npy
Metadata: data/interim/methylation/tcga_primary_tumor_methylation_shared_hm27_hm450_final_case_level_beta_values.build_metadata.json


In [45]:
# =============================================================================
# Reload the completed shared HM27/HM450 matrix state from persisted artifacts
# =============================================================================

if not FINAL_SHARED_METHYLATION_MATRIX_PATH.is_file():
    raise FileNotFoundError(
        "The final shared HM27/HM450 methylation matrix was not found."
    )

if not SHARED_BUILD_METADATA_PATH.is_file():
    raise FileNotFoundError(
        "The shared HM27/HM450 methylation build metadata was not found."
    )

with SHARED_BUILD_METADATA_PATH.open("r", encoding="utf-8") as metadata_file:
    shared_build_metadata = json.load(metadata_file)

shared_matrix_shape = tuple(
    int(value) for value in shared_build_metadata["matrix_shape"]
)

published_shared_matrix = np.load(
    FINAL_SHARED_METHYLATION_MATRIX_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

shared_reload_checks = {
    "final_matrix_exists": FINAL_SHARED_METHYLATION_MATRIX_PATH.is_file(),
    "metadata_status_is_published": (
        shared_build_metadata.get("status") == "published"
    ),
    "published_shape_matches_metadata": (
        published_shared_matrix.shape == shared_matrix_shape
    ),
    "published_dtype_is_float32": (
        published_shared_matrix.dtype == SHARED_METHYLATION_MATRIX_DTYPE
    ),
    "published_matrix_is_fortran_contiguous": (
        published_shared_matrix.flags.f_contiguous
    ),
}

print("Reloaded shared HM27/HM450 matrix checks:")

for check_name, check_passed in shared_reload_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(shared_reload_checks.values()):
    raise ValueError(
        "Reloaded shared HM27/HM450 methylation matrix validation failed."
    )

print()
print("Shared HM27/HM450 matrix state reloaded.")
print(
    f"Shape: {shared_matrix_shape[0]:,} probes x "
    f"{shared_matrix_shape[1]:,} cases"
)
print(
    "Matrix: "
    f"{project_relative_path(FINAL_SHARED_METHYLATION_MATRIX_PATH)}"
)
print(
    "Metadata: "
    f"{project_relative_path(SHARED_BUILD_METADATA_PATH)}"
)

del published_shared_matrix
gc.collect()

Reloaded shared HM27/HM450 matrix checks:
final_matrix_exists: True
metadata_status_is_published: True
published_shape_matches_metadata: True
published_dtype_is_float32: True
published_matrix_is_fortran_contiguous: True

Shared HM27/HM450 matrix state reloaded.
Shape: 23,356 probes x 9,965 cases
Matrix: data/interim/methylation/tcga_primary_tumor_methylation_shared_hm27_hm450_final_case_level_beta_values.npy
Metadata: data/interim/methylation/tcga_primary_tumor_methylation_shared_hm27_hm450_final_case_level_beta_values.build_metadata.json


11

In [46]:
# =============================================================================
# Validate or reload the final shared HM27/HM450 beta-value matrix
# =============================================================================

SHARED_BUILD_METADATA_TEMP_PATH = (
    SHARED_BUILD_METADATA_PATH.with_name(
        SHARED_BUILD_METADATA_PATH.stem
        + ".tmp"
        + SHARED_BUILD_METADATA_PATH.suffix
    )
)


def scan_shared_matrix_columns(
    *,
    matrix,
    matrix_shape,
    progress_label,
):
    observed_missing_beta_count = 0
    observed_out_of_range_beta_count = 0
    observed_beta_min = np.inf
    observed_beta_max = -np.inf

    for start in range(
        0,
        matrix_shape[1],
        SHARED_MATRIX_CHUNK_COLUMNS,
    ):
        stop = min(
            start + SHARED_MATRIX_CHUNK_COLUMNS,
            matrix_shape[1],
        )

        block = np.asarray(matrix[:, start:stop])
        missing_mask = np.isnan(block)

        observed_missing_beta_count += int(missing_mask.sum())

        observed_values_mask = ~missing_mask

        if observed_values_mask.any():
            observed_values = block[observed_values_mask]

            observed_beta_min = min(
                observed_beta_min,
                float(observed_values.min()),
            )
            observed_beta_max = max(
                observed_beta_max,
                float(observed_values.max()),
            )

            observed_out_of_range_beta_count += int(
                (
                    (observed_values < 0.0)
                    | (observed_values > 1.0)
                ).sum()
            )

            del observed_values

        del block

        print(
            f"{progress_label}: "
            f"{stop:,}/{matrix_shape[1]:,}"
        )

    return {
        "observed_missing_beta_count": int(observed_missing_beta_count),
        "observed_out_of_range_beta_count": int(
            observed_out_of_range_beta_count
        ),
        "observed_beta_min": float(observed_beta_min),
        "observed_beta_max": float(observed_beta_max),
    }


def validate_representative_source_columns(
    *,
    platform_name,
    source_matrix_path,
    source_row_indices,
    source_column_indices,
    target_column_indices,
    target_matrix,
):
    source_matrix = np.load(
        source_matrix_path,
        mmap_mode="r",
        allow_pickle=False,
    )

    representative_positions = np.unique(
        np.array(
            [
                0,
                len(source_column_indices) // 2,
                len(source_column_indices) - 1,
            ],
            dtype="int64",
        )
    )

    for position in representative_positions:
        source_column_index = int(source_column_indices[position])
        target_column_index = int(target_column_indices[position])

        expected_column = source_matrix[
            source_row_indices,
            source_column_index,
        ]

        observed_column = target_matrix[
            :,
            target_column_index,
        ]

        if not np.array_equal(
            observed_column,
            expected_column,
            equal_nan=True,
        ):
            raise ValueError(
                f"Representative shared {platform_name} column "
                "does not match the source matrix."
            )

    del source_matrix
    gc.collect()


if FINAL_SHARED_METHYLATION_MATRIX_PATH.is_file():
    print(
        "The final shared HM27/HM450 methylation matrix already exists. "
        "Reloading the published artifact."
    )

    if not SHARED_BUILD_METADATA_PATH.is_file():
        raise FileNotFoundError(
            "The shared methylation build metadata was not found."
        )

    with SHARED_BUILD_METADATA_PATH.open(
        "r",
        encoding="utf-8",
    ) as metadata_file:
        shared_build_metadata = json.load(metadata_file)

    shared_matrix_shape = tuple(
        int(value)
        for value in shared_build_metadata["matrix_shape"]
    )

    published_shared_matrix = np.load(
        FINAL_SHARED_METHYLATION_MATRIX_PATH,
        mmap_mode="r",
        allow_pickle=False,
    )

    published_checks = {
        "final_matrix_exists": FINAL_SHARED_METHYLATION_MATRIX_PATH.is_file(),
        "published_shape_matches_metadata": (
            published_shared_matrix.shape == shared_matrix_shape
        ),
        "published_dtype_is_float32": (
            published_shared_matrix.dtype == SHARED_METHYLATION_MATRIX_DTYPE
        ),
        "published_matrix_is_fortran_contiguous": (
            published_shared_matrix.flags.f_contiguous
        ),
    }

    print("Published shared matrix checks:")

    for check_name, check_passed in published_checks.items():
        print(f"{check_name}: {check_passed}")

    if not all(published_checks.values()):
        raise ValueError(
            "Published shared methylation matrix validation failed."
        )

    validate_representative_source_columns(
        platform_name=HM27_PLATFORM,
        source_matrix_path=FINAL_HM27_MATRIX_PATH,
        source_row_indices=shared_hm27_matrix_row_indices,
        source_column_indices=shared_hm27_source_column_indices,
        target_column_indices=shared_hm27_target_column_indices,
        target_matrix=published_shared_matrix,
    )

    validate_representative_source_columns(
        platform_name=HM450_PLATFORM,
        source_matrix_path=FINAL_HM450_MATRIX_PATH,
        source_row_indices=shared_hm450_matrix_row_indices,
        source_column_indices=shared_hm450_source_column_indices,
        target_column_indices=shared_hm450_target_column_indices,
        target_matrix=published_shared_matrix,
    )

    shared_published_metadata = dict(shared_build_metadata)

    if shared_published_metadata.get("status") != "published":
        shared_published_metadata["status"] = "published"
        shared_published_metadata["published_at_utc"] = datetime.now(
            timezone.utc
        ).isoformat()
        shared_published_metadata[
            "publication_recovered_from_existing_final_matrix"
        ] = True

        SHARED_BUILD_METADATA_TEMP_PATH.write_text(
            json.dumps(
                shared_published_metadata,
                indent=2,
                sort_keys=True,
            )
            + "\n",
            encoding="utf-8",
        )

        SHARED_BUILD_METADATA_TEMP_PATH.replace(
            SHARED_BUILD_METADATA_PATH
        )

        shared_build_metadata = dict(shared_published_metadata)

        print()
        print(
            "Shared metadata status was repaired to 'published' "
            "because the final matrix already exists."
        )

    print()
    print("Shared HM27/HM450 matrix state reloaded.")
    print(
        f"Shape: {shared_matrix_shape[0]:,} probes x "
        f"{shared_matrix_shape[1]:,} cases"
    )
    print(
        "Matrix: "
        f"{project_relative_path(FINAL_SHARED_METHYLATION_MATRIX_PATH)}"
    )
    print(
        "Metadata: "
        f"{project_relative_path(SHARED_BUILD_METADATA_PATH)}"
    )

    del published_shared_matrix
    gc.collect()

else:
    if not TEMP_FINAL_SHARED_METHYLATION_MATRIX_PATH.is_file():
        raise FileNotFoundError(
            "The temporary shared methylation matrix was not found."
        )

    if not SHARED_BUILD_METADATA_PATH.is_file():
        raise FileNotFoundError(
            "The shared methylation build metadata was not found."
        )

    with SHARED_BUILD_METADATA_PATH.open(
        "r",
        encoding="utf-8",
    ) as metadata_file:
        shared_build_metadata = json.load(metadata_file)

    temporary_shared_matrix = np.load(
        TEMP_FINAL_SHARED_METHYLATION_MATRIX_PATH,
        mmap_mode="r",
        allow_pickle=False,
    )

    shared_publication_checks = {
        "temporary_matrix_exists": (
            TEMP_FINAL_SHARED_METHYLATION_MATRIX_PATH.is_file()
        ),
        "final_matrix_not_yet_published": (
            not FINAL_SHARED_METHYLATION_MATRIX_PATH.is_file()
        ),
        "temporary_shape_matches_expected": (
            temporary_shared_matrix.shape == shared_matrix_shape
        ),
        "temporary_dtype_is_float32": (
            temporary_shared_matrix.dtype == SHARED_METHYLATION_MATRIX_DTYPE
        ),
        "temporary_matrix_is_fortran_contiguous": (
            temporary_shared_matrix.flags.f_contiguous
        ),
        "metadata_shape_matches_expected": (
            shared_build_metadata["matrix_shape"]
            == [
                int(shared_matrix_shape[0]),
                int(shared_matrix_shape[1]),
            ]
        ),
        "metadata_processed_columns_match_mapping": (
            int(shared_build_metadata["processed_column_count"])
            == len(shared_sample_mapping)
        ),
    }

    observed_summary = scan_shared_matrix_columns(
        matrix=temporary_shared_matrix,
        matrix_shape=shared_matrix_shape,
        progress_label="Validated shared columns",
    )

    observed_missing_beta_count = observed_summary[
        "observed_missing_beta_count"
    ]
    observed_out_of_range_beta_count = observed_summary[
        "observed_out_of_range_beta_count"
    ]
    observed_beta_min = observed_summary["observed_beta_min"]
    observed_beta_max = observed_summary["observed_beta_max"]

    validate_representative_source_columns(
        platform_name=HM27_PLATFORM,
        source_matrix_path=FINAL_HM27_MATRIX_PATH,
        source_row_indices=shared_hm27_matrix_row_indices,
        source_column_indices=shared_hm27_source_column_indices,
        target_column_indices=shared_hm27_target_column_indices,
        target_matrix=temporary_shared_matrix,
    )

    validate_representative_source_columns(
        platform_name=HM450_PLATFORM,
        source_matrix_path=FINAL_HM450_MATRIX_PATH,
        source_row_indices=shared_hm450_matrix_row_indices,
        source_column_indices=shared_hm450_source_column_indices,
        target_column_indices=shared_hm450_target_column_indices,
        target_matrix=temporary_shared_matrix,
    )

    shared_total_beta_values = int(np.prod(shared_matrix_shape))

    shared_publication_checks.update(
        {
            "observed_missing_matches_metadata": (
                observed_missing_beta_count
                == int(shared_build_metadata["selected_missing_beta_count"])
            ),
            "observed_nonmissing_matches_metadata": (
                shared_total_beta_values
                - observed_missing_beta_count
                == int(shared_build_metadata["selected_nonmissing_beta_count"])
            ),
            "observed_beta_values_are_in_unit_interval": (
                observed_out_of_range_beta_count == 0
            ),
        }
    )

    print()
    print("Shared HM27/HM450 publication checks:")

    for check_name, check_passed in shared_publication_checks.items():
        print(f"{check_name}: {check_passed}")

    if not all(shared_publication_checks.values()):
        failed_checks = [
            check_name
            for check_name, check_passed
            in shared_publication_checks.items()
            if not check_passed
        ]

        raise ValueError(
            "Shared methylation publication validation failed: "
            + ", ".join(failed_checks)
        )

    shared_published_metadata = dict(shared_build_metadata)

    shared_published_metadata.update(
        {
            "status": "published",
            "published_at_utc": datetime.now(timezone.utc).isoformat(),
            "observed_missing_beta_count": int(
                observed_missing_beta_count
            ),
            "observed_missing_beta_fraction": float(
                observed_missing_beta_count
                / shared_total_beta_values
            ),
            "observed_beta_min": float(observed_beta_min),
            "observed_beta_max": float(observed_beta_max),
            "observed_out_of_range_beta_count": int(
                observed_out_of_range_beta_count
            ),
        }
    )

The final shared HM27/HM450 methylation matrix already exists. Reloading the published artifact.
Published shared matrix checks:
final_matrix_exists: True
published_shape_matches_metadata: True
published_dtype_is_float32: True
published_matrix_is_fortran_contiguous: True

Shared HM27/HM450 matrix state reloaded.
Shape: 23,356 probes x 9,965 cases
Matrix: data/interim/methylation/tcga_primary_tumor_methylation_shared_hm27_hm450_final_case_level_beta_values.npy
Metadata: data/interim/methylation/tcga_primary_tumor_methylation_shared_hm27_hm450_final_case_level_beta_values.build_metadata.json


In [47]:
# =============================================================================
# Complete or reload shared HM27/HM450 matrix publication
# =============================================================================

SHARED_BUILD_METADATA_TEMP_PATH = (
    SHARED_BUILD_METADATA_PATH.with_name(
        SHARED_BUILD_METADATA_PATH.stem
        + ".tmp"
        + SHARED_BUILD_METADATA_PATH.suffix
    )
)


if FINAL_SHARED_METHYLATION_MATRIX_PATH.is_file():
    print(
        "The final shared HM27/HM450 methylation matrix already exists. "
        "Skipping publication and reloading the published artifact."
    )

    if not SHARED_BUILD_METADATA_PATH.is_file():
        raise FileNotFoundError(
            "The shared HM27/HM450 methylation metadata was not found."
        )

    with SHARED_BUILD_METADATA_PATH.open("r", encoding="utf-8") as metadata_file:
        shared_published_metadata = json.load(metadata_file)

    if shared_published_metadata.get("status") != "published":
        raise ValueError(
            "The final shared methylation matrix exists, but its metadata "
            "is not marked as published."
        )

    shared_build_metadata = dict(shared_published_metadata)

    shared_matrix_shape = tuple(
        int(value)
        for value in shared_published_metadata["matrix_shape"]
    )

    published_shared_matrix = np.load(
        FINAL_SHARED_METHYLATION_MATRIX_PATH,
        mmap_mode="r",
        allow_pickle=False,
    )

    published_checks = {
        "final_matrix_exists": FINAL_SHARED_METHYLATION_MATRIX_PATH.is_file(),
        "metadata_status_is_published": (
            shared_published_metadata.get("status") == "published"
        ),
        "published_shape_matches_metadata": (
            published_shared_matrix.shape == shared_matrix_shape
        ),
        "published_dtype_is_float32": (
            published_shared_matrix.dtype == SHARED_METHYLATION_MATRIX_DTYPE
        ),
        "published_matrix_is_fortran_contiguous": (
            published_shared_matrix.flags.f_contiguous
        ),
    }

    print("Published shared matrix checks:")

    for check_name, check_passed in published_checks.items():
        print(f"{check_name}: {check_passed}")

    if not all(published_checks.values()):
        raise ValueError(
            "Published shared methylation matrix validation failed."
        )

    if "observed_missing_beta_count" in shared_published_metadata:
        observed_missing_beta_count = int(
            shared_published_metadata["observed_missing_beta_count"]
        )
    elif "selected_missing_beta_count" in shared_published_metadata:
        observed_missing_beta_count = int(
            shared_published_metadata["selected_missing_beta_count"]
        )
    else:
        raise KeyError(
            "Shared methylation metadata does not contain a missing beta-value count."
        )

    shared_total_beta_values = int(np.prod(shared_matrix_shape))

    print()
    print("Shared HM27/HM450 matrix state reloaded.")
    print(
        f"Shape: {shared_matrix_shape[0]:,} probes x "
        f"{shared_matrix_shape[1]:,} cases"
    )
    print(
        f"Selected missing beta-values: "
        f"{observed_missing_beta_count:,} "
        f"({observed_missing_beta_count / shared_total_beta_values:.4%})"
    )
    print(
        "Matrix: "
        f"{project_relative_path(FINAL_SHARED_METHYLATION_MATRIX_PATH)}"
    )
    print(
        "Metadata: "
        f"{project_relative_path(SHARED_BUILD_METADATA_PATH)}"
    )

    del published_shared_matrix
    gc.collect()

else:
    if not TEMP_FINAL_SHARED_METHYLATION_MATRIX_PATH.is_file():
        raise FileNotFoundError(
            "The temporary shared methylation matrix was not found."
        )

    if "shared_published_metadata" not in globals():
        raise NameError(
            "shared_published_metadata is not available. "
            "Run the shared matrix publication-validation cell first."
        )

    shared_build_metadata = dict(shared_published_metadata)

    SHARED_BUILD_METADATA_TEMP_PATH.write_text(
        json.dumps(
            shared_published_metadata,
            indent=2,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )

    try:
        del temporary_shared_matrix
    except NameError:
        pass

    gc.collect()

    TEMP_FINAL_SHARED_METHYLATION_MATRIX_PATH.replace(
        FINAL_SHARED_METHYLATION_MATRIX_PATH
    )

    SHARED_BUILD_METADATA_TEMP_PATH.replace(
        SHARED_BUILD_METADATA_PATH
    )

    published_shared_matrix = np.load(
        FINAL_SHARED_METHYLATION_MATRIX_PATH,
        mmap_mode="r",
        allow_pickle=False,
    )

    published_checks = {
        "final_matrix_exists": FINAL_SHARED_METHYLATION_MATRIX_PATH.is_file(),
        "temporary_matrix_was_removed": (
            not TEMP_FINAL_SHARED_METHYLATION_MATRIX_PATH.is_file()
        ),
        "published_shape_matches_expected": (
            published_shared_matrix.shape == shared_matrix_shape
        ),
        "published_dtype_is_float32": (
            published_shared_matrix.dtype == SHARED_METHYLATION_MATRIX_DTYPE
        ),
        "published_matrix_is_fortran_contiguous": (
            published_shared_matrix.flags.f_contiguous
        ),
    }

    print("Published shared matrix checks:")

    for check_name, check_passed in published_checks.items():
        print(f"{check_name}: {check_passed}")

    if not all(published_checks.values()):
        raise ValueError(
            "Published shared methylation matrix validation failed."
        )

    print()
    print("Shared HM27/HM450 matrix published.")
    print(
        f"Shape: {shared_matrix_shape[0]:,} probes x "
        f"{shared_matrix_shape[1]:,} cases"
    )
    print(
        "Matrix: "
        f"{project_relative_path(FINAL_SHARED_METHYLATION_MATRIX_PATH)}"
    )
    print(
        "Metadata: "
        f"{project_relative_path(SHARED_BUILD_METADATA_PATH)}"
    )

    del published_shared_matrix
    gc.collect()

The final shared HM27/HM450 methylation matrix already exists. Skipping publication and reloading the published artifact.
Published shared matrix checks:
final_matrix_exists: True
metadata_status_is_published: True
published_shape_matches_metadata: True
published_dtype_is_float32: True
published_matrix_is_fortran_contiguous: True

Shared HM27/HM450 matrix state reloaded.
Shape: 23,356 probes x 9,965 cases
Selected missing beta-values: 752,180 (0.3232%)
Matrix: data/interim/methylation/tcga_primary_tumor_methylation_shared_hm27_hm450_final_case_level_beta_values.npy
Metadata: data/interim/methylation/tcga_primary_tumor_methylation_shared_hm27_hm450_final_case_level_beta_values.build_metadata.json


In [49]:
# =============================================================================
# Publish final sample and probe mapping artifacts
# =============================================================================

import json
from datetime import datetime, timezone


INTEGRATION_METADATA_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


required_publication_objects = [
    "final_sample_mapping",
    "final_probe_mapping",
    "FINAL_SAMPLE_MAPPING_PATH",
    "FINAL_PROBE_MAPPING_PATH",
    "FINAL_CONSUMABLE_TEMP_PATHS",
    "hm27_sample_mapping",
    "hm450_sample_mapping",
    "shared_sample_mapping",
]

missing_publication_objects = [
    object_name
    for object_name in required_publication_objects
    if object_name not in globals()
]

if missing_publication_objects:
    raise NameError(
        "Missing final-publication objects: "
        + ", ".join(missing_publication_objects)
    )


def complete_integer_index(series, expected_count):
    values = pd.to_numeric(
        series,
        errors="raise",
    ).to_numpy(dtype=np.int64)

    return (
        len(values) == expected_count
        and np.array_equal(
            values,
            np.arange(expected_count, dtype=np.int64),
        )
    )


def complete_nullable_integer_index(series, expected_count):
    values = (
        pd.to_numeric(
            series.dropna(),
            errors="raise",
        )
        .astype("int64")
        .to_numpy()
    )

    return (
        len(values) == expected_count
        and np.array_equal(
            values,
            np.arange(expected_count, dtype=np.int64),
        )
    )


def truthy_all(series):
    return (
        series.astype("string")
        .str.lower()
        .isin(["true", "1"])
        .all()
    )


def string_values(series):
    return (
        series.astype("string")
        .fillna("<NA>")
        .tolist()
    )


def validate_written_sample_mapping(written_sample_mapping):
    expected_hm27_count = int(len(hm27_sample_mapping))
    expected_hm450_count = int(len(hm450_sample_mapping))
    expected_sample_count = int(len(final_sample_mapping))

    expected_platform_counts = {
        str(platform): int(count)
        for platform, count in (
            final_sample_mapping["methylation_platform"]
            .astype("string")
            .value_counts()
            .sort_index()
            .items()
        )
    }

    observed_platform_counts = {
        str(platform): int(count)
        for platform, count in (
            written_sample_mapping["methylation_platform"]
            .astype("string")
            .value_counts()
            .sort_index()
            .items()
        )
    }

    return {
        "sample_mapping_file_is_not_empty": (
            not written_sample_mapping.empty
        ),
        "sample_mapping_shape_matches_memory": (
            tuple(written_sample_mapping.shape)
            == tuple(final_sample_mapping.shape)
        ),
        "sample_mapping_columns_match_memory": (
            written_sample_mapping.columns.tolist()
            == final_sample_mapping.columns.tolist()
        ),
        "case_ids_match_memory_order": (
            string_values(written_sample_mapping[CASE_KEY])
            == string_values(final_sample_mapping[CASE_KEY])
        ),
        "sample_ids_match_memory_order": (
            string_values(written_sample_mapping[SAMPLE_KEY])
            == string_values(final_sample_mapping[SAMPLE_KEY])
        ),
        "rna_file_ids_match_memory_order": (
            string_values(written_sample_mapping[RNA_FILE_KEY])
            == string_values(final_sample_mapping[RNA_FILE_KEY])
        ),
        "methylation_file_ids_match_memory_order": (
            string_values(written_sample_mapping[METHYLATION_FILE_KEY])
            == string_values(final_sample_mapping[METHYLATION_FILE_KEY])
        ),
        "one_row_per_case": (
            written_sample_mapping[CASE_KEY].astype("string").is_unique
        ),
        "final_sample_column_indices_are_complete": (
            complete_integer_index(
                written_sample_mapping["final_sample_column_index"],
                expected_sample_count,
            )
        ),
        "rna_final_matrix_column_indices_are_complete": (
            complete_integer_index(
                written_sample_mapping["rna_final_matrix_column_index"],
                expected_sample_count,
            )
        ),
        "shared_matrix_column_indices_are_complete": (
            complete_integer_index(
                written_sample_mapping["shared_matrix_column_index"],
                expected_sample_count,
            )
        ),
        "hm27_matrix_column_indices_are_complete": (
            complete_nullable_integer_index(
                written_sample_mapping["hm27_matrix_column_index"],
                expected_hm27_count,
            )
        ),
        "hm450_matrix_column_indices_are_complete": (
            complete_nullable_integer_index(
                written_sample_mapping["hm450_matrix_column_index"],
                expected_hm450_count,
            )
        ),
        "platform_counts_match_memory": (
            observed_platform_counts == expected_platform_counts
        ),
        "all_rna_payloads_exist": (
            truthy_all(written_sample_mapping["rna_payload_exists"])
        ),
        "all_methylation_payloads_exist": (
            truthy_all(written_sample_mapping["methylation_payload_exists"])
        ),
    }


def validate_written_probe_mapping(written_probe_mapping):
    expected_probe_counts = {
        str(representation): int(count)
        for representation, count in (
            final_probe_mapping
            .groupby("representation", sort=False)
            .size()
            .items()
        )
    }

    observed_probe_counts = {
        str(representation): int(count)
        for representation, count in (
            written_probe_mapping
            .groupby("representation", sort=False)
            .size()
            .items()
        )
    }

    representation_index_checks = {}
    representation_probe_checks = {}

    for representation, expected_count in expected_probe_counts.items():
        observed_rows = written_probe_mapping.loc[
            written_probe_mapping["representation"]
            .astype("string")
            .eq(representation)
        ]

        expected_rows = final_probe_mapping.loc[
            final_probe_mapping["representation"]
            .astype("string")
            .eq(representation)
        ]

        representation_index_checks[
            f"{representation}_row_indices_are_complete"
        ] = complete_integer_index(
            observed_rows["matrix_row_index"],
            expected_count,
        )

        representation_probe_checks[
            f"{representation}_probe_ids_match_memory_order"
        ] = (
            string_values(observed_rows["probe_id"])
            == string_values(expected_rows["probe_id"])
        )

    return {
        "probe_mapping_file_is_not_empty": (
            not written_probe_mapping.empty
        ),
        "probe_mapping_shape_matches_memory": (
            tuple(written_probe_mapping.shape)
            == tuple(final_probe_mapping.shape)
        ),
        "probe_mapping_columns_match_memory": (
            written_probe_mapping.columns.tolist()
            == final_probe_mapping.columns.tolist()
        ),
        "probe_representation_counts_match_memory": (
            observed_probe_counts == expected_probe_counts
        ),
        "all_probe_qc_flags_are_true": (
            truthy_all(written_probe_mapping["probe_qc_eligible"])
        ),
        **representation_index_checks,
        **representation_probe_checks,
    }


def publish_or_reuse_csv_artifact(
    *,
    artifact_name,
    dataframe,
    output_path,
    temporary_path,
    validator,
):
    temporary_path.unlink(missing_ok=True)

    if output_path.is_file():
        print(
            f"{artifact_name} already exists. "
            "Reloading the persisted artifact."
        )
    else:
        dataframe.to_csv(
            temporary_path,
            index=False,
        )

        temporary_dataframe = pd.read_csv(
            temporary_path,
            low_memory=False,
        )

        temporary_checks = validator(temporary_dataframe)

        print(f"{artifact_name} temporary checks:")
        for check_name, check_passed in temporary_checks.items():
            print(f"{check_name}: {check_passed}")

        if not all(temporary_checks.values()):
            temporary_path.unlink(missing_ok=True)
            raise ValueError(
                f"{artifact_name} temporary validation failed."
            )

        temporary_path.replace(output_path)

    written_dataframe = pd.read_csv(
        output_path,
        low_memory=False,
    )

    written_checks = validator(written_dataframe)

    print()
    print(f"{artifact_name} written-artifact checks:")
    for check_name, check_passed in written_checks.items():
        print(f"{check_name}: {check_passed}")

    if not all(written_checks.values()):
        failed_checks = [
            check_name
            for check_name, check_passed in written_checks.items()
            if not check_passed
        ]

        raise ValueError(
            f"{artifact_name} written validation failed: "
            + ", ".join(failed_checks)
        )

    return written_dataframe


published_final_sample_mapping = publish_or_reuse_csv_artifact(
    artifact_name="final_sample_mapping",
    dataframe=final_sample_mapping,
    output_path=FINAL_SAMPLE_MAPPING_PATH,
    temporary_path=FINAL_CONSUMABLE_TEMP_PATHS["sample_mapping"],
    validator=validate_written_sample_mapping,
)

published_final_probe_mapping = publish_or_reuse_csv_artifact(
    artifact_name="final_probe_mapping",
    dataframe=final_probe_mapping,
    output_path=FINAL_PROBE_MAPPING_PATH,
    temporary_path=FINAL_CONSUMABLE_TEMP_PATHS["probe_mapping"],
    validator=validate_written_probe_mapping,
)


print()
print("Final mapping artifacts published.")
print(
    "Sample mapping: "
    f"{project_relative_path(FINAL_SAMPLE_MAPPING_PATH)}"
)
print(
    "Probe mapping: "
    f"{project_relative_path(FINAL_PROBE_MAPPING_PATH)}"
)
print(f"Primary cases: {len(published_final_sample_mapping):,}")
print(f"Probe-mapping rows: {len(published_final_probe_mapping):,}")

# =============================================================================
# Publish final multi-omic consumables metadata
# =============================================================================

import gc
import json
from datetime import datetime, timezone


HASH_FINAL_NUMERIC_MATRICES = True


required_metadata_objects = [
    "published_final_sample_mapping",
    "published_final_probe_mapping",
    "FINAL_CONSUMABLES_METADATA_PATH",
    "FINAL_CONSUMABLE_PATHS",
    "FINAL_CONSUMABLE_TEMP_PATHS",
    "FINAL_RNA_MATRIX_PATH",
    "FINAL_RNA_FEATURE_INDEX_PATH",
    "FINAL_HM27_MATRIX_PATH",
    "FINAL_HM450_MATRIX_PATH",
    "FINAL_SHARED_METHYLATION_MATRIX_PATH",
]

missing_metadata_objects = [
    object_name
    for object_name in required_metadata_objects
    if object_name not in globals()
]

if missing_metadata_objects:
    raise NameError(
        "Missing final-metadata objects: "
        + ", ".join(missing_metadata_objects)
    )


def file_artifact_record(path, *, compute_sha256=True):
    if not path.is_file():
        raise FileNotFoundError(
            f"Required final artifact was not found: {path}"
        )

    record = {
        "path": project_relative_path(path),
        "size_bytes": int(path.stat().st_size),
    }

    if compute_sha256:
        print(
            "Hashing artifact: "
            f"{project_relative_path(path)}"
        )
        record["sha256"] = calculate_sha256(path)
    else:
        record["sha256"] = None
        record["sha256_policy"] = (
            "not recomputed in this final metadata cell"
        )

    return record


def npy_artifact_record(
    path,
    *,
    row_axis,
    column_axis,
    compute_sha256=True,
):
    matrix = np.load(
        path,
        mmap_mode="r",
        allow_pickle=False,
    )

    record = file_artifact_record(
        path,
        compute_sha256=compute_sha256,
    )

    record.update(
        {
            "shape": [int(value) for value in matrix.shape],
            "dtype": matrix.dtype.name,
            "fortran_order": bool(matrix.flags.f_contiguous),
            "row_axis": row_axis,
            "column_axis": column_axis,
        }
    )

    del matrix
    gc.collect()

    return record


def csv_artifact_record(path, dataframe, *, compute_sha256=True):
    record = file_artifact_record(
        path,
        compute_sha256=compute_sha256,
    )

    record.update(
        {
            "rows": int(dataframe.shape[0]),
            "columns": int(dataframe.shape[1]),
            "column_names": dataframe.columns.tolist(),
        }
    )

    return record


def count_matrix_missing_values(path, *, chunk_columns=256):
    matrix = np.load(
        path,
        mmap_mode="r",
        allow_pickle=False,
    )

    missing_count = 0

    for start in range(0, matrix.shape[1], chunk_columns):
        stop = min(start + chunk_columns, matrix.shape[1])

        block = np.asarray(matrix[:, start:stop])
        missing_count += int(np.isnan(block).sum())

        del block

    shape = tuple(int(value) for value in matrix.shape)

    del matrix
    gc.collect()

    return int(missing_count), shape


def optional_source_record(path):
    if not path.is_file():
        return {
            "path": project_relative_path(path),
            "exists": False,
        }

    compute_hash = path.suffix.lower() != ".npy"

    return {
        **file_artifact_record(
            path,
            compute_sha256=compute_hash,
        ),
        "exists": True,
    }


for artifact_name, output_path in FINAL_CONSUMABLE_PATHS.items():
    if artifact_name == "metadata":
        continue

    if not output_path.is_file():
        raise FileNotFoundError(
            "A required final consumable is missing before metadata "
            f"publication: {artifact_name}"
        )


metadata_temporary_path = FINAL_CONSUMABLE_TEMP_PATHS["metadata"]
metadata_temporary_path.unlink(missing_ok=True)

if FINAL_CONSUMABLES_METADATA_PATH.is_file():
    print(
        "Final consumables metadata already exists. "
        "Reloading the persisted artifact."
    )

    with FINAL_CONSUMABLES_METADATA_PATH.open(
        "r",
        encoding="utf-8",
    ) as metadata_file:
        final_consumables_metadata = json.load(metadata_file)

else:
    hm27_missing_beta_count, hm27_matrix_shape_from_disk = (
        count_matrix_missing_values(FINAL_HM27_MATRIX_PATH)
    )

    final_artifact_records = {
        "rna_matrix": npy_artifact_record(
            FINAL_RNA_MATRIX_PATH,
            row_axis="gene",
            column_axis="final_sample_column_index",
            compute_sha256=HASH_FINAL_NUMERIC_MATRICES,
        ),
        "rna_feature_index": csv_artifact_record(
            FINAL_RNA_FEATURE_INDEX_PATH,
            pd.read_csv(FINAL_RNA_FEATURE_INDEX_PATH, low_memory=False),
        ),
        "hm27_matrix": npy_artifact_record(
            FINAL_HM27_MATRIX_PATH,
            row_axis="hm27_probe_mapping.matrix_row_index",
            column_axis="hm27_matrix_column_index",
            compute_sha256=HASH_FINAL_NUMERIC_MATRICES,
        ),
        "hm450_matrix": npy_artifact_record(
            FINAL_HM450_MATRIX_PATH,
            row_axis="hm450_probe_mapping.matrix_row_index",
            column_axis="hm450_matrix_column_index",
            compute_sha256=HASH_FINAL_NUMERIC_MATRICES,
        ),
        "shared_methylation_matrix": npy_artifact_record(
            FINAL_SHARED_METHYLATION_MATRIX_PATH,
            row_axis="shared_probe_mapping.matrix_row_index",
            column_axis="shared_matrix_column_index",
            compute_sha256=HASH_FINAL_NUMERIC_MATRICES,
        ),
        "sample_mapping": csv_artifact_record(
            FINAL_SAMPLE_MAPPING_PATH,
            published_final_sample_mapping,
        ),
        "probe_mapping": csv_artifact_record(
            FINAL_PROBE_MAPPING_PATH,
            published_final_probe_mapping,
        ),
    }

    final_artifact_records["hm27_matrix"].update(
        {
            "selected_missing_beta_count": int(
                hm27_missing_beta_count
            ),
            "selected_nonmissing_beta_count": int(
                np.prod(hm27_matrix_shape_from_disk)
                - hm27_missing_beta_count
            ),
        }
    )

    if "hm450_build_metadata" in globals():
        final_artifact_records["hm450_matrix"].update(
            {
                "selected_missing_beta_count": int(
                    hm450_build_metadata[
                        "selected_missing_beta_count"
                    ]
                ),
                "selected_nonmissing_beta_count": int(
                    hm450_build_metadata[
                        "selected_nonmissing_beta_count"
                    ]
                ),
                "build_metadata_path": project_relative_path(
                    HM450_BUILD_METADATA_PATH
                ),
            }
        )

    if "shared_published_metadata" in globals():
        final_artifact_records[
            "shared_methylation_matrix"
        ].update(
            {
                "selected_missing_beta_count": int(
                    shared_published_metadata[
                        "selected_missing_beta_count"
                    ]
                ),
                "selected_nonmissing_beta_count": int(
                    shared_published_metadata[
                        "selected_nonmissing_beta_count"
                    ]
                ),
                "observed_missing_beta_count": int(
                    shared_published_metadata.get(
                        "observed_missing_beta_count",
                        shared_published_metadata[
                            "selected_missing_beta_count"
                        ],
                    )
                ),
                "build_metadata_path": project_relative_path(
                    SHARED_BUILD_METADATA_PATH
                ),
            }
        )

    final_consumables_metadata = {
        "artifact_type": (
            "tcga_primary_tumor_multiomic_final_consumables"
        ),
        "status": "published",
        "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
        "notebook": "203_tcga_multiomic_integration.ipynb",
        "analysis_layer": "phase2_tumor_discovery_layer",
        "cohort_definition": {
            "primary_case_count": int(
                len(published_final_sample_mapping)
            ),
            "selection_unit": "one primary tumor case",
            "case_id_column": CASE_KEY,
            "sample_id_column": SAMPLE_KEY,
            "rna_file_id_column": RNA_FILE_KEY,
            "methylation_file_id_column": METHYLATION_FILE_KEY,
            "methylation_platform_counts": {
                str(platform): int(count)
                for platform, count in (
                    published_final_sample_mapping[
                        "methylation_platform"
                    ]
                    .astype("string")
                    .value_counts()
                    .sort_index()
                    .items()
                )
            },
        },
        "selection_policy": {
            "case_level_mapping_path": project_relative_path(
                FINAL_MULTIOMIC_PAIR_MAPPING_PATH
            ),
            "case_level_selection_policy_path": project_relative_path(
                CASE_LEVEL_SELECTION_POLICY_PATH
            ),
            "unresolved_cases_excluded": int(
                len(unresolved_case_ids)
            )
            if "unresolved_case_ids" in globals()
            else None,
            "no_arbitrary_case_level_tiebreak": True,
        },
        "matrix_policy": {
            "rna_representation": (
                "unstranded raw counts; no normalization in notebook 203"
            ),
            "methylation_representation": (
                "raw beta-values; NaN retained; no imputation, "
                "normalization, or batch correction"
            ),
            "hm27_hm450_policy": (
                "platform-specific matrices remain separate; the shared "
                "matrix contains eligible probes present in both platforms"
            ),
        },
        "probe_mapping": {
            "representation_counts": {
                str(representation): int(count)
                for representation, count in (
                    published_final_probe_mapping
                    .groupby("representation", sort=False)
                    .size()
                    .items()
                )
            },
        },
        "final_artifacts": final_artifact_records,
        "source_artifacts": {
            "qc_annotated_pair_inventory": optional_source_record(
                METHYLATION_QC_ANNOTATED_PAIR_INVENTORY_PATH
            ),
            "rna_candidate_raw_count_matrix": optional_source_record(
                RAW_COUNT_MATRIX_PATH
            ),
            "rna_gene_feature_index": optional_source_record(
                RNA_GENE_FEATURE_INDEX_PATH
            ),
            "rna_file_qc_inventory": optional_source_record(
                RNA_FILE_QC_INVENTORY_PATH
            ),
            "rna_raw_counts_metadata": optional_source_record(
                RNA_RAW_COUNTS_METADATA_PATH
            ),
            "methylation_file_qc_metrics": optional_source_record(
                METHYLATION_FILE_QC_METRICS_PATH
            ),
            "probe_level_qc_metrics": optional_source_record(
                PROBE_LEVEL_QC_METRICS_PATH
            ),
            "shared_probe_qc_inventory": optional_source_record(
                SHARED_PROBE_QC_INVENTORY_PATH
            ),
            "rna_file_index": optional_source_record(
                RNA_FILE_INDEX_PATH
            ),
            "rna_cohort_freeze": optional_source_record(
                RNA_COHORT_FREEZE_PATH
            ),
            "methylation_file_index": optional_source_record(
                METHYLATION_FILE_INDEX_PATH
            ),
            "methylation_cohort_freeze": optional_source_record(
                METHYLATION_COHORT_FREEZE_PATH
            ),
        },
    }

    metadata_temporary_path.write_text(
        json.dumps(
            final_consumables_metadata,
            indent=2,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )

    with metadata_temporary_path.open(
        "r",
        encoding="utf-8",
    ) as metadata_file:
        temporary_metadata = json.load(metadata_file)

    metadata_write_checks = {
        "temporary_metadata_status_is_published": (
            temporary_metadata.get("status") == "published"
        ),
        "temporary_metadata_case_count_matches_mapping": (
            int(
                temporary_metadata["cohort_definition"][
                    "primary_case_count"
                ]
            )
            == len(published_final_sample_mapping)
        ),
        "temporary_metadata_has_all_final_artifacts": (
            set(temporary_metadata["final_artifacts"])
            == set(FINAL_CONSUMABLE_PATHS) - {"metadata"}
        ),
    }

    print("Final consumables metadata temporary checks:")
    for check_name, check_passed in metadata_write_checks.items():
        print(f"{check_name}: {check_passed}")

    if not all(metadata_write_checks.values()):
        metadata_temporary_path.unlink(missing_ok=True)
        raise ValueError(
            "Final consumables metadata temporary validation failed."
        )

    metadata_temporary_path.replace(
        FINAL_CONSUMABLES_METADATA_PATH
    )


with FINAL_CONSUMABLES_METADATA_PATH.open(
    "r",
    encoding="utf-8",
) as metadata_file:
    final_consumables_metadata = json.load(metadata_file)


metadata_checks = {
    "metadata_file_exists": (
        FINAL_CONSUMABLES_METADATA_PATH.is_file()
    ),
    "metadata_status_is_published": (
        final_consumables_metadata.get("status") == "published"
    ),
    "metadata_case_count_matches_mapping": (
        int(
            final_consumables_metadata["cohort_definition"][
                "primary_case_count"
            ]
        )
        == len(published_final_sample_mapping)
    ),
    "metadata_sample_mapping_path_matches": (
        final_consumables_metadata["final_artifacts"][
            "sample_mapping"
        ]["path"]
        == project_relative_path(FINAL_SAMPLE_MAPPING_PATH)
    ),
    "metadata_probe_mapping_path_matches": (
        final_consumables_metadata["final_artifacts"][
            "probe_mapping"
        ]["path"]
        == project_relative_path(FINAL_PROBE_MAPPING_PATH)
    ),
}

print()
print("Final consumables metadata checks:")
for check_name, check_passed in metadata_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(metadata_checks.values()):
    raise ValueError(
        "Final consumables metadata validation failed."
    )


print()
print("Final multi-omic consumables metadata published.")
print(
    "Metadata: "
    f"{project_relative_path(FINAL_CONSUMABLES_METADATA_PATH)}"
)


# =============================================================================
# Final multi-omic consumable closure checks
# =============================================================================

import gc
import json


with FINAL_CONSUMABLES_METADATA_PATH.open(
    "r",
    encoding="utf-8",
) as metadata_file:
    final_consumables_metadata = json.load(metadata_file)


closure_sample_mapping = pd.read_csv(
    FINAL_SAMPLE_MAPPING_PATH,
    low_memory=False,
)

closure_probe_mapping = pd.read_csv(
    FINAL_PROBE_MAPPING_PATH,
    low_memory=False,
)

closure_rna_matrix = np.load(
    FINAL_RNA_MATRIX_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

closure_hm27_matrix = np.load(
    FINAL_HM27_MATRIX_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

closure_hm450_matrix = np.load(
    FINAL_HM450_MATRIX_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

closure_shared_matrix = np.load(
    FINAL_SHARED_METHYLATION_MATRIX_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

closure_rna_feature_index = pd.read_csv(
    FINAL_RNA_FEATURE_INDEX_PATH,
    low_memory=False,
)


def metadata_path_matches(artifact_name, path):
    return (
        final_consumables_metadata["final_artifacts"][
            artifact_name
        ]["path"]
        == project_relative_path(path)
    )


final_inprogress_paths_absent = all(
    not temporary_path.is_file()
    for temporary_path in FINAL_CONSUMABLE_TEMP_PATHS.values()
)


closure_checks = {
    "all_final_consumable_paths_exist": all(
        output_path.is_file()
        for output_path in FINAL_CONSUMABLE_PATHS.values()
    ),
    "all_inprogress_paths_are_absent": final_inprogress_paths_absent,
    "metadata_status_is_published": (
        final_consumables_metadata.get("status") == "published"
    ),
    "metadata_case_count_matches_sample_mapping": (
        int(
            final_consumables_metadata["cohort_definition"][
                "primary_case_count"
            ]
        )
        == len(closure_sample_mapping)
    ),
    "sample_mapping_has_one_row_per_case": (
        closure_sample_mapping[CASE_KEY].astype("string").is_unique
    ),
    "sample_mapping_final_indices_are_complete": (
        complete_integer_index(
            closure_sample_mapping["final_sample_column_index"],
            len(closure_sample_mapping),
        )
    ),
    "rna_matrix_shape_matches_feature_and_sample_axes": (
        closure_rna_matrix.shape
        == (
            len(closure_rna_feature_index),
            len(closure_sample_mapping),
        )
    ),
    "rna_matrix_dtype_is_uint32": (
        closure_rna_matrix.dtype == np.dtype("uint32")
    ),
    "rna_matrix_is_fortran_contiguous": (
        closure_rna_matrix.flags.f_contiguous
    ),
    "hm27_matrix_shape_matches_mappings": (
        closure_hm27_matrix.shape
        == (
            int(
                (
                    closure_probe_mapping["representation"]
                    .astype("string")
                    .eq("hm27")
                ).sum()
            ),
            int(
                closure_sample_mapping[
                    "hm27_matrix_column_index"
                ].notna().sum()
            ),
        )
    ),
    "hm450_matrix_shape_matches_mappings": (
        closure_hm450_matrix.shape
        == (
            int(
                (
                    closure_probe_mapping["representation"]
                    .astype("string")
                    .eq("hm450")
                ).sum()
            ),
            int(
                closure_sample_mapping[
                    "hm450_matrix_column_index"
                ].notna().sum()
            ),
        )
    ),
    "shared_matrix_shape_matches_mappings": (
        closure_shared_matrix.shape
        == (
            int(
                (
                    closure_probe_mapping["representation"]
                    .astype("string")
                    .eq(SHARED_METHYLATION_REPRESENTATION)
                ).sum()
            ),
            len(closure_sample_mapping),
        )
    ),
    "hm27_matrix_dtype_is_float32": (
        closure_hm27_matrix.dtype == np.dtype("float32")
    ),
    "hm450_matrix_dtype_is_float32": (
        closure_hm450_matrix.dtype == np.dtype("float32")
    ),
    "shared_matrix_dtype_is_float32": (
        closure_shared_matrix.dtype == np.dtype("float32")
    ),
    "hm27_matrix_is_fortran_contiguous": (
        closure_hm27_matrix.flags.f_contiguous
    ),
    "hm450_matrix_is_fortran_contiguous": (
        closure_hm450_matrix.flags.f_contiguous
    ),
    "shared_matrix_is_fortran_contiguous": (
        closure_shared_matrix.flags.f_contiguous
    ),
    "metadata_rna_matrix_path_matches": (
        metadata_path_matches("rna_matrix", FINAL_RNA_MATRIX_PATH)
    ),
    "metadata_hm27_matrix_path_matches": (
        metadata_path_matches("hm27_matrix", FINAL_HM27_MATRIX_PATH)
    ),
    "metadata_hm450_matrix_path_matches": (
        metadata_path_matches("hm450_matrix", FINAL_HM450_MATRIX_PATH)
    ),
    "metadata_shared_matrix_path_matches": (
        metadata_path_matches(
            "shared_methylation_matrix",
            FINAL_SHARED_METHYLATION_MATRIX_PATH,
        )
    ),
    "metadata_sample_mapping_path_matches": (
        metadata_path_matches("sample_mapping", FINAL_SAMPLE_MAPPING_PATH)
    ),
    "metadata_probe_mapping_path_matches": (
        metadata_path_matches("probe_mapping", FINAL_PROBE_MAPPING_PATH)
    ),
}


print("Final multi-omic consumable closure checks:")

for check_name, check_passed in closure_checks.items():
    print(f"{check_name}: {check_passed}")


if not all(closure_checks.values()):
    failed_checks = [
        check_name
        for check_name, check_passed in closure_checks.items()
        if not check_passed
    ]

    raise ValueError(
        "Final multi-omic consumable closure failed: "
        + ", ".join(failed_checks)
    )


print()
print("Final multi-omic consumable artifacts validated.")
print(f"Primary cases: {len(closure_sample_mapping):,}")
print(f"RNA matrix shape: {closure_rna_matrix.shape}")
print(f"HM27 matrix shape: {closure_hm27_matrix.shape}")
print(f"HM450 matrix shape: {closure_hm450_matrix.shape}")
print(f"Shared methylation matrix shape: {closure_shared_matrix.shape}")
print(
    "Metadata: "
    f"{project_relative_path(FINAL_CONSUMABLES_METADATA_PATH)}"
)


del closure_rna_matrix
del closure_hm27_matrix
del closure_hm450_matrix
del closure_shared_matrix
gc.collect()

# =============================================================================
# Publish final sample and probe mapping artifacts
# =============================================================================

import json
from datetime import datetime, timezone


INTEGRATION_METADATA_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


required_publication_objects = [
    "final_sample_mapping",
    "final_probe_mapping",
    "FINAL_SAMPLE_MAPPING_PATH",
    "FINAL_PROBE_MAPPING_PATH",
    "FINAL_CONSUMABLE_TEMP_PATHS",
    "hm27_sample_mapping",
    "hm450_sample_mapping",
    "shared_sample_mapping",
]

missing_publication_objects = [
    object_name
    for object_name in required_publication_objects
    if object_name not in globals()
]

if missing_publication_objects:
    raise NameError(
        "Missing final-publication objects: "
        + ", ".join(missing_publication_objects)
    )


def complete_integer_index(series, expected_count):
    values = pd.to_numeric(
        series,
        errors="raise",
    ).to_numpy(dtype=np.int64)

    return (
        len(values) == expected_count
        and np.array_equal(
            values,
            np.arange(expected_count, dtype=np.int64),
        )
    )


def complete_nullable_integer_index(series, expected_count):
    values = (
        pd.to_numeric(
            series.dropna(),
            errors="raise",
        )
        .astype("int64")
        .to_numpy()
    )

    return (
        len(values) == expected_count
        and np.array_equal(
            values,
            np.arange(expected_count, dtype=np.int64),
        )
    )


def truthy_all(series):
    return (
        series.astype("string")
        .str.lower()
        .isin(["true", "1"])
        .all()
    )


def string_values(series):
    return (
        series.astype("string")
        .fillna("<NA>")
        .tolist()
    )


def validate_written_sample_mapping(written_sample_mapping):
    expected_hm27_count = int(len(hm27_sample_mapping))
    expected_hm450_count = int(len(hm450_sample_mapping))
    expected_sample_count = int(len(final_sample_mapping))

    expected_platform_counts = {
        str(platform): int(count)
        for platform, count in (
            final_sample_mapping["methylation_platform"]
            .astype("string")
            .value_counts()
            .sort_index()
            .items()
        )
    }

    observed_platform_counts = {
        str(platform): int(count)
        for platform, count in (
            written_sample_mapping["methylation_platform"]
            .astype("string")
            .value_counts()
            .sort_index()
            .items()
        )
    }

    return {
        "sample_mapping_file_is_not_empty": (
            not written_sample_mapping.empty
        ),
        "sample_mapping_shape_matches_memory": (
            tuple(written_sample_mapping.shape)
            == tuple(final_sample_mapping.shape)
        ),
        "sample_mapping_columns_match_memory": (
            written_sample_mapping.columns.tolist()
            == final_sample_mapping.columns.tolist()
        ),
        "case_ids_match_memory_order": (
            string_values(written_sample_mapping[CASE_KEY])
            == string_values(final_sample_mapping[CASE_KEY])
        ),
        "sample_ids_match_memory_order": (
            string_values(written_sample_mapping[SAMPLE_KEY])
            == string_values(final_sample_mapping[SAMPLE_KEY])
        ),
        "rna_file_ids_match_memory_order": (
            string_values(written_sample_mapping[RNA_FILE_KEY])
            == string_values(final_sample_mapping[RNA_FILE_KEY])
        ),
        "methylation_file_ids_match_memory_order": (
            string_values(written_sample_mapping[METHYLATION_FILE_KEY])
            == string_values(final_sample_mapping[METHYLATION_FILE_KEY])
        ),
        "one_row_per_case": (
            written_sample_mapping[CASE_KEY].astype("string").is_unique
        ),
        "final_sample_column_indices_are_complete": (
            complete_integer_index(
                written_sample_mapping["final_sample_column_index"],
                expected_sample_count,
            )
        ),
        "rna_final_matrix_column_indices_are_complete": (
            complete_integer_index(
                written_sample_mapping["rna_final_matrix_column_index"],
                expected_sample_count,
            )
        ),
        "shared_matrix_column_indices_are_complete": (
            complete_integer_index(
                written_sample_mapping["shared_matrix_column_index"],
                expected_sample_count,
            )
        ),
        "hm27_matrix_column_indices_are_complete": (
            complete_nullable_integer_index(
                written_sample_mapping["hm27_matrix_column_index"],
                expected_hm27_count,
            )
        ),
        "hm450_matrix_column_indices_are_complete": (
            complete_nullable_integer_index(
                written_sample_mapping["hm450_matrix_column_index"],
                expected_hm450_count,
            )
        ),
        "platform_counts_match_memory": (
            observed_platform_counts == expected_platform_counts
        ),
        "all_rna_payloads_exist": (
            truthy_all(written_sample_mapping["rna_payload_exists"])
        ),
        "all_methylation_payloads_exist": (
            truthy_all(written_sample_mapping["methylation_payload_exists"])
        ),
    }


def validate_written_probe_mapping(written_probe_mapping):
    expected_probe_counts = {
        str(representation): int(count)
        for representation, count in (
            final_probe_mapping
            .groupby("representation", sort=False)
            .size()
            .items()
        )
    }

    observed_probe_counts = {
        str(representation): int(count)
        for representation, count in (
            written_probe_mapping
            .groupby("representation", sort=False)
            .size()
            .items()
        )
    }

    representation_index_checks = {}
    representation_probe_checks = {}

    for representation, expected_count in expected_probe_counts.items():
        observed_rows = written_probe_mapping.loc[
            written_probe_mapping["representation"]
            .astype("string")
            .eq(representation)
        ]

        expected_rows = final_probe_mapping.loc[
            final_probe_mapping["representation"]
            .astype("string")
            .eq(representation)
        ]

        representation_index_checks[
            f"{representation}_row_indices_are_complete"
        ] = complete_integer_index(
            observed_rows["matrix_row_index"],
            expected_count,
        )

        representation_probe_checks[
            f"{representation}_probe_ids_match_memory_order"
        ] = (
            string_values(observed_rows["probe_id"])
            == string_values(expected_rows["probe_id"])
        )

    return {
        "probe_mapping_file_is_not_empty": (
            not written_probe_mapping.empty
        ),
        "probe_mapping_shape_matches_memory": (
            tuple(written_probe_mapping.shape)
            == tuple(final_probe_mapping.shape)
        ),
        "probe_mapping_columns_match_memory": (
            written_probe_mapping.columns.tolist()
            == final_probe_mapping.columns.tolist()
        ),
        "probe_representation_counts_match_memory": (
            observed_probe_counts == expected_probe_counts
        ),
        "all_probe_qc_flags_are_true": (
            truthy_all(written_probe_mapping["probe_qc_eligible"])
        ),
        **representation_index_checks,
        **representation_probe_checks,
    }


def publish_or_reuse_csv_artifact(
    *,
    artifact_name,
    dataframe,
    output_path,
    temporary_path,
    validator,
):
    temporary_path.unlink(missing_ok=True)

    if output_path.is_file():
        print(
            f"{artifact_name} already exists. "
            "Reloading the persisted artifact."
        )
    else:
        dataframe.to_csv(
            temporary_path,
            index=False,
        )

        temporary_dataframe = pd.read_csv(
            temporary_path,
            low_memory=False,
        )

        temporary_checks = validator(temporary_dataframe)

        print(f"{artifact_name} temporary checks:")
        for check_name, check_passed in temporary_checks.items():
            print(f"{check_name}: {check_passed}")

        if not all(temporary_checks.values()):
            temporary_path.unlink(missing_ok=True)
            raise ValueError(
                f"{artifact_name} temporary validation failed."
            )

        temporary_path.replace(output_path)

    written_dataframe = pd.read_csv(
        output_path,
        low_memory=False,
    )

    written_checks = validator(written_dataframe)

    print()
    print(f"{artifact_name} written-artifact checks:")
    for check_name, check_passed in written_checks.items():
        print(f"{check_name}: {check_passed}")

    if not all(written_checks.values()):
        failed_checks = [
            check_name
            for check_name, check_passed in written_checks.items()
            if not check_passed
        ]

        raise ValueError(
            f"{artifact_name} written validation failed: "
            + ", ".join(failed_checks)
        )

    return written_dataframe


published_final_sample_mapping = publish_or_reuse_csv_artifact(
    artifact_name="final_sample_mapping",
    dataframe=final_sample_mapping,
    output_path=FINAL_SAMPLE_MAPPING_PATH,
    temporary_path=FINAL_CONSUMABLE_TEMP_PATHS["sample_mapping"],
    validator=validate_written_sample_mapping,
)

published_final_probe_mapping = publish_or_reuse_csv_artifact(
    artifact_name="final_probe_mapping",
    dataframe=final_probe_mapping,
    output_path=FINAL_PROBE_MAPPING_PATH,
    temporary_path=FINAL_CONSUMABLE_TEMP_PATHS["probe_mapping"],
    validator=validate_written_probe_mapping,
)


print()
print("Final mapping artifacts published.")
print(
    "Sample mapping: "
    f"{project_relative_path(FINAL_SAMPLE_MAPPING_PATH)}"
)
print(
    "Probe mapping: "
    f"{project_relative_path(FINAL_PROBE_MAPPING_PATH)}"
)
print(f"Primary cases: {len(published_final_sample_mapping):,}")
print(f"Probe-mapping rows: {len(published_final_probe_mapping):,}")

# =============================================================================
# Publish final multi-omic consumables metadata
# =============================================================================

import gc
import json
from datetime import datetime, timezone


HASH_FINAL_NUMERIC_MATRICES = True


required_metadata_objects = [
    "published_final_sample_mapping",
    "published_final_probe_mapping",
    "FINAL_CONSUMABLES_METADATA_PATH",
    "FINAL_CONSUMABLE_PATHS",
    "FINAL_CONSUMABLE_TEMP_PATHS",
    "FINAL_RNA_MATRIX_PATH",
    "FINAL_RNA_FEATURE_INDEX_PATH",
    "FINAL_HM27_MATRIX_PATH",
    "FINAL_HM450_MATRIX_PATH",
    "FINAL_SHARED_METHYLATION_MATRIX_PATH",
]

missing_metadata_objects = [
    object_name
    for object_name in required_metadata_objects
    if object_name not in globals()
]

if missing_metadata_objects:
    raise NameError(
        "Missing final-metadata objects: "
        + ", ".join(missing_metadata_objects)
    )


def file_artifact_record(path, *, compute_sha256=True):
    if not path.is_file():
        raise FileNotFoundError(
            f"Required final artifact was not found: {path}"
        )

    record = {
        "path": project_relative_path(path),
        "size_bytes": int(path.stat().st_size),
    }

    if compute_sha256:
        print(
            "Hashing artifact: "
            f"{project_relative_path(path)}"
        )
        record["sha256"] = calculate_sha256(path)
    else:
        record["sha256"] = None
        record["sha256_policy"] = (
            "not recomputed in this final metadata cell"
        )

    return record


def npy_artifact_record(
    path,
    *,
    row_axis,
    column_axis,
    compute_sha256=True,
):
    matrix = np.load(
        path,
        mmap_mode="r",
        allow_pickle=False,
    )

    record = file_artifact_record(
        path,
        compute_sha256=compute_sha256,
    )

    record.update(
        {
            "shape": [int(value) for value in matrix.shape],
            "dtype": matrix.dtype.name,
            "fortran_order": bool(matrix.flags.f_contiguous),
            "row_axis": row_axis,
            "column_axis": column_axis,
        }
    )

    del matrix
    gc.collect()

    return record


def csv_artifact_record(path, dataframe, *, compute_sha256=True):
    record = file_artifact_record(
        path,
        compute_sha256=compute_sha256,
    )

    record.update(
        {
            "rows": int(dataframe.shape[0]),
            "columns": int(dataframe.shape[1]),
            "column_names": dataframe.columns.tolist(),
        }
    )

    return record


def count_matrix_missing_values(path, *, chunk_columns=256):
    matrix = np.load(
        path,
        mmap_mode="r",
        allow_pickle=False,
    )

    missing_count = 0

    for start in range(0, matrix.shape[1], chunk_columns):
        stop = min(start + chunk_columns, matrix.shape[1])

        block = np.asarray(matrix[:, start:stop])
        missing_count += int(np.isnan(block).sum())

        del block

    shape = tuple(int(value) for value in matrix.shape)

    del matrix
    gc.collect()

    return int(missing_count), shape


def optional_source_record(path):
    if not path.is_file():
        return {
            "path": project_relative_path(path),
            "exists": False,
        }

    compute_hash = path.suffix.lower() != ".npy"

    return {
        **file_artifact_record(
            path,
            compute_sha256=compute_hash,
        ),
        "exists": True,
    }


for artifact_name, output_path in FINAL_CONSUMABLE_PATHS.items():
    if artifact_name == "metadata":
        continue

    if not output_path.is_file():
        raise FileNotFoundError(
            "A required final consumable is missing before metadata "
            f"publication: {artifact_name}"
        )


metadata_temporary_path = FINAL_CONSUMABLE_TEMP_PATHS["metadata"]
metadata_temporary_path.unlink(missing_ok=True)

if FINAL_CONSUMABLES_METADATA_PATH.is_file():
    print(
        "Final consumables metadata already exists. "
        "Reloading the persisted artifact."
    )

    with FINAL_CONSUMABLES_METADATA_PATH.open(
        "r",
        encoding="utf-8",
    ) as metadata_file:
        final_consumables_metadata = json.load(metadata_file)

else:
    hm27_missing_beta_count, hm27_matrix_shape_from_disk = (
        count_matrix_missing_values(FINAL_HM27_MATRIX_PATH)
    )

    final_artifact_records = {
        "rna_matrix": npy_artifact_record(
            FINAL_RNA_MATRIX_PATH,
            row_axis="gene",
            column_axis="final_sample_column_index",
            compute_sha256=HASH_FINAL_NUMERIC_MATRICES,
        ),
        "rna_feature_index": csv_artifact_record(
            FINAL_RNA_FEATURE_INDEX_PATH,
            pd.read_csv(FINAL_RNA_FEATURE_INDEX_PATH, low_memory=False),
        ),
        "hm27_matrix": npy_artifact_record(
            FINAL_HM27_MATRIX_PATH,
            row_axis="hm27_probe_mapping.matrix_row_index",
            column_axis="hm27_matrix_column_index",
            compute_sha256=HASH_FINAL_NUMERIC_MATRICES,
        ),
        "hm450_matrix": npy_artifact_record(
            FINAL_HM450_MATRIX_PATH,
            row_axis="hm450_probe_mapping.matrix_row_index",
            column_axis="hm450_matrix_column_index",
            compute_sha256=HASH_FINAL_NUMERIC_MATRICES,
        ),
        "shared_methylation_matrix": npy_artifact_record(
            FINAL_SHARED_METHYLATION_MATRIX_PATH,
            row_axis="shared_probe_mapping.matrix_row_index",
            column_axis="shared_matrix_column_index",
            compute_sha256=HASH_FINAL_NUMERIC_MATRICES,
        ),
        "sample_mapping": csv_artifact_record(
            FINAL_SAMPLE_MAPPING_PATH,
            published_final_sample_mapping,
        ),
        "probe_mapping": csv_artifact_record(
            FINAL_PROBE_MAPPING_PATH,
            published_final_probe_mapping,
        ),
    }

    final_artifact_records["hm27_matrix"].update(
        {
            "selected_missing_beta_count": int(
                hm27_missing_beta_count
            ),
            "selected_nonmissing_beta_count": int(
                np.prod(hm27_matrix_shape_from_disk)
                - hm27_missing_beta_count
            ),
        }
    )

    if "hm450_build_metadata" in globals():
        final_artifact_records["hm450_matrix"].update(
            {
                "selected_missing_beta_count": int(
                    hm450_build_metadata[
                        "selected_missing_beta_count"
                    ]
                ),
                "selected_nonmissing_beta_count": int(
                    hm450_build_metadata[
                        "selected_nonmissing_beta_count"
                    ]
                ),
                "build_metadata_path": project_relative_path(
                    HM450_BUILD_METADATA_PATH
                ),
            }
        )

    if "shared_published_metadata" in globals():
        final_artifact_records[
            "shared_methylation_matrix"
        ].update(
            {
                "selected_missing_beta_count": int(
                    shared_published_metadata[
                        "selected_missing_beta_count"
                    ]
                ),
                "selected_nonmissing_beta_count": int(
                    shared_published_metadata[
                        "selected_nonmissing_beta_count"
                    ]
                ),
                "observed_missing_beta_count": int(
                    shared_published_metadata.get(
                        "observed_missing_beta_count",
                        shared_published_metadata[
                            "selected_missing_beta_count"
                        ],
                    )
                ),
                "build_metadata_path": project_relative_path(
                    SHARED_BUILD_METADATA_PATH
                ),
            }
        )

    final_consumables_metadata = {
        "artifact_type": (
            "tcga_primary_tumor_multiomic_final_consumables"
        ),
        "status": "published",
        "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
        "notebook": "203_tcga_multiomic_integration.ipynb",
        "analysis_layer": "phase2_tumor_discovery_layer",
        "cohort_definition": {
            "primary_case_count": int(
                len(published_final_sample_mapping)
            ),
            "selection_unit": "one primary tumor case",
            "case_id_column": CASE_KEY,
            "sample_id_column": SAMPLE_KEY,
            "rna_file_id_column": RNA_FILE_KEY,
            "methylation_file_id_column": METHYLATION_FILE_KEY,
            "methylation_platform_counts": {
                str(platform): int(count)
                for platform, count in (
                    published_final_sample_mapping[
                        "methylation_platform"
                    ]
                    .astype("string")
                    .value_counts()
                    .sort_index()
                    .items()
                )
            },
        },
        "selection_policy": {
            "case_level_mapping_path": project_relative_path(
                FINAL_MULTIOMIC_PAIR_MAPPING_PATH
            ),
            "case_level_selection_policy_path": project_relative_path(
                CASE_LEVEL_SELECTION_POLICY_PATH
            ),
            "unresolved_cases_excluded": int(
                len(unresolved_case_ids)
            )
            if "unresolved_case_ids" in globals()
            else None,
            "no_arbitrary_case_level_tiebreak": True,
        },
        "matrix_policy": {
            "rna_representation": (
                "unstranded raw counts; no normalization in notebook 203"
            ),
            "methylation_representation": (
                "raw beta-values; NaN retained; no imputation, "
                "normalization, or batch correction"
            ),
            "hm27_hm450_policy": (
                "platform-specific matrices remain separate; the shared "
                "matrix contains eligible probes present in both platforms"
            ),
        },
        "probe_mapping": {
            "representation_counts": {
                str(representation): int(count)
                for representation, count in (
                    published_final_probe_mapping
                    .groupby("representation", sort=False)
                    .size()
                    .items()
                )
            },
        },
        "final_artifacts": final_artifact_records,
        "source_artifacts": {
            "qc_annotated_pair_inventory": optional_source_record(
                METHYLATION_QC_ANNOTATED_PAIR_INVENTORY_PATH
            ),
            "rna_candidate_raw_count_matrix": optional_source_record(
                RAW_COUNT_MATRIX_PATH
            ),
            "rna_gene_feature_index": optional_source_record(
                RNA_GENE_FEATURE_INDEX_PATH
            ),
            "rna_file_qc_inventory": optional_source_record(
                RNA_FILE_QC_INVENTORY_PATH
            ),
            "rna_raw_counts_metadata": optional_source_record(
                RNA_RAW_COUNTS_METADATA_PATH
            ),
            "methylation_file_qc_metrics": optional_source_record(
                METHYLATION_FILE_QC_METRICS_PATH
            ),
            "probe_level_qc_metrics": optional_source_record(
                PROBE_LEVEL_QC_METRICS_PATH
            ),
            "shared_probe_qc_inventory": optional_source_record(
                SHARED_PROBE_QC_INVENTORY_PATH
            ),
            "rna_file_index": optional_source_record(
                RNA_FILE_INDEX_PATH
            ),
            "rna_cohort_freeze": optional_source_record(
                RNA_COHORT_FREEZE_PATH
            ),
            "methylation_file_index": optional_source_record(
                METHYLATION_FILE_INDEX_PATH
            ),
            "methylation_cohort_freeze": optional_source_record(
                METHYLATION_COHORT_FREEZE_PATH
            ),
        },
    }

    metadata_temporary_path.write_text(
        json.dumps(
            final_consumables_metadata,
            indent=2,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )

    with metadata_temporary_path.open(
        "r",
        encoding="utf-8",
    ) as metadata_file:
        temporary_metadata = json.load(metadata_file)

    metadata_write_checks = {
        "temporary_metadata_status_is_published": (
            temporary_metadata.get("status") == "published"
        ),
        "temporary_metadata_case_count_matches_mapping": (
            int(
                temporary_metadata["cohort_definition"][
                    "primary_case_count"
                ]
            )
            == len(published_final_sample_mapping)
        ),
        "temporary_metadata_has_all_final_artifacts": (
            set(temporary_metadata["final_artifacts"])
            == set(FINAL_CONSUMABLE_PATHS) - {"metadata"}
        ),
    }

    print("Final consumables metadata temporary checks:")
    for check_name, check_passed in metadata_write_checks.items():
        print(f"{check_name}: {check_passed}")

    if not all(metadata_write_checks.values()):
        metadata_temporary_path.unlink(missing_ok=True)
        raise ValueError(
            "Final consumables metadata temporary validation failed."
        )

    metadata_temporary_path.replace(
        FINAL_CONSUMABLES_METADATA_PATH
    )


with FINAL_CONSUMABLES_METADATA_PATH.open(
    "r",
    encoding="utf-8",
) as metadata_file:
    final_consumables_metadata = json.load(metadata_file)


metadata_checks = {
    "metadata_file_exists": (
        FINAL_CONSUMABLES_METADATA_PATH.is_file()
    ),
    "metadata_status_is_published": (
        final_consumables_metadata.get("status") == "published"
    ),
    "metadata_case_count_matches_mapping": (
        int(
            final_consumables_metadata["cohort_definition"][
                "primary_case_count"
            ]
        )
        == len(published_final_sample_mapping)
    ),
    "metadata_sample_mapping_path_matches": (
        final_consumables_metadata["final_artifacts"][
            "sample_mapping"
        ]["path"]
        == project_relative_path(FINAL_SAMPLE_MAPPING_PATH)
    ),
    "metadata_probe_mapping_path_matches": (
        final_consumables_metadata["final_artifacts"][
            "probe_mapping"
        ]["path"]
        == project_relative_path(FINAL_PROBE_MAPPING_PATH)
    ),
}

print()
print("Final consumables metadata checks:")
for check_name, check_passed in metadata_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(metadata_checks.values()):
    raise ValueError(
        "Final consumables metadata validation failed."
    )


print()
print("Final multi-omic consumables metadata published.")
print(
    "Metadata: "
    f"{project_relative_path(FINAL_CONSUMABLES_METADATA_PATH)}"
)

# =============================================================================
# Final multi-omic consumable closure checks
# =============================================================================

import gc
import json


with FINAL_CONSUMABLES_METADATA_PATH.open(
    "r",
    encoding="utf-8",
) as metadata_file:
    final_consumables_metadata = json.load(metadata_file)


closure_sample_mapping = pd.read_csv(
    FINAL_SAMPLE_MAPPING_PATH,
    low_memory=False,
)

closure_probe_mapping = pd.read_csv(
    FINAL_PROBE_MAPPING_PATH,
    low_memory=False,
)

closure_rna_matrix = np.load(
    FINAL_RNA_MATRIX_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

closure_hm27_matrix = np.load(
    FINAL_HM27_MATRIX_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

closure_hm450_matrix = np.load(
    FINAL_HM450_MATRIX_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

closure_shared_matrix = np.load(
    FINAL_SHARED_METHYLATION_MATRIX_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

closure_rna_feature_index = pd.read_csv(
    FINAL_RNA_FEATURE_INDEX_PATH,
    low_memory=False,
)


def metadata_path_matches(artifact_name, path):
    return (
        final_consumables_metadata["final_artifacts"][
            artifact_name
        ]["path"]
        == project_relative_path(path)
    )


final_inprogress_paths_absent = all(
    not temporary_path.is_file()
    for temporary_path in FINAL_CONSUMABLE_TEMP_PATHS.values()
)


closure_checks = {
    "all_final_consumable_paths_exist": all(
        output_path.is_file()
        for output_path in FINAL_CONSUMABLE_PATHS.values()
    ),
    "all_inprogress_paths_are_absent": final_inprogress_paths_absent,
    "metadata_status_is_published": (
        final_consumables_metadata.get("status") == "published"
    ),
    "metadata_case_count_matches_sample_mapping": (
        int(
            final_consumables_metadata["cohort_definition"][
                "primary_case_count"
            ]
        )
        == len(closure_sample_mapping)
    ),
    "sample_mapping_has_one_row_per_case": (
        closure_sample_mapping[CASE_KEY].astype("string").is_unique
    ),
    "sample_mapping_final_indices_are_complete": (
        complete_integer_index(
            closure_sample_mapping["final_sample_column_index"],
            len(closure_sample_mapping),
        )
    ),
    "rna_matrix_shape_matches_feature_and_sample_axes": (
        closure_rna_matrix.shape
        == (
            len(closure_rna_feature_index),
            len(closure_sample_mapping),
        )
    ),
    "rna_matrix_dtype_is_uint32": (
        closure_rna_matrix.dtype == np.dtype("uint32")
    ),
    "rna_matrix_is_fortran_contiguous": (
        closure_rna_matrix.flags.f_contiguous
    ),
    "hm27_matrix_shape_matches_mappings": (
        closure_hm27_matrix.shape
        == (
            int(
                (
                    closure_probe_mapping["representation"]
                    .astype("string")
                    .eq("hm27")
                ).sum()
            ),
            int(
                closure_sample_mapping[
                    "hm27_matrix_column_index"
                ].notna().sum()
            ),
        )
    ),
    "hm450_matrix_shape_matches_mappings": (
        closure_hm450_matrix.shape
        == (
            int(
                (
                    closure_probe_mapping["representation"]
                    .astype("string")
                    .eq("hm450")
                ).sum()
            ),
            int(
                closure_sample_mapping[
                    "hm450_matrix_column_index"
                ].notna().sum()
            ),
        )
    ),
    "shared_matrix_shape_matches_mappings": (
        closure_shared_matrix.shape
        == (
            int(
                (
                    closure_probe_mapping["representation"]
                    .astype("string")
                    .eq(SHARED_METHYLATION_REPRESENTATION)
                ).sum()
            ),
            len(closure_sample_mapping),
        )
    ),
    "hm27_matrix_dtype_is_float32": (
        closure_hm27_matrix.dtype == np.dtype("float32")
    ),
    "hm450_matrix_dtype_is_float32": (
        closure_hm450_matrix.dtype == np.dtype("float32")
    ),
    "shared_matrix_dtype_is_float32": (
        closure_shared_matrix.dtype == np.dtype("float32")
    ),
    "hm27_matrix_is_fortran_contiguous": (
        closure_hm27_matrix.flags.f_contiguous
    ),
    "hm450_matrix_is_fortran_contiguous": (
        closure_hm450_matrix.flags.f_contiguous
    ),
    "shared_matrix_is_fortran_contiguous": (
        closure_shared_matrix.flags.f_contiguous
    ),
    "metadata_rna_matrix_path_matches": (
        metadata_path_matches("rna_matrix", FINAL_RNA_MATRIX_PATH)
    ),
    "metadata_hm27_matrix_path_matches": (
        metadata_path_matches("hm27_matrix", FINAL_HM27_MATRIX_PATH)
    ),
    "metadata_hm450_matrix_path_matches": (
        metadata_path_matches("hm450_matrix", FINAL_HM450_MATRIX_PATH)
    ),
    "metadata_shared_matrix_path_matches": (
        metadata_path_matches(
            "shared_methylation_matrix",
            FINAL_SHARED_METHYLATION_MATRIX_PATH,
        )
    ),
    "metadata_sample_mapping_path_matches": (
        metadata_path_matches("sample_mapping", FINAL_SAMPLE_MAPPING_PATH)
    ),
    "metadata_probe_mapping_path_matches": (
        metadata_path_matches("probe_mapping", FINAL_PROBE_MAPPING_PATH)
    ),
}


print("Final multi-omic consumable closure checks:")

for check_name, check_passed in closure_checks.items():
    print(f"{check_name}: {check_passed}")


if not all(closure_checks.values()):
    failed_checks = [
        check_name
        for check_name, check_passed in closure_checks.items()
        if not check_passed
    ]

    raise ValueError(
        "Final multi-omic consumable closure failed: "
        + ", ".join(failed_checks)
    )


print()
print("Final multi-omic consumable artifacts validated.")
print(f"Primary cases: {len(closure_sample_mapping):,}")
print(f"RNA matrix shape: {closure_rna_matrix.shape}")
print(f"HM27 matrix shape: {closure_hm27_matrix.shape}")
print(f"HM450 matrix shape: {closure_hm450_matrix.shape}")
print(f"Shared methylation matrix shape: {closure_shared_matrix.shape}")
print(
    "Metadata: "
    f"{project_relative_path(FINAL_CONSUMABLES_METADATA_PATH)}"
)


del closure_rna_matrix
del closure_hm27_matrix
del closure_hm450_matrix
del closure_shared_matrix
gc.collect()

final_sample_mapping temporary checks:
sample_mapping_file_is_not_empty: True
sample_mapping_shape_matches_memory: True
sample_mapping_columns_match_memory: True
case_ids_match_memory_order: True
sample_ids_match_memory_order: True
rna_file_ids_match_memory_order: True
methylation_file_ids_match_memory_order: True
one_row_per_case: True
final_sample_column_indices_are_complete: True
rna_final_matrix_column_indices_are_complete: True
shared_matrix_column_indices_are_complete: True
hm27_matrix_column_indices_are_complete: True
hm450_matrix_column_indices_are_complete: True
platform_counts_match_memory: True
all_rna_payloads_exist: True
all_methylation_payloads_exist: True

final_sample_mapping written-artifact checks:
sample_mapping_file_is_not_empty: True
sample_mapping_shape_matches_memory: True
sample_mapping_columns_match_memory: True
case_ids_match_memory_order: True
sample_ids_match_memory_order: True
rna_file_ids_match_memory_order: True
methylation_file_ids_match_memory_order: Tr

44

## Notebook 203 closure

Notebook 203 finalizes the TCGA primary-tumor multi-omic consumable layer for downstream tumor-discovery analyses.

The final cohort contains **9,965 primary tumor cases**, with exactly one selected RNA-seq–methylation pair per case. The case-level policy retained **36** residual multi-sample cases by joint QC dominance and excluded **8** unresolved cases without applying arbitrary tie-breakers based on UUIDs, file names, file order, or platform.

The published final consumables are:

- final RNA-seq raw-count matrix: `60,660 × 9,965`, `uint32`;
- final HM27 methylation beta-value matrix: `24,303 × 1,620`, `float32`;
- final HM450 methylation beta-value matrix: `407,696 × 8,345`, `float32`;
- final shared HM27/HM450 methylation beta-value matrix: `23,356 × 9,965`, `float32`;
- final case-level sample mapping: `9,965` rows;
- final methylation probe mapping: `455,355` rows;
- final consumables metadata JSON.

All final artifacts were validated from disk. The closure checks confirmed that all expected final paths exist, no in-progress publication paths remain, matrix shapes match the persisted sample/probe mappings, matrix dtypes and Fortran-contiguous layout are as expected, and metadata paths and cohort counts match the published artifacts.

The HM27 and HM450 one-time construction cells are intentionally preserved as Markdown archival cells because rebuilding those matrices is computationally expensive. Routine reruns of this notebook rely on the validated persisted artifacts and reload checks rather than reconstructing platform-specific methylation matrices.

No RNA-seq normalization, gene filtering, methylation imputation, methylation normalization, batch correction, or downstream modeling is performed in this notebook. Those steps remain deferred to downstream notebooks, using the published 203 consumables as the reproducible input contract.